# AI Technologies (Basic Course) · Notebook 06
## Topic 4. Machine learning and neural networks in the recognition of military objects

| Lesson | Type, duration | Title |
|---|---|---|
| **4.1** | Lecture, 2 h | Machine learning in the recognition of military objects: formation of a data set, HOG features, SVM and Random Forest classifiers, metrics of classification |
| **4.2** | Group class, 2 h | Neural networks: the neuron, the fully connected network, the convolutional network, MobileNet, transfer learning and export of the model to ONNX |

Course: *AI Technologies (Basic Course)*, AIT-BC · Content module 2 · Kruty Heroes Military Institute of Telecommunications and Information Technology, Kyiv, 2026.

**How to work with this notebook**

1. Run the cells from top to bottom: *Runtime → Run all*, or `Shift+Enter` cell by cell. Section 0 is always run first; after a restart of the runtime it is run again.
2. Every block begins with a text cell: the purpose, the formulas, the parameters of the form, the expected result and the control question.
3. The blocks are demonstrations with measurements on real data: change the parameters of the forms and run the cell again. Blocks of Lesson 4.2 use the results of Lesson 4.1 (features, crops); if a block of Lesson 4.1 was not run, the block of Lesson 4.2 computes what it needs itself.
4. A GPU is not required: every block runs on the processor of the Colab session in 1–4 minutes with the default parameters. With a GPU (*Runtime → Change runtime type → T4 GPU*) the blocks of Lesson 4.2 run faster.
5. The results are saved in the course project on Google Drive (`outputs`, `models`, `project.json`), created in Notebook 02.

**Data.** Only real images are used, with the annotation of the authors of the data sets:
- **Military vehicles** — 9444 photographs of 24 types of military vehicles (tanks, infantry fighting vehicles, armoured personnel carriers, artillery, multiple rocket launchers, air defence and missile systems) with labels of the type and of the class;
- **SULAND** — 33 771 frames of 55 video sequences of a camera moved over grass, fallen leaves and gravel with surrogates of the anti-personnel surface landmines PFM-1 ("butterfly") and PMA-2 ("starfish"), recorded in Italy and in the USA, with boxes of the mines in the YOLO format in two versions of the annotation (the original one and the refined one).

**Contents**

- **0. Preparation** — 0.1 runtime, course library and data sets · 0.2 course project
- **Lesson 4.1 (lecture)**
  - 4.1.1 The task of recognition and the data set of military vehicles
  - 4.1.2 Audit of the data set: duplicates and contradictory labels
  - 4.1.3 Annotation of frames: the YOLO format, statistics and quality of the annotation
  - 4.1.4 HOG features
  - 4.1.5 SVM and Random Forest classifiers; accuracy, precision, recall, F1
  - 4.1.6 Division of the data set: random frames or whole sequences
- **Lesson 4.2 (group class)**
  - 4.2.1 The artificial neuron
  - 4.2.2 The fully connected network
  - 4.2.3 The convolutional network: convolution and feature maps
  - 4.2.4 MobileNet: depthwise separable convolution and the cost of computation
  - 4.2.5 Transfer learning: features of a network trained on ImageNet
  - 4.2.6 Export of the model to ONNX and check of the inference
- **Summary**: self-check of Lessons 4.1–4.2, data sources and licences

---
# 0. Preparation

### Block 0.1. Runtime, course library and data sets

**Purpose.** To prepare the runtime: the libraries, the course library `aitbc` (version 2.5.0 adds the readers of the data sets of military vehicles and of SULAND, the HOG and CNN features and the diagrams of the networks) and the open data sets of the lesson.

**Parameters.**
- `CACHE_ON_DRIVE` — keep the downloaded files on Google Drive (they remain after the end of the session); otherwise they are kept on the disk of the session.
- `PREFETCH_ALL` — download now the data of blocks 4.1.1–4.1.5: the two parquet files of military vehicles (166 MB), the two archives of the SULAND annotation (16 MB) and 12 frames of SULAND (6 MB). The 360 frames of block 4.1.6 (about 190 MB) are downloaded in that block.

**Output.** A table of versions, the processor or GPU of the session and the size of the cache.

In [ ]:
#@title 0.1 Runtime, course library and data sets (run once) { display-mode: "form" }
CACHE_ON_DRIVE = False #@param {type:"boolean"}
PREFETCH_ALL = True #@param {type:"boolean"}

import sys, os, platform, time, json, math, importlib, shutil
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
import sklearn
import skimage
import torch
import torchvision
pd.set_option("display.max_colwidth", 200)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
WORK_DIR = "/content" if IN_COLAB else os.getcwd()

LIB_SOURCE = r'''"""
aitbc - course library of "AI Technologies (Basic Course)" (AIT-BC).

Kruty Heroes Military Institute of Telecommunications and Information Technology,
Kyiv, 2026.

All demonstrations of the course use REAL open data sets recorded by UAVs and
ground vehicles. Only the files that a block needs are downloaded: large archives
are read selectively with HTTP range requests, and every file is cached in
DATA_ROOT (a Google Drive folder keeps the cache between sessions).

Data sets
---------
HIT-UAV   2898 thermal (LWIR) images 640x512 and 5 thermal videos. DJI Matrice
          M210 V2 with a Zenmuse XT2 camera (25 mm lens, field of view 25 x 20 deg),
          flight altitude 60-130 m, camera angle 30-90 deg, day and night. Boxes of
          Person, Car, Bicycle, OtherVehicle, DontCare.
          Suo J. et al. Scientific Data 10, 227 (2023). Licence CC BY 4.0.
Kust4K    4024 pixel-aligned visible (RGB) and thermal (TIR) image pairs 640x512
          from a hexacopter over urban roads, day and night; pixel labels of 8 classes.
          Scientific Data 12, 1701 (2025). Licence CC BY 4.0.
VisDrone  RGB images of drones over cities (VisDrone2019-DET validation part,
          548 images) with boxes of pedestrians, vehicles and other objects.
          Zhu P. et al. IEEE TPAMI 44(11), 7380-7399 (2022).
GOOSE     camera images of a ground vehicle on roads, field tracks, in forests and on
          a training area, with pixel labels of 64 classes (validation part, 962 images).
          IEEE ICRA 2024. Licence CC BY-SA 4.0.
M3OT      synchronised visible and thermal image sequences of two DJI Mavic 3T drones
          (100-120 m, 10 fps, 640x512, day, dusk, night) with vehicle tracks.
          Nie Z. et al. Scientific Data 12, 1927 (2025). Licence CC BY 4.0.

TII-RATM  frames 640x480 (125 fps), IMU 500 Hz and motion capture of an FPV racing quadrotor
          flown by human pilots in an indoor arena with racing gates (wide-angle camera
          155 x 115 deg). Bosello M. et al. IEEE RA-L 9(4), 3799-3806 (2024). Licence CC BY 4.0.
MilVeh    9444 photographs of 24 types of military vehicles collected from web sites
          (Kricheli J. S. et al., arXiv:2407.15192, 2024; Hugging Face
          leibnitz-lab/military_vehicles, licence declared: MIT).
SULAND    33771 frames 1440x1920 of video sequences of a camera over grass, leaves and
          gravel with surrogates of the surface landmines PFM-1 and PMA-2 (Italy and USA);
          Vivoli E. et al. Remote Sensing 16(4), 677 (2024); refined annotation SULAND_v2,
          Lekhak S. et al., arXiv:2607.28996 (2026). Licence CC BY 4.0.

Units: metres, seconds, degrees. Images are NumPy arrays H x W (grey) or H x W x 3 (RGB).
"""

__version__ = "2.6.0"

import csv
import io
import json
import math
import os
import struct
import time
import urllib.error
import urllib.request
import zlib
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import cv2

DATA_ROOT = os.environ.get("AITBC_DATA") or (
    "/content/aitbc_data" if os.path.isdir("/content") else os.path.abspath("aitbc_data"))


def set_data_root(path):
    """Set the folder of the download cache (for example a folder on Google Drive)."""
    global DATA_ROOT
    DATA_ROOT = os.path.abspath(path)
    os.makedirs(DATA_ROOT, exist_ok=True)
    return DATA_ROOT


def _cache(*parts):
    p = os.path.join(DATA_ROOT, *parts)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    return p


def cache_size():
    """Number of files and size of the download cache, MB."""
    n, s = 0, 0
    for dp, _, fn in os.walk(DATA_ROOT):
        for f in fn:
            n += 1
            s += os.path.getsize(os.path.join(dp, f))
    return n, s / 1e6


# =============================================================================
# 1. Downloading
# =============================================================================
class _HTTPSRedirect(urllib.request.HTTPRedirectHandler):
    """Follow redirects, keep the Range header and upgrade http:// targets to https://."""

    def redirect_request(self, req, fp, code, msg, headers, newurl):
        if newurl.startswith("http://"):
            newurl = "https://" + newurl[len("http://"):]
        new = super().redirect_request(req, fp, code, msg, headers, newurl)
        if new is not None and req.has_header("Range"):
            new.add_header("Range", req.get_header("Range"))
        return new


_OPENER = urllib.request.build_opener(_HTTPSRedirect)
_OPENER.addheaders = [("User-Agent", "aitbc-course/2.0")]


def _urlopen(url, headers=None, timeout=60, retries=4):
    """Open url with retries; a new Request object is built for every attempt."""
    err = None
    for k in range(retries):
        try:
            return _OPENER.open(urllib.request.Request(url, headers=dict(headers or {})), timeout=timeout)
        except urllib.error.HTTPError as e:
            if e.code in (400, 401, 403, 404, 416):
                raise
            err = e
        except Exception as e:                      # network error: retry after a pause
            err = e
        time.sleep(1.5 * (k + 1))
    raise err


def _retry(transfer, retries=5):
    """Run a complete transfer (request and reading of the whole body) with retries: a broken connection,
    a read timeout or an incomplete body repeats the transfer after a pause; client errors are not repeated."""
    err = None
    for k in range(retries):
        try:
            return transfer()
        except urllib.error.HTTPError as e:
            if e.code in (400, 401, 403, 404, 416):
                raise
            err = e
        except Exception as e:                      # timeout, reset connection, incomplete read
            err = e
        time.sleep(2.0 * (k + 1))
    raise err


def _read_url(url, headers=None, timeout=60, min_bytes=0, retries=5):
    """Whole body of a response (e.g. of a range request) with retries; a body shorter than min_bytes is
    an incomplete transfer and is repeated."""
    def transfer():
        with _OPENER.open(urllib.request.Request(url, headers=dict(headers or {})), timeout=timeout) as r:
            data = r.read()
        if len(data) < min_bytes:
            raise IOError(f"incomplete transfer: {len(data)} of at least {min_bytes} bytes")
        return data
    return _retry(transfer, retries)


def fetch(url, path, timeout=120, retries=5):
    """Download url to path with retries; nothing is done when the file is already in the cache.
    The file appears under its name only after a complete transfer."""
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = path + ".part"

    def transfer():
        with _OPENER.open(urllib.request.Request(url), timeout=timeout) as r, open(tmp, "wb") as fh:
            expected = int(r.headers.get("Content-Length") or 0)
            n = 0
            while True:
                b = r.read(1 << 20)
                if not b:
                    break
                fh.write(b)
                n += len(b)
        if expected and n < expected:
            raise IOError(f"incomplete download of {url}: {n} of {expected} bytes")
        os.replace(tmp, path)
        return path
    return _retry(transfer, retries)


def fetch_many(jobs, threads=8):
    """jobs: list of (url, path); parallel download, returns the list of paths."""
    with ThreadPoolExecutor(threads) as ex:
        return list(ex.map(lambda j: fetch(*j), jobs))


class _RangeFile(io.RawIOBase):
    """Seekable read-only file over HTTP range requests (reads the directory of a remote zip)."""

    def __init__(self, url, chunk=1 << 17, timeout=60):
        self.url, self.chunk, self.timeout = url, chunk, timeout
        r = _urlopen(url, {"Range": "bytes=0-0"}, timeout)
        cr = r.headers.get("Content-Range")
        r.read()
        if r.status != 206 or not cr:
            raise IOError(f"the server of {url} does not support range requests")
        self.size = int(cr.split("/")[-1])
        self.pos, self.buf_start, self.buf = 0, 0, b""

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else (self.pos + off if whence == 1 else self.size + off)
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = min(n, self.size - self.pos)
        if n <= 0:
            return b""
        if not (self.buf_start <= self.pos and self.pos + n <= self.buf_start + len(self.buf)):
            start = self.pos
            end = min(self.size, start + max(n, self.chunk)) - 1
            self.buf = _read_url(self.url, {"Range": f"bytes={start}-{end}"}, self.timeout, min_bytes=end - start + 1)
            self.buf_start = start
        off = self.pos - self.buf_start
        out = self.buf[off:off + n]
        self.pos += len(out)
        return out

    def readinto(self, b):
        d = self.read(len(b))
        b[:len(d)] = d
        return len(d)


class RemoteZip:
    """Selective reading of members of a large remote zip archive: only the central
    directory and the requested members are transferred; members are cached on disk."""

    def __init__(self, url, cache_name):
        import zipfile
        self.url, self.cache_name = url, cache_name
        idx_path = _cache("_index", cache_name + ".json")
        if os.path.exists(idx_path):
            with open(idx_path) as fh:
                self.index = json.load(fh)
        else:
            zf = zipfile.ZipFile(_RangeFile(url))
            self.index = {i.filename: (i.header_offset, i.compress_size, i.file_size, i.compress_type)
                          for i in zf.infolist() if not i.filename.endswith("/")}
            with open(idx_path, "w") as fh:
                json.dump(self.index, fh)

    def namelist(self):
        return list(self.index)

    def _fetch_member(self, name):
        path = _cache(self.cache_name, name)
        if os.path.exists(path):
            return path
        off, csize, usize, ctype = self.index[name]
        raw = _read_url(self.url, {"Range": f"bytes={off}-{off + 30 + 1024 + csize}"}, min_bytes=30 + csize)
        if raw[:4] != b"PK\x03\x04":
            raise IOError(f"bad local header of {name}")
        n, m = struct.unpack("<HH", raw[26:30])
        data = raw[30 + n + m: 30 + n + m + csize]
        if ctype == 8:
            data = zlib.decompressobj(-15).decompress(data)
        elif ctype != 0:
            raise IOError(f"unsupported compression {ctype} in {name}")
        if len(data) != usize:
            raise IOError(f"incomplete member {name}")
        with open(path + ".part", "wb") as fh:
            fh.write(data)
        os.replace(path + ".part", path)
        return path

    def get(self, names, threads=8):
        """Download members (a name or a list of names); returns local path(s)."""
        single = isinstance(names, str)
        names = [names] if single else list(names)
        with ThreadPoolExecutor(threads) as ex:
            paths = list(ex.map(self._fetch_member, names))
        return paths[0] if single else paths


def _rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f"cannot read {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# =============================================================================
# 2. Data sets
# =============================================================================
class HITUAV:
    """HIT-UAV: thermal images and videos of a DJI Matrice M210 V2 with a Zenmuse XT2.

    File name T_HHH_AA_W_NNNNN: T = 0 day / 1 night, HHH = flight altitude, m,
    AA = camera angle, deg (90 = vertical view), W = 0 sunny / 1 rainy.
    Images are 8-bit white-hot (warm = bright) after the automatic gain control
    of the camera: brightness is a relative, not a calibrated, temperature."""

    PALETTE = "white-hot"
    BASE = "https://raw.githubusercontent.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset/main/"
    CLASSES = ("Person", "Car", "Bicycle", "OtherVehicle", "DontCare")
    COLORS = {"Person": (0, 229, 255), "Car": (255, 235, 59), "Bicycle": (118, 255, 3),
              "OtherVehicle": (255, 152, 0), "DontCare": (190, 190, 190)}
    CAMERA = dict(model="DJI Zenmuse XT2, uncooled VOx microbolometer", width=640, height=512,
                  focal_mm=25.0, pixel_pitch_um=17.0, hfov_deg=25.0, vfov_deg=20.0, band_um="7.5-13.5")
    EXAMPLES = ('1_60_30_0_00047.jpg', '0_60_60_0_01734.jpg', '1_70_90_0_07776.jpg', '0_100_50_0_08162.jpg', '1_120_90_0_06161.jpg', '0_130_40_0_05603.jpg', '1_90_70_0_04990.jpg', '0_110_80_0_08518.jpg')
    VIDEOS = {"60m-30_1.mov": "60 m, camera angle 30 deg: people on a sports ground",
              "60m-40_3.mov": "60 m, 40 deg: cars on a parking lot",
              "70m-90_1.mov": "70 m, 90 deg (vertical view): people on a square",
              "120m-30_3.mov": "120 m, 30 deg: road along buildings",
              "130m-60_3.mov": "130 m, 60 deg: parking lot and roads"}

    def __init__(self, splits=("train", "val", "test")):
        import pandas as pd
        imgs, objs = [], []
        jobs = [(self.BASE + f"normal_json/annotations/{sp}.json", _cache("hituav", "annotations", sp + ".json"))
                for sp in splits]
        for sp, path in zip(splits, fetch_many(jobs, 3)):
            with open(path) as fh:
                d = json.load(fh)
            cat = {c["id"]: c["name"] for c in d["categories"]}
            id2f = {}
            for im in d["images"]:
                fn = im.get("filename") or im.get("file_name")
                id2f[im["id"]] = fn
                t, h, a, w = fn.split("_")[:4]
                imgs.append(dict(file=fn, split=sp, night=t == "1", altitude=int(h), angle=int(a), rainy=w == "1"))
            for an in d.get("annotation", d.get("annotations", [])):
                x, y, w, h = an["bbox"]
                objs.append(dict(ann_id=an["id"], file=id2f[an["image_id"]], cls=cat[an["category_id"]],
                                 x1=float(x), y1=float(y), x2=float(x + w), y2=float(y + h)))
        # oriented boxes [xc, yc, w, h, angle]: true width and length of rotated objects
        rjobs = [(self.BASE + f"rotate_json/annotations/{sp}.json", _cache("hituav", "annotations", "rot_" + sp + ".json"))
                 for sp in splits]
        rot = {}
        for path in fetch_many(rjobs, 3):
            with open(path) as fh:
                d = json.load(fh)
            for an in d.get("annotations", d.get("annotation", [])):
                b = an["bbox"]
                rot[an["id"]] = (min(b[2], b[3]), max(b[2], b[3]), b[4] if len(b) > 4 else 0.0)
        self.images = pd.DataFrame(imgs).sort_values("file").reset_index(drop=True)
        o = pd.DataFrame(objs)
        o["w"], o["h"] = o.x2 - o.x1, o.y2 - o.y1
        o["width_px"] = o.ann_id.map(lambda i: rot.get(i, (np.nan,) * 3)[0])     # oriented box: short side
        o["length_px"] = o.ann_id.map(lambda i: rot.get(i, (np.nan,) * 3)[1])    # oriented box: long side
        self.objects = o.merge(self.images[["file", "night", "altitude", "angle"]], on="file", how="left")
        self._split = dict(zip(self.images.file, self.images.split))
        self._groups = {f: g for f, g in self.objects.groupby("file")}
        counts = self.objects.pivot_table(index="file", columns="cls", values="x1", aggfunc="count", fill_value=0)
        self.images = self.images.merge(counts, left_on="file", right_index=True, how="left").fillna(0)

    def _url(self, f):
        return self.BASE + f"normal_json/{self._split[f]}/{f}"

    def path(self, file):
        return fetch(self._url(file), _cache("hituav", self._split[file], file))

    def prefetch(self, files, threads=8):
        return fetch_many([(self._url(f), _cache("hituav", self._split[f], f)) for f in files], threads)

    def load(self, file):
        """Thermal image (H x W uint8) and the list of annotated objects."""
        img = cv2.imread(self.path(file), cv2.IMREAD_GRAYSCALE)
        g = self._groups.get(file)
        objs = [] if g is None else [dict(cls=r.cls, box=[r.x1, r.y1, r.x2, r.y2]) for r in g.itertuples()]
        return img, objs

    def video(self, name):
        return fetch(self.BASE + "video_sample/" + name, _cache("hituav", "video", name))

    @staticmethod
    def video_geometry(name):
        """Flight altitude, m, and camera angle, deg, from a video name such as '60m-30_1.mov'."""
        alt, rest = name.split("m-")
        return int(alt), int(rest.split("_")[0])

    def gsd(self, altitude, angle=90):
        """Ground sampling distance at the image centre, m/px (formula of Lesson 1.1)."""
        rng = altitude / max(math.sin(math.radians(angle)), 1e-3)
        return 2 * rng * math.tan(math.radians(self.CAMERA["hfov_deg"]) / 2) / self.CAMERA["width"]


class Kust4K:
    """Kust4K: pixel-aligned RGB and thermal (TIR) pairs 640x512 of a hexacopter
    HET-3021C over urban roads, day (id ...D) and night (id ...N), with pixel labels.

    The thermal images are pseudo-colour (palette from purple = cold to yellow = warm);
    tir_gray is their luminance, a relative thermal brightness. The data set contains
    frames with SIMULATED failures (visible channel set to zero, thermal channel
    blurred) made by its authors; they are excluded from the curated lists below."""

    PALETTE = "pseudo-colour (purple = cold, yellow = warm)"
    URL = "https://ndownloader.figshare.com/files/{}"
    FILES = {"rgb": 55979147, "tir": 55979165, "seg": 55979159}
    CLASSES = ("unlabelled", "road", "building", "motorcycle", "car", "truck", "tree", "person", "traffic facilities")
    COLORS = np.array([(0, 0, 0), (150, 150, 150), (229, 57, 53), (255, 152, 0), (255, 235, 59),
                       (236, 64, 122), (46, 160, 67), (0, 229, 255), (30, 136, 229)], np.uint8)
    OBJECTS = (3, 4, 5, 7)
    CAMERA = dict(platform="hexacopter HET-3021C", tir_focal_mm=9.1, tir_pixel_um=12.0,
                  rgb="1920x1080, f = 8 mm, registered to the thermal image and resized to 640x512")
    DAY = ("00720D", "00719D", "00031D", "01944D", "00851D", "00580D", "00990D", "00355D", "02094D", "02293D")
    NIGHT = ("02566N", "02807N", "02921N", "03135N", "03685N", "03895N", "03780N", "04004N", "02867N", "02599N")

    def __init__(self):
        self._z = {}

    def _zip(self, key):
        if key not in self._z:
            self._z[key] = RemoteZip(self.URL.format(self.FILES[key]), "kust4k_" + key)
        return self._z[key]

    def prefetch(self, ids, threads=6):
        names = [i + ".png" for i in ids]
        with ThreadPoolExecutor(3) as ex:
            list(ex.map(lambda k: self._zip(k).get(names, threads), ("rgb", "tir", "seg")))

    def pair(self, fid):
        """dict(id, night, rgb, tir, tir_gray, labels) for a frame id such as '00720D'."""
        self.prefetch([fid])
        name = fid + ".png"
        rgb = _rgb(self._zip("rgb").get(name))
        tir = _rgb(self._zip("tir").get(name))
        lab = cv2.imread(self._zip("seg").get(name), cv2.IMREAD_UNCHANGED)
        return dict(id=fid, night=fid.endswith("N"), rgb=rgb, tir=tir,
                    tir_gray=cv2.cvtColor(tir, cv2.COLOR_RGB2GRAY), labels=lab)


class VisDrone:
    """VisDrone2019-DET validation part: RGB images of drones with object boxes.
    Annotation line: x, y, w, h, score, category, truncation, occlusion."""

    URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip"
    CLASSES = ("ignored region", "pedestrian", "people", "bicycle", "car", "van", "truck", "tricycle",
               "awning-tricycle", "bus", "motor", "others")
    COLORS = {"pedestrian": (0, 229, 255), "people": (0, 176, 255), "bicycle": (118, 255, 3), "car": (255, 235, 59),
              "van": (255, 193, 7), "truck": (255, 152, 0), "tricycle": (186, 104, 200),
              "awning-tricycle": (156, 39, 176), "bus": (244, 67, 54), "motor": (236, 64, 122)}

    CURATED = ('0000213_03920_d_0000243', '0000291_03801_d_0000887', '0000117_00112_d_0000087', '0000001_08414_d_0000013', '0000155_00401_d_0000001', '0000193_01705_d_0000112', '0000295_02400_d_0000033', '0000289_04601_d_0000834', '0000215_00000_d_0000256', '0000327_04001_d_0000731')

    def __init__(self):
        self.z = RemoteZip(self.URL, "visdrone_val")

    def names(self):
        return sorted(n.split("/")[-1][:-4] for n in self.z.namelist() if n.endswith(".jpg"))

    def _members(self, name):
        return [f"VisDrone2019-DET-val/images/{name}.jpg", f"VisDrone2019-DET-val/annotations/{name}.txt"]

    def prefetch(self, names, threads=8):
        self.z.get([m for n in names for m in self._members(n)], threads)

    def load(self, name):
        """RGB image and the list of objects (ignored regions and 'others' are skipped)."""
        pi, pa = self.z.get(self._members(name))
        objs = []
        with open(pa) as fh:
            for line in fh:
                v = line.strip().strip(",").split(",")
                if len(v) < 8:
                    continue
                x, y, w, h, _, c, trunc, occ = (int(float(s)) for s in v[:8])
                if c in (0, 11):
                    continue
                objs.append(dict(cls=self.CLASSES[c], box=[x, y, x + w, y + h], truncation=trunc, occlusion=occ))
        return _rgb(pi), objs


class M3OT:
    """M3OT: synchronised visible (RGB) and thermal (IR) image sequences of two DJI Mavic 3T
    drones flying at 100-120 m (10 frames per second, 640 x 512 px), day, dusk and night, with
    multi-object tracking annotation of vehicles (MOT format: frame, id, x, y, w, h, flag, class,
    confidence). The track identifiers are the same in the RGB and IR sequences of one drone.
    The two channels were aligned by the authors with an affine transform; a residual spatial
    offset of several pixels and a time offset of up to several frames remain (Lesson 1.4).

    The IR frames are 8-bit grey images in the BLACK-HOT palette (warm = dark). The data
    descriptor does not state the palette; it is established from the scene: in all nine
    sequences of drone 1 the road surface is darker than vegetation by day, at dusk and at night,
    while asphalt is, as a rule, the warmer surface (Lesson 1.3, task 1.3.4). Vehicles are
    bright, i.e. cooler than the road. to_white_hot() gives the white-hot convention of HIT-UAV."""

    PALETTE = "black-hot"
    URL = "https://ndownloader.figshare.com/files/52023875"
    CAMERA = dict(platform="DJI Mavic 3T", altitude_m="100-120", fps=10, width=640, height=512,
                  rgb="telephoto camera, f = 29.85 mm (35 mm equivalent 162 mm), frames registered to IR",
                  ir="uncooled thermal camera 640 x 512, f = 9.1 mm (35 mm equivalent 40 mm)")
    SEQUENCES = {"1-01": "day, suburban road", "1-02": "night, city crossroads", "1-03": "day, city road",
                 "1-04": "day, suburban highway", "1-05": "day, crossroads", "1-06": "dusk, highway",
                 "1-07": "day, crossroads", "1-08": "dusk, crossroads", "1-09": "day, city street with parked cars"}

    def __init__(self):
        self.z = RemoteZip(self.URL, "m3ot")
        self._seq = {}
        for n in self.z.namelist():
            parts = n.split("/")        # M3OT/<drone>/<rgb|ir>/<split>/<sequence>/img1/<frame>.PNG
            if len(parts) == 7 and parts[5] == "img1":
                self._seq.setdefault((parts[4].rstrip("T"), parts[2]), []).append(n)
        for k in self._seq:
            self._seq[k].sort()

    def sequences(self):
        return sorted({k[0] for k in self._seq})

    def n_frames(self, seq, channel="rgb"):
        return len(self._seq[(seq, channel)])

    def frame_names(self, seq, channel="rgb", start=0, count=None, step=1):
        names = self._seq[(seq, channel)]
        end = len(names) if count is None else min(len(names), start + count * step)
        return names[start:end:step]

    def prefetch(self, seq, channel="rgb", start=0, count=None, step=1, threads=8):
        return self.z.get(self.frame_names(seq, channel, start, count, step), threads)

    def frames(self, seq, channel="rgb", start=0, count=None, step=1):
        """List of RGB images (H x W x 3) of a sequence; IR frames are grey images in 3 equal channels."""
        return [_rgb(p) for p in self.prefetch(seq, channel, start, count, step)]

    def tracks(self, seq, channel="rgb"):
        """Annotation table: frame (from 1), id, x, y, w, h, cx, cy."""
        import pandas as pd
        names = self._seq[(seq, channel)]
        gt = names[0].rsplit("/img1/", 1)[0] + "/gt/gt.txt"
        g = pd.read_csv(self.z.get(gt), header=None, names=["frame", "id", "x", "y", "w", "h", "flag", "cls", "conf"])
        g["cx"], g["cy"] = g.x + g.w / 2, g.y + g.h / 2
        return g[["frame", "id", "x", "y", "w", "h", "cx", "cy"]]

    def objects(self, table, frame):
        """Objects of one frame in the format of draw_boxes."""
        t = table[table.frame == frame]
        return [dict(cls="vehicle", id=int(r.id), box=[r.x, r.y, r.x + r.w, r.y + r.h]) for r in t.itertuples()]


class GOOSE:
    """GOOSE: front camera of a ground vehicle (2048x1000) with pixel labels of 64 classes."""

    URL = "https://goose-dataset.de/storage/goose_2d_val.zip"
    TRAVERSABLE = ("asphalt", "cobble", "gravel", "soil", "low_grass", "moss", "leaves", "snow",
                   "road_marking", "pedestrian_crossing", "sidewalk", "bikeway")
    CONDITIONAL = ("high_grass", "crops", "debris", "tree_root")
    IGNORE = ("undefined", "sky", "ego_vehicle", "outlier")
    CURATED = (
        'labels/val/2022-07-22_flight/2022-07-22_flight__0078_1658494246011438339_labelids.png',
        'labels/val/2022-07-22_flight/2022-07-22_flight__0158_1658494465090456964_labelids.png',
        'labels/val/2022-08-30_siegertsbrunn_feldwege/2022-08-30_siegertsbrunn_feldwege__0528_1661860582736903436_labelids.png',
        'labels/val/2022-08-30_siegertsbrunn_feldwege/2022-08-30_siegertsbrunn_feldwege__0551_1661860697315889089_labelids.png',
        'labels/val/2022-09-21_garching_uebungsplatz_2/2022-09-21_garching_uebungsplatz_2__0002_1663755194931038715_labelids.png',
        'labels/val/2022-09-21_garching_uebungsplatz_2/2022-09-21_garching_uebungsplatz_2__0038_1663755353192423465_labelids.png',
        'labels/val/2023-03-03_garching_2/2023-03-03_garching_2__0114_1677850404544550420_labelids.png',
        'labels/val/2023-03-03_garching_2/2023-03-03_garching_2__0156_1677850669766430912_labelids.png',
        'labels/val/2022-12-07_aying_hills/2022-12-07_aying_hills__0007_1670420722571435228_labelids.png',
        'labels/val/2022-12-07_aying_hills/2022-12-07_aying_hills__0057_1670421449778652057_labelids.png',
        'labels/val/2022-12-07_aying_hills/2022-12-07_aying_hills__0037_1670421230548919820_labelids.png',
        'labels/val/2023-05-15_neubiberg_rain/2023-05-15_neubiberg_rain__0650_1684157928693659670_labelids.png',
    )

    def __init__(self):
        self.z = RemoteZip(self.URL, "goose_val")
        self.names, self.colors = {}, {}
        with open(self.z.get("goose_label_mapping.csv"), newline="") as fh:
            for row in csv.DictReader(fh):
                k = int(row["label_key"])
                h = row["hex"].lstrip("#")
                self.names[k] = row["class_name"]
                self.colors[k] = tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))
        self.lut = np.zeros((256, 3), np.uint8)
        for k, c in self.colors.items():
            self.lut[k] = c
        self.id_of = {v: k for k, v in self.names.items()}

    def frames(self):
        return sorted(n for n in self.z.namelist() if n.endswith("_labelids.png"))

    @staticmethod
    def _image_member(label_member):
        return label_member.replace("labels/", "images/").replace("_labelids.png", "_windshield_vis.png")

    def prefetch(self, members, threads=6):
        self.z.get([m for lm in members for m in (self._image_member(lm), lm)], threads)

    def load(self, label_member, scale=0.5):
        """RGB image and label map (both resized by `scale`)."""
        pi, pl = self.z.get([self._image_member(label_member), label_member])
        img, lab = _rgb(pi), cv2.imread(pl, cv2.IMREAD_UNCHANGED)
        if scale != 1:
            w, h = int(img.shape[1] * scale), int(img.shape[0] * scale)
            img = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)
            lab = cv2.resize(lab, (w, h), interpolation=cv2.INTER_NEAREST)
        return img, lab

    def traversability(self, lab):
        """0 = ignored (sky, own vehicle), 1 = traversable, 2 = conditionally traversable, 3 = obstacle."""
        lut = np.full(256, 3, np.uint8)
        for group, v in ((self.TRAVERSABLE, 1), (self.CONDITIONAL, 2), (self.IGNORE, 0)):
            for n in group:
                if n in self.id_of:
                    lut[self.id_of[n]] = v
        return lut[lab]


# =============================================================================
# 3. Camera geometry
# =============================================================================
def gsd_nadir(altitude, hfov_deg, width_px):
    """Ground sampling distance of a vertical view, m/px: GSD = 2 H tan(alpha/2) / N."""
    return 2 * altitude * math.tan(math.radians(hfov_deg) / 2) / width_px


def hfov_from_sensor(focal_mm, pixel_um, width_px):
    """Horizontal field of view, deg, from the focal length and the pixel pitch."""
    return math.degrees(2 * math.atan(width_px * pixel_um * 1e-3 / 2 / focal_mm))


# =============================================================================
# 4. Degradations of the video signal (applied to real frames)
# =============================================================================
def fractal_noise(shape, rng, octaves=5, start=3, persistence=0.5):
    """Smooth random field in [0, 1] (sum of up-sampled random grids)."""
    h, w = shape
    out = np.zeros((h, w), np.float32)
    amp, m = 1.0, max(h, w)
    for o in range(octaves):
        n = start * (2 ** o)
        grid = rng.random((max(2, int(round(n * h / m)) + 1), max(2, int(round(n * w / m)) + 1)), dtype=np.float32)
        out += amp * cv2.resize(grid, (w, h), interpolation=cv2.INTER_CUBIC)
        amp *= persistence
    out -= out.min()
    return out / max(float(out.max()), 1e-6)


def _f(img):
    return img.astype(np.float32)


def _u8(x):
    return np.clip(x, 0, 255).astype(np.uint8)


def motion_blur(img, length=9.0, angle=0.0):
    """Linear motion blur: the point spread function is a line of `length` px at `angle` deg."""
    if length < 1.5:
        return img.copy()
    size = int(math.ceil(length)) | 1
    k = np.zeros((size * 4, size * 4), np.uint8)
    c = size * 2
    dx = math.cos(math.radians(angle)) * (length - 1) * 2
    dy = -math.sin(math.radians(angle)) * (length - 1) * 2
    cv2.line(k, (int(c - dx), int(c - dy)), (int(c + dx), int(c + dy)), 255, 4, cv2.LINE_AA)
    k = cv2.resize(k.astype(np.float32), (size, size), interpolation=cv2.INTER_AREA)
    return cv2.filter2D(img, -1, k / max(k.sum(), 1e-6), borderType=cv2.BORDER_REFLECT)


def defocus(img, sigma=1.5):
    return img.copy() if sigma <= 0 else cv2.GaussianBlur(img, (0, 0), sigma)


def add_noise(img, sigma=10.0, seed=None):
    rng = np.random.default_rng(seed)
    return _u8(_f(img) + rng.normal(0, sigma, img.shape))


def low_light(img, level=0.15, gain_max=8.0, seed=None):
    """Dark scene and automatic gain: light reduced by `level`, photon and read noise added."""
    if level >= 0.999:
        return img.copy()
    rng = np.random.default_rng(seed)
    lin = (_f(img) / 255.0) ** 2.2 * level
    full_well = 3000.0
    noisy = lin + rng.standard_normal(lin.shape).astype(np.float32) * np.sqrt(lin / full_well + (4.0 / full_well) ** 2)
    gain = float(np.clip(0.16 / max(float(lin.mean()), 1e-6), 1.0, gain_max))
    return _u8(np.clip(noisy * gain, 0, 1) ** (1 / 2.2) * 255)


def haze(img, density=0.4, airlight=200.0):
    """Uniform haze: I = J t + A (1 - t), transmission t = 1 - density."""
    t = 1.0 - float(np.clip(density, 0, 0.98))
    return _u8(_f(img) * t + airlight * (1 - t))


def smoke(img, density=0.5, seed=0, color=170.0, opacity=0.92):
    """Smoke: spatially variable veil."""
    if density <= 0:
        return img.copy()
    n = fractal_noise(img.shape[:2], np.random.default_rng(seed))
    m = np.clip((n - (1.0 - density)) * 3.0, 0, 1) * opacity
    if img.ndim == 3:
        m = m[..., None]
    return _u8(_f(img) * (1 - m) + color * m)


def jpeg(img, quality=30):
    """JPEG compression artefacts."""
    if quality >= 100:
        return img.copy()
    x = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    _, buf = cv2.imencode(".jpg", x, [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
    y = cv2.imdecode(buf, cv2.IMREAD_UNCHANGED)
    return y if img.ndim == 2 else cv2.cvtColor(y, cv2.COLOR_BGR2RGB)


def downscale(img, factor=0.5):
    """Loss of resolution: down-sample by `factor`, then up-sample back."""
    if factor >= 0.999:
        return img.copy()
    h, w = img.shape[:2]
    small = cv2.resize(img, (max(1, int(w * factor)), max(1, int(h * factor))), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)


def rolling_shutter(img, amplitude=6.0, cycles=1.5, phase=0.0):
    """Rolling-shutter distortion: every row is shifted by a sinusoid of the row number."""
    if amplitude <= 0:
        return img.copy()
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    shift = amplitude * np.sin(2 * np.pi * cycles * yy / h + phase)
    return cv2.remap(img, xx + shift, yy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)


def analog_interference(img, strength=0.5, seed=None):
    """Analog FPV video link with a weak or jammed signal: colour loss, snow, line
    noise, horizontal tearing (loss of synchronisation) and a rolling brightness bar."""
    if strength <= 0:
        return img.copy()
    rng = np.random.default_rng(seed)
    h = img.shape[0]
    x = _f(img)
    if x.ndim == 3:
        gray = x.mean(axis=2, keepdims=True)
        x = gray + (x - gray) * (1 - 0.85 * min(strength * 1.3, 1))
    x += rng.normal(0, 45 * strength, x.shape)
    for r in rng.integers(0, h, int(h * 0.06 * strength)):
        x[r] += rng.normal(70, 40) * strength
    if strength > 0.25:
        y0, band = rng.integers(0, h), int(h * 0.25 * strength)
        offs = np.cumsum(rng.normal(0, 3.0 * strength, band))
        for i in range(band):
            r = (y0 + i) % h
            x[r] = np.roll(x[r], int(offs[i] + 30 * strength), axis=0)
    bar = 1.0 + 0.3 * strength * np.sin(2 * np.pi * (np.arange(h, dtype=np.float32)[:, None] / h * 1.3 + rng.random()))
    return _u8(x * (bar if x.ndim == 2 else bar[..., None]))


def packet_loss(img, rate=0.05, block=16, seed=None):
    """Digital link with lost packets: damaged macroblock slices are filled with a smear
    of the row above or with a grey block."""
    if rate <= 0:
        return img.copy()
    rng = np.random.default_rng(seed)
    out = img.copy()
    gh, gw = img.shape[0] // block, img.shape[1] // block
    lost = rng.random((gh, gw)) < rate
    for r in range(gh):
        c = 0
        while c < gw:
            if lost[r, c]:
                run = int(rng.integers(1, max(2, gw // 3)))
                lost[r, c:c + run] = True
                c += run
            c += 1
    for r, c in zip(*np.nonzero(lost)):
        ys, xs = slice(r * block, (r + 1) * block), slice(c * block, (c + 1) * block)
        if r > 0 and rng.random() < 0.6:
            out[ys, xs] = out[r * block - 1:r * block, xs]
        else:
            out[ys, xs] = 128 if img.ndim == 2 else (60, 128, 60)
    return out


def degrade(img, sensor="visible", motion_px=0.0, motion_angle=0.0, defocus_sigma=0.0, light=1.0, noise=0.0,
            haze_density=0.0, smoke_density=0.0, rolling=0.0, scale=1.0, jpeg_quality=100, analog=0.0,
            packet=0.0, seed=0):
    """Chain of degradations in physical order: atmosphere -> optics and motion -> sensor ->
    resolution -> compression -> radio link. For the thermal sensor haze and smoke are
    attenuated (long-wave infrared passes thin smoke and haze) and low light does not apply."""
    x, ir = img, sensor == "thermal"
    if haze_density > 0:
        x = haze(x, haze_density * (0.25 if ir else 1.0), airlight=128 if ir else 200)
    if smoke_density > 0:
        x = smoke(x, smoke_density, seed=seed, color=128 if ir else 170, opacity=0.18 if ir else 0.92)
    if motion_px > 0:
        x = motion_blur(x, motion_px, motion_angle)
    if defocus_sigma > 0:
        x = defocus(x, defocus_sigma)
    if rolling > 0:
        x = rolling_shutter(x, rolling)
    if light < 1.0 and not ir:
        x = low_light(x, light, seed=seed)
    if noise > 0:
        x = add_noise(x, noise, seed=seed + 1)
    if scale < 1.0:
        x = downscale(x, scale)
    if jpeg_quality < 100:
        x = jpeg(x, jpeg_quality)
    if analog > 0:
        x = analog_interference(x, analog, seed=seed + 2)
    if packet > 0:
        x = packet_loss(x, packet, seed=seed + 3)
    return x


# =============================================================================
# 5. Measures, classical detection, tracking
# =============================================================================
def gray(img):
    return img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)


def psnr(ref, test):
    """Peak signal-to-noise ratio, dB."""
    mse = float(np.mean((_f(ref) - _f(test)) ** 2))
    return float("inf") if mse == 0 else 10 * math.log10(255.0 ** 2 / mse)


def ssim(ref, test):
    """Structural similarity index of grey levels (Gaussian window 11 x 11, sigma 1.5)."""
    a, b = _f(gray(ref)), _f(gray(test))
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    blur = lambda z: cv2.GaussianBlur(z, (11, 11), 1.5)
    ma, mb = blur(a), blur(b)
    va, vb, cab = blur(a * a) - ma * ma, blur(b * b) - mb * mb, blur(a * b) - ma * mb
    return float((((2 * ma * mb + c1) * (2 * cab + c2)) / ((ma * ma + mb * mb + c1) * (va + vb + c2))).mean())


def sharpness(img):
    """Variance of the Laplacian: a measure of focus (small values = blurred image)."""
    return float(cv2.Laplacian(gray(img), cv2.CV_64F).var())


def to_white_hot(img, palette):
    """Thermal frame in the white-hot convention (warm = bright) as a 3-channel uint8 image.
    palette: "white-hot" (HIT-UAV), "black-hot" (M3OT, inverted here) or a pseudo-colour palette
    whose luminance grows with the temperature (Kust4K: purple = cold, yellow = warm)."""
    g = gray(img)
    if palette.startswith("black"):
        g = 255 - g
    return np.dstack([g, g, g])


def vegetation_mask(rgb, exg_min=0.12, v_min=35, erode_px=15):
    """Vegetation of a visible RGB frame by the excess-green index ExG = (2G - R - B) / (R + G + B).
    The mask is eroded by erode_px so that a residual offset between the visible and the thermal
    channel does not move it onto neighbouring surfaces."""
    f = rgb.astype(np.float32)
    r, g, b = f[..., 0], f[..., 1], f[..., 2]
    exg = (2 * g - r - b) / (r + g + b + 1e-6)
    m = ((exg > exg_min) & (f.max(axis=2) > v_min)).astype(np.uint8)
    if erode_px > 1:
        m = cv2.erode(m, np.ones((erode_px, erode_px), np.uint8))
    return m.astype(bool)


def surroundings_mask(shape, boxes, scale=1.0, gap=3):
    """Background ring around annotated objects (e.g. the road surface around vehicles): every box
    [x1, y1, x2, y2] is expanded by scale x its longer side; the boxes themselves, expanded by gap px,
    are excluded."""
    ring = np.zeros(shape[:2], np.uint8)
    core = np.zeros(shape[:2], np.uint8)
    for x1, y1, x2, y2 in boxes:
        s = int(round(scale * max(x2 - x1, y2 - y1)))
        cv2.rectangle(ring, (int(x1) - s, int(y1) - s), (int(x2) + s, int(y2) + s), 1, -1)
        cv2.rectangle(core, (int(x1) - gap, int(y1) - gap), (int(x2) + gap, int(y2) + gap), 1, -1)
    return (ring > 0) & (core == 0)


def palette_check(thermal, road, vegetation, margin=15, min_pixels=200):
    """Palette of a thermal frame established from the scene. In a road scene the asphalt is, as a
    rule, warmer than vegetation (by day it absorbs most of the solar radiation, at night it gives off
    the stored heat), therefore a road darker than vegetation indicates the black-hot palette
    (warm = dark) and a brighter road the white-hot convention (warm = bright). A shaded or wet road
    and sunlit tree crowns can reverse the order in single frames: check several frames."""
    g = gray(thermal)
    road = road & ~vegetation
    nr, nv = int(road.sum()), int(vegetation.sum())
    r = float(np.median(g[road])) if nr >= min_pixels else float("nan")
    v = float(np.median(g[vegetation])) if nv >= min_pixels else float("nan")
    if np.isnan(r) or np.isnan(v):
        verdict = "not determined: too few road or vegetation pixels"
    elif r < v - margin:
        verdict = "road darker than vegetation: black-hot (warm = dark)"
    elif r > v + margin:
        verdict = "road brighter than vegetation: white-hot (warm = bright)"
    else:
        verdict = f"difference below {margin} grey levels: not determined"
    return dict(road_median=r, vegetation_median=v, road_pixels=nr, vegetation_pixels=nv, verdict=verdict)


def signal_to_clutter(img, mask, exclude=None, gap=2, ring=8):
    """Signal-to-clutter ratio of an object: |mean of object - mean of background ring| /
    standard deviation of the background ring (grey levels)."""
    g = _f(gray(img))
    m = mask.astype(np.uint8)
    k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * gap + 1, 2 * gap + 1))
    k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * (gap + ring) + 1, 2 * (gap + ring) + 1))
    bg = (cv2.dilate(m, k2) > 0) & (cv2.dilate(m, k1) == 0)
    if exclude is not None:
        bg &= ~exclude
    t, b = g[m > 0], g[bg]
    if t.size < 3 or b.size < 10:
        return float("nan")
    return abs(float(t.mean()) - float(b.mean())) / max(float(b.std()), 1.0)


def box_iou(a, b):
    """Intersection over union of two boxes [x1, y1, x2, y2]."""
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def boxes_from_labels(labels, class_ids, names, min_area=20):
    """Boxes of connected regions of the given classes in a label map."""
    out = []
    for c in class_ids:
        n, cc, stats, _ = cv2.connectedComponentsWithStats((labels == c).astype(np.uint8), connectivity=8)
        for i in range(1, n):
            x, y, w, h, a = stats[i]
            if a >= min_area:
                out.append(dict(cls=names[c], class_id=int(c), box=[float(x), float(y), float(x + w), float(y + h)],
                                area=int(a)))
    return out


def patch_labels(labels, tile, object_ids, names, min_px=10):
    """Patch classification from a label map: every tile receives the object class with
    the largest number of pixels in it, or 'background'."""
    rows, cols = labels.shape[0] // tile, labels.shape[1] // tile
    grid = np.empty((rows, cols), object)
    for r in range(rows):
        for c in range(cols):
            t = labels[r * tile:(r + 1) * tile, c * tile:(c + 1) * tile]
            n, k = max((int((t == k).sum()), k) for k in object_ids)
            grid[r, c] = names[k] if n >= min_px else "background"
    return grid


def detect_hotspots(img, k=5.0, se_px=21, min_area=4, max_area=5000):
    """Classical detector of small warm objects in a white-hot thermal image: white
    top-hat (local contrast) -> threshold mean + k std -> connected components.
    Returns boxes (N, 4) [x1, y1, x2, y2] and the binary mask."""
    g = cv2.GaussianBlur(gray(img), (3, 3), 0)
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(se_px) | 1, int(se_px) | 1))
    th = cv2.morphologyEx(g, cv2.MORPH_TOPHAT, se).astype(np.float32)
    mask = (th > th.mean() + k * th.std()).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    boxes = [[x, y, x + w, y + h] for x, y, w, h, a in stats[1:] if min_area <= a <= max_area]
    return np.array(boxes, float).reshape(-1, 4), mask


def expected_size_px(gsd, size_m=0.45, margin_px=6.0):
    """Expected size of an object in an annotated box, px: size_m / GSD + margin_px.
    Defaults: nominal shoulder width of a person 0.45 m and a margin of 6 px (blur, thermal halo,
    annotation margin). For comparison, a least-squares fit to the HIT-UAV boxes of persons in
    vertical views gives d = 0.38 m and m = 6.6 px; block 1.1.1-B computes the fit for every class."""
    return size_m / gsd + margin_px


def detect_warm_objects(img, gsd, k=6.0, size_m=0.45, margin_px=6.0):
    """Hot-spot detector adapted to the ground sampling distance: the structuring element
    and the admissible blob areas follow the expected size of a person in pixels."""
    s = expected_size_px(gsd, size_m, margin_px)
    return detect_hotspots(img, k=k, se_px=int(max(7, round(2.5 * s))) | 1,
                           min_area=max(3, int((0.3 * s) ** 2)), max_area=int((2.0 * s) ** 2))


def match_detections(pred, objects, target_classes, pad=2.0):
    """A detection is correct when its centre lies inside a (padded) box of an object of a
    target class. Detections inside boxes of other annotated objects are counted apart.
    Returns dict(tp, fp, other, missed, recall, precision)."""
    tgt = [o for o in objects if o["cls"] in target_classes]
    oth = [o for o in objects if o["cls"] not in target_classes]
    used, tp, fp, other = set(), [], [], []
    inside = lambda o, cx, cy: (o["box"][0] - pad <= cx <= o["box"][2] + pad and o["box"][1] - pad <= cy <= o["box"][3] + pad)
    for b in pred:
        cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
        hit = next((i for i, o in enumerate(tgt) if i not in used and inside(o, cx, cy)), None)
        if hit is not None:
            used.add(hit)
            tp.append(list(b))
        elif any(inside(o, cx, cy) for o in oth) or any(inside(o, cx, cy) for o in tgt):
            other.append(list(b))
        else:
            fp.append(list(b))
    n = len(tgt)
    return dict(tp=tp, fp=fp, other=other, missed=[o for i, o in enumerate(tgt) if i not in used],
                recall=len(tp) / n if n else float("nan"), precision=len(tp) / max(len(tp) + len(fp), 1))


def global_shift(prev, cur):
    """Translation of the image content from `prev` to `cur` (phase correlation), px."""
    (dx, dy), _ = cv2.phaseCorrelate(_f(gray(prev)), _f(gray(cur)))
    return dx, dy


class NearestNeighbourTracker:
    """Simple multi-object tracker: the detections of a frame are linked to the tracks of
    the previous frame by the smallest distance after compensation of the camera motion.
    A track is closed after `max_missed` frames without a detection."""

    def __init__(self, max_dist=12.0, max_missed=4):
        self.max_dist, self.max_missed = max_dist, max_missed
        self.tracks, self.closed, self.next_id = {}, {}, 1
        self.offset = np.zeros(2)                    # accumulated camera motion, px

    def update(self, centres, shift=(0.0, 0.0), frame=0):
        """centres: (N, 2) detections of the frame; shift: image motion since the previous frame.
        Returns the list of (track id, centre) of the frame."""
        self.offset += np.asarray(shift, float)
        pts = np.asarray(centres, float).reshape(-1, 2) - self.offset      # coordinates of the first frame
        ids = list(self.tracks)
        pairs = []
        for i, tid in enumerate(ids):
            last = self.tracks[tid]["hist"][-1][1]
            for j, p in enumerate(pts):
                d = float(np.hypot(*(p - last)))
                if d <= self.max_dist:
                    pairs.append((d, tid, j))
        pairs.sort()
        used_t, used_d, out = set(), set(), []
        for d, tid, j in pairs:
            if tid in used_t or j in used_d:
                continue
            used_t.add(tid)
            used_d.add(j)
            self.tracks[tid]["hist"].append((frame, pts[j]))
            self.tracks[tid]["missed"] = 0
            out.append((tid, pts[j] + self.offset))
        for tid in ids:
            if tid not in used_t:
                self.tracks[tid]["missed"] += 1
                if self.tracks[tid]["missed"] > self.max_missed:
                    self.closed[tid] = self.tracks.pop(tid)
        for j, p in enumerate(pts):
            if j not in used_d:
                self.tracks[self.next_id] = dict(hist=[(frame, p)], missed=0)
                out.append((self.next_id, p + self.offset))
                self.next_id += 1
        return out

    def history(self, tid):
        """Trajectory of a track in the coordinates of the current frame."""
        t = self.tracks.get(tid) or self.closed.get(tid)
        return np.array([p for _, p in t["hist"]]) + self.offset


def track_warm_objects(frames, gsd, fps, k=5.0, max_speed_ms=2.5, max_missed=4, min_len=12, min_disp_m=2.0):
    """Detection and tracking of warm objects in a thermal video:
    detect_warm_objects -> camera motion by phase correlation -> NearestNeighbourTracker.
    A track is reported as a moving object when it lasts at least `min_len` frames and its
    start and end points are at least `min_disp_m` apart on the ground.
    Returns (tracker, moving: {id: trajectory (K, 2) in the last frame}, rows: list of
    (frame, track id, x, y), detections per frame)."""
    tr = NearestNeighbourTracker(max_dist=max(5.0, max_speed_ms / gsd / max(fps, 1e-6) + 3.0), max_missed=max_missed)
    prev, rows, n_det = None, [], []
    for i, f in enumerate(frames):
        g = gray(f)
        b, _ = detect_warm_objects(g, gsd, k=k)
        c = np.c_[(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2] if len(b) else np.zeros((0, 2))
        shift = global_shift(prev, g) if prev is not None else (0.0, 0.0)
        for tid, p in tr.update(c, shift, i):
            rows.append((i, tid, float(p[0]), float(p[1])))
        n_det.append(len(b))
        prev = g
    moving = {}
    for tid, t in {**tr.closed, **tr.tracks}.items():
        pts = np.array([p for _, p in t["hist"]])
        if len(pts) >= min_len and float(np.hypot(*(pts[-1] - pts[0]))) * gsd >= min_disp_m:
            moving[tid] = pts + tr.offset
    return tr, moving, rows, n_det


def video_info(path):
    """Properties of a video file read by OpenCV."""
    cap = cv2.VideoCapture(path)
    info = dict(path=path, opened=cap.isOpened(), fps=cap.get(cv2.CAP_PROP_FPS),
                frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                codec=int(cap.get(cv2.CAP_PROP_FOURCC)).to_bytes(4, "little").decode("latin-1", "replace"))
    cap.release()
    info["duration_s"] = info["frames"] / info["fps"] if info["fps"] else float("nan")
    info["size_mb"] = os.path.getsize(path) / 1e6 if os.path.exists(path) else float("nan")
    return info


def iter_video(path, every_n=1, max_frames=None, start=0):
    """Generator of (frame number, timestamp s, RGB frame) of every n-th frame of a video:
    frames are decoded one by one, so a long video does not fill the memory."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 1.0
    if start:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    k, n_out = start, 0
    try:
        while True:
            ok, f = cap.read()
            if not ok or (max_frames is not None and n_out >= max_frames):
                break
            if (k - start) % every_n == 0:
                yield k, k / fps, cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
                n_out += 1
            k += 1
    finally:
        cap.release()


def write_video(path, frames, fps, codec="mp4v"):
    """Write RGB or grey frames to a video file; returns the path. Codec 'mp4v' (.mp4) works in
    Colab and in most OpenCV builds; 'MJPG' (.avi) stores every frame as JPEG."""
    h, w = frames[0].shape[:2]
    wr = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*codec), float(fps), (w, h))
    if not wr.isOpened():
        raise IOError(f"OpenCV cannot write the codec {codec}")
    for f in frames:
        f = to_rgb(f)
        if f.shape[:2] != (h, w):
            f = cv2.resize(f, (w, h))
        wr.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
    wr.release()
    return path


def contact_sheet(frames, cols=6, width=1600, labels=None, font=0.5):
    """Grid of frames for a quick visual review of a sequence."""
    n = len(frames)
    rows = int(math.ceil(n / cols))
    tw = width // cols
    th = int(round(tw * frames[0].shape[0] / frames[0].shape[1]))
    sheet = np.zeros((rows * th, cols * tw, 3), np.uint8)
    for i, f in enumerate(frames):
        r, c = divmod(i, cols)
        t = cv2.resize(to_rgb(f), (tw, th), interpolation=cv2.INTER_AREA)
        if labels is not None:
            cv2.rectangle(t, (0, 0), (tw, 18), (0, 0, 0), -1)
            cv2.putText(t, str(labels[i]), (4, 13), cv2.FONT_HERSHEY_SIMPLEX, font * 0.8, (255, 255, 0), 1, cv2.LINE_AA)
        sheet[r * th:(r + 1) * th, c * tw:(c + 1) * tw] = t
    return sheet


def frame_difference(a, b):
    """Mean absolute difference of the grey levels of two frames."""
    return float(np.mean(np.abs(_f(gray(a)) - _f(gray(b)))))


def grad_magnitude(img, sigma=1.5):
    """Normalised gradient magnitude; edges of a visible and a thermal image of the same scene
    are similar even when their brightness is not, so it is used to compare the two channels."""
    g = cv2.GaussianBlur(_f(gray(img)), (0, 0), sigma)
    m = np.hypot(cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1))
    return (m - m.mean()) / (m.std() + 1e-6)


def channel_shift(img_a, img_b):
    """Translation between two images of different sensors (phase correlation of gradient
    magnitudes): (dx, dy, response). The response is low when the images have little common structure."""
    a, b = grad_magnitude(img_a), grad_magnitude(img_b)
    win = cv2.createHanningWindow((a.shape[1], a.shape[0]), cv2.CV_32F)
    (dx, dy), resp = cv2.phaseCorrelate(a, b, win)
    return float(dx), float(dy), float(resp)


def track_offsets(table_a, table_b, lag=0):
    """Offsets of the centres of the same track (same id) in two annotated channels:
    channel b at frame t is compared with channel a at frame t + lag. Returns arrays dx, dy."""
    bb = table_b[["id", "frame", "cx", "cy"]].copy()
    bb["frame"] = bb["frame"] + lag
    m = bb.merge(table_a[["id", "frame", "cx", "cy"]], on=["id", "frame"], suffixes=("_b", "_a"))
    return (m.cx_b - m.cx_a).values, (m.cy_b - m.cy_a).values


def read_video(path, step=1, max_frames=None, start=0):
    """Frames of a video file (BGR from OpenCV converted to RGB) and the frame rate."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames, k = [], 0
    while True:
        ok, f = cap.read()
        if not ok or (max_frames and len(frames) >= max_frames):
            break
        if k % step == 0:
            frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        k += 1
    cap.release()
    return frames, fps


# =============================================================================
# 6. Visualisation
# =============================================================================
def to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if img.ndim == 2 else img


def colorize(labels, colors):
    """Label map -> RGB image with the colour table `colors` (array K x 3)."""
    return np.asarray(colors, np.uint8)[np.clip(labels, 0, len(colors) - 1)]


def overlay(img, color_img, alpha=0.5, mask=None):
    base, col = _f(to_rgb(img)), _f(color_img)
    a = np.full(base.shape[:2], alpha, np.float32) if mask is None else mask.astype(np.float32) * alpha
    return _u8(base * (1 - a[..., None]) + col * a[..., None])


def draw_boxes(img, objects, colors=None, labels=True, scores=None, ids=None, thickness=1, font=0.4, color=None):
    """Draw boxes of objects (dicts with 'box' and 'cls')."""
    out = to_rgb(img).copy()
    for i, o in enumerate(objects):
        x1, y1, x2, y2 = [int(round(v)) for v in o["box"]]
        c = color if color is not None else (colors or {}).get(o["cls"], (255, 255, 0))
        c = tuple(int(v) for v in c)
        cv2.rectangle(out, (x1, y1), (x2, y2), c, thickness, cv2.LINE_AA)
        txt = []
        if labels:
            txt.append(str(o["cls"]))
        if ids is not None:
            txt.append(f"#{ids[i]}")
        if scores is not None:
            txt.append(f"{scores[i]:.2f}")
        if txt:
            s = " ".join(txt)
            (tw, th), _ = cv2.getTextSize(s, cv2.FONT_HERSHEY_SIMPLEX, font, 1)
            y0 = max(y1 - 3, th + 2)
            cv2.rectangle(out, (x1, y0 - th - 2), (x1 + tw + 2, y0 + 2), c, -1)
            cv2.putText(out, s, (x1 + 1, y0), cv2.FONT_HERSHEY_SIMPLEX, font, (0, 0, 0), 1, cv2.LINE_AA)
    return out


def legend(ax, names, colors, loc="lower right", fontsize=7, ncol=1):
    import matplotlib.patches as mpatches
    ax.legend(handles=[mpatches.Patch(color=np.array(c) / 255.0, label=n) for n, c in zip(names, colors)],
              loc=loc, fontsize=fontsize, ncol=ncol, framealpha=0.85)


def setup_plots():
    import matplotlib.pyplot as plt
    plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 8,
                         "ytick.labelsize": 8, "legend.fontsize": 8, "figure.facecolor": "white"})


def catalogue():
    """Table of the data sets of the course."""
    return [
        ("HIT-UAV", "UAV DJI M210 V2, thermal camera Zenmuse XT2 (640x512, 25 x 20 deg)",
         "2898 thermal images, 5 thermal videos; boxes of persons and vehicles; 60-130 m, day and night",
         "CC BY 4.0", "Suo J. et al., Scientific Data 10, 227 (2023)",
         "https://github.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset"),
        ("Kust4K", "UAV hexacopter HET-3021C, visible + thermal cameras",
         "4024 pixel-aligned RGB-thermal pairs 640x512 over urban roads, day and night; pixel labels of 8 classes",
         "CC BY 4.0", "Scientific Data 12, 1701 (2025)", "https://doi.org/10.6084/m9.figshare.29476610"),
        ("VisDrone2019-DET (val)", "camera drones of different types, 14 Chinese cities",
         "548 RGB images; boxes of pedestrians, cars, vans, trucks, buses, motors and others",
         "research use, see the data set site", "Zhu P. et al., IEEE TPAMI 44(11), 7380-7399 (2022)",
         "https://github.com/VisDrone/VisDrone-Dataset"),
        ("GOOSE (val)", "ground vehicle, front RGB camera 2048x1000",
         "962 images of roads, field tracks, forests and a training area; pixel labels of 64 classes",
         "CC BY-SA 4.0", "IEEE ICRA 2024 (arXiv:2310.16788)", "https://goose-dataset.de"),
        ("Military vehicles", "photographs from web sites (ground, side and oblique views)",
         "9444 images of 24 types of military vehicles, 7 groups", "MIT (declared by the authors)",
         "Kricheli J. S. et al., arXiv:2407.15192 (2024)", "https://huggingface.co/datasets/leibnitz-lab/military_vehicles"),
        ("SULAND_v2", "handheld camera iPhone 13 moved over the ground (view of a ground robot)",
         "33771 frames 1440x1920 of 55 sequences; boxes of surrogates of the landmines PFM-1 and PMA-2",
         "CC BY 4.0", "Vivoli E. et al., Remote Sensing 16(4), 677 (2024); Lekhak S. et al., arXiv:2607.28996 (2026)",
         "https://huggingface.co/datasets/SagarLekhak/SULAND_v2_RGB_Surface_Landmine_Dataset"),
    ]


# =====================================================================================
# Topic 2. Digital image, video frame formation and preprocessing
# =====================================================================================
def quantize(img, bits):
    """Brightness quantised to `bits` bits per channel (2**bits levels), shown again in 0...255."""
    step = 256.0 / (2 ** int(bits))
    return np.clip(np.floor(img.astype(np.float32) / step) * step + step / 2, 0, 255).astype(np.uint8)


def entropy(img):
    """Entropy of the brightness histogram, bit (8 bit = all 256 levels used equally)."""
    hist = np.bincount(gray(img).ravel(), minlength=256).astype(float)
    p = hist[hist > 0] / hist.sum()
    return float(-(p * np.log2(p)).sum())


def rms_contrast(img):
    """RMS contrast: standard deviation of the grey levels."""
    return float(_f(gray(img)).std())


def average_gradient(img):
    """Average gradient: mean of sqrt((gx^2 + gy^2) / 2) of the grey levels (central differences)."""
    g = _f(gray(img))
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=1)
    gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=1)
    return float(np.sqrt((gx ** 2 + gy ** 2) / 2.0).mean())


def hsv_mask(rgb, h=(35, 85), s=(40, 255), v=(40, 255)):
    """Pixels whose hue (degrees, 0...360), saturation and value (0...255) lie in the given ranges.
    OpenCV stores the hue of 8-bit images as degrees / 2 (0...179); a range with h[0] > h[1]
    passes through 0 degrees (red)."""
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    hh = hsv[..., 0].astype(np.int32) * 2
    mh = ((hh >= h[0]) & (hh <= h[1])) if h[0] <= h[1] else ((hh >= h[0]) | (hh <= h[1]))
    return mh & (hsv[..., 1] >= s[0]) & (hsv[..., 1] <= s[1]) & (hsv[..., 2] >= v[0]) & (hsv[..., 2] <= v[1])


def mask_scores(pred, truth, valid=None):
    """IoU, precision and recall of a binary mask against a reference mask; pixels outside `valid` are ignored."""
    if valid is not None:
        pred, truth = pred & valid, truth & valid
    tp = int((pred & truth).sum())
    fp = int((pred & ~truth).sum())
    fn = int((~pred & truth).sum())
    return dict(iou=tp / max(tp + fp + fn, 1), precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1))


def box_mask(shape, box):
    """Binary mask of a box [x1, y1, x2, y2]."""
    m = np.zeros(shape[:2], bool)
    x1, y1, x2, y2 = (int(round(v)) for v in box)
    m[max(0, y1):max(0, y2), max(0, x1):max(0, x2)] = True
    return m


def colour_planes(rgb, thermal=None):
    """8-bit planes of the colour models of a frame: R, G, B, grey (luma Y = 0.299 R + 0.587 G + 0.114 B),
    H (degrees / 2), S, V and, if given, the thermal channel."""
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    planes = {"R": rgb[..., 0], "G": rgb[..., 1], "B": rgb[..., 2], "grey": cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY),
              "H": hsv[..., 0], "S": hsv[..., 1], "V": hsv[..., 2]}
    if thermal is not None:
        planes["thermal"] = gray(thermal)
    return planes


def wide_angle_lens(img, barrel=0.2, vignetting=0.0):
    """Wide-angle lens of an FPV camera: barrel distortion and vignetting. The output pixel at the
    normalised radius r (r = 1 in the corners) shows the point of the ideal image at r (1 + b r^2) / (1 + b):
    the corners stay in place, the centre is magnified by 1 + b, straight lines bend outwards.
    OpenCV describes the same distortion with a negative coefficient k1 of its camera model.
    Vignetting: the brightness is multiplied by 1 - v r^2."""
    h, w = img.shape[:2]
    cx, cy = (w - 1) / 2.0, (h - 1) / 2.0
    rmax = math.hypot(cx, cy)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    dx, dy = (xx - cx) / rmax, (yy - cy) / rmax
    r2 = dx * dx + dy * dy
    f = (1 + barrel * r2) / (1 + barrel)
    out = cv2.remap(img, (cx + dx * f * rmax).astype(np.float32), (cy + dy * f * rmax).astype(np.float32),
                    cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    if vignetting > 0:
        v = np.clip(1 - vignetting * r2, 0, 1)
        out = _u8(_f(out) * (v if out.ndim == 2 else v[..., None]))
    return out


def free_space_loss_db(distance_m, freq_mhz):
    """Free-space path loss, dB: 20 lg(d, km) + 20 lg(f, MHz) + 32.44."""
    return 20 * math.log10(max(distance_m, 1.0) / 1000.0) + 20 * math.log10(freq_mhz) + 32.44


def link_budget(p_tx_mw, distance_m, freq_mhz=5800.0, g_tx_dbi=2.0, g_rx_dbi=2.0, extra_loss_db=0.0,
                bandwidth_mhz=20.0, noise_figure_db=6.0, interference_db=0.0):
    """Line-of-sight radio link: received power P_rx = P_tx + G_tx + G_rx - L_fs - L_add, dBm; noise power
    N = -174 + 10 lg(B, Hz) + NF, dBm, raised by `interference_db` (jamming); carrier-to-noise ratio CNR = P_rx - N."""
    p_tx = 10 * math.log10(p_tx_mw)
    fspl = free_space_loss_db(distance_m, freq_mhz)
    p_rx = p_tx + g_tx_dbi + g_rx_dbi - fspl - extra_loss_db
    noise = -174.0 + 10 * math.log10(bandwidth_mhz * 1e6) + noise_figure_db + interference_db
    return dict(p_tx_dbm=p_tx, fspl_db=fspl, p_rx_dbm=p_rx, noise_dbm=noise, cnr_db=p_rx - noise)


def analog_link_frame(img, cnr_db, threshold_db=10.0, fm_gain_db=25.0, seed=0):
    """Frame at the output of an analog FM video receiver (ILLUSTRATIVE model of the course, not a standard):
    above the FM threshold the picture has Gaussian noise with SNR = CNR + fm_gain_db; below the threshold
    impulse noise ("sparkles"), loss of colour and loss of line synchronisation grow with the deficit
    threshold - CNR; at a deficit of 6 dB and more the picture is replaced by noise."""
    rng = np.random.default_rng(seed)
    x = _f(img)
    x = x + rng.normal(0, 255.0 * 10 ** (-(cnr_db + fm_gain_db) / 20.0), x.shape)
    deficit = threshold_db - cnr_db
    if deficit >= 6:
        return _u8(rng.normal(128, 60, img.shape))
    if deficit > 0:
        imp = rng.random(x.shape[:2]) < min(0.5, 0.012 * deficit ** 2)
        vals = rng.choice(np.array([0.0, 255.0], np.float32), size=int(imp.sum()))
        x[imp] = vals[:, None] if x.ndim == 3 else vals
        return analog_interference(_u8(x), strength=min(1.0, deficit / 6.0), seed=seed)
    return _u8(x)


def jpeg_for_budget(img, bytes_per_frame):
    """Highest JPEG quality (5...95) whose file of this frame fits into `bytes_per_frame`: intra-frame
    equivalent of a digital video link with a given bit rate. Returns (decoded frame, quality, bytes)."""
    bgr = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    best = None
    lo, hi = 5, 95
    while lo <= hi:
        q = (lo + hi) // 2
        ok, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, q])
        if len(buf) <= bytes_per_frame:
            best, lo = (q, buf), q + 1
        else:
            hi = q - 1
    if best is None:
        ok, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, 5])
        best = (5, buf)
    dec = cv2.imdecode(best[1], cv2.IMREAD_UNCHANGED)
    return (dec if dec.ndim == 2 else cv2.cvtColor(dec, cv2.COLOR_BGR2RGB)), best[0], len(best[1])


def digital_packet_loss_rate(cnr_db, required_db=8.0, width_db=0.4):
    """Packet loss rate of a digital link near its sensitivity (ILLUSTRATIVE logistic model of the "cliff
    effect"): 0.5 at CNR = required_db, 0.1 % at a margin of about 2.8 dB."""
    return float(1.0 / (1.0 + math.exp((cnr_db - required_db) / width_db)))


def noise_sigma(img, exclude_edges=0.1):
    """Standard deviation of the noise of a frame by the method of Immerkaer (1996): the frame is filtered with
    the mask [[1,-2,1],[-2,4,-2],[1,-2,1]], which suppresses the smooth structure of the scene, and
    sigma = sqrt(pi/2) / 6 * mean |filtered|. Edges and fine texture increase the estimate, therefore the
    share `exclude_edges` of the pixels with the strongest gradient of the smoothed frame is excluded.
    For a colour frame the estimate is the mean over the R, G, B channels (noise of one channel)."""
    if img.ndim == 3 and not (np.array_equal(img[..., 0], img[..., 1]) and np.array_equal(img[..., 1], img[..., 2])):
        return float(np.mean([noise_sigma(img[..., c], exclude_edges) for c in range(3)]))
    g = _f(gray(img))
    k = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)
    r = np.abs(cv2.filter2D(g, -1, k))[1:-1, 1:-1]
    if exclude_edges > 0:
        sm = cv2.GaussianBlur(g, (0, 0), 2.0)           # edges of the scene, not of the noise
        gx = cv2.Sobel(sm, cv2.CV_32F, 1, 0, ksize=3)[1:-1, 1:-1]
        gy = cv2.Sobel(sm, cv2.CV_32F, 0, 1, ksize=3)[1:-1, 1:-1]
        gm = gx * gx + gy * gy
        r = r[gm <= np.quantile(gm, 1 - exclude_edges)]
    return float(math.sqrt(math.pi / 2) * r.mean() / 6.0)


def add_gaussian_noise(img, sigma, seed=0):
    """Additive Gaussian noise with the standard deviation sigma (grey levels)."""
    rng = np.random.default_rng(seed)
    return _u8(_f(img) + rng.normal(0, sigma, img.shape))


def add_impulse_noise(img, amount, seed=0):
    """Impulse ("salt and pepper") noise: the share `amount` of the pixels becomes black or white."""
    rng = np.random.default_rng(seed)
    out = img.copy()
    m = rng.random(img.shape[:2]) < amount
    vals = rng.choice(np.array([0, 255], np.uint8), size=int(m.sum()))
    out[m] = vals[:, None] if img.ndim == 3 else vals
    return out


def equalize(img, method="clahe", clip=2.0, tiles=8):
    """Global histogram equalisation (method "he") or CLAHE ("clahe": tiles x tiles regions, histogram of every
    region clipped at clip x its mean height) of a grey frame or of the lightness L* of a colour frame (CIE L*a*b*)."""
    def one(g):
        if method == "he":
            return cv2.equalizeHist(g)
        return cv2.createCLAHE(clipLimit=float(clip), tileGridSize=(int(tiles), int(tiles))).apply(g)
    if img.ndim == 2:
        return one(img)
    if np.array_equal(img[..., 0], img[..., 1]) and np.array_equal(img[..., 1], img[..., 2]):
        g = one(img[..., 0])
        return np.dstack([g, g, g])
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    lab[..., 0] = one(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def letterbox(img, size=640, stride=32, auto=False, scaleup=True, color=114):
    """Letterbox transformation to the input of a detector: the frame is scaled by r = min(S / H, S / W)
    without a change of its aspect ratio and padded symmetrically with the grey value `color` to S x S
    (auto=True: only to the nearest multiple of `stride`). Returns the image, r and the padding (left, top)."""
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    if not scaleup:
        r = min(r, 1.0)
    nw, nh = int(round(w * r)), int(round(h * r))
    dw, dh = size - nw, size - nh
    if auto:
        dw, dh = dw % stride, dh % stride
    dw, dh = dw / 2.0, dh / 2.0
    if (nw, nh) != (w, h):
        img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    value = (color, color, color) if img.ndim == 3 else color
    return cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=value), r, (left, top)


def boxes_to_input(boxes, r, pad):
    """Boxes [x1, y1, x2, y2] of the frame mapped to the letterboxed input: x' = r x + left, y' = r y + top."""
    return [[b[0] * r + pad[0], b[1] * r + pad[1], b[2] * r + pad[0], b[3] * r + pad[1]] for b in boxes]


def boxes_from_input(boxes, r, pad):
    """Inverse mapping of boxes of the letterboxed input to the frame: x = (x' - left) / r, y = (y' - top) / r."""
    return [[(b[0] - pad[0]) / r, (b[1] - pad[1]) / r, (b[2] - pad[0]) / r, (b[3] - pad[1]) / r] for b in boxes]


def to_input_tensor(img):
    """Standard input of a neural network: RGB, float32 in 0...1, layout N x C x H x W (N = 1)."""
    x = to_rgb(img).astype(np.float32) / 255.0
    return np.ascontiguousarray(x.transpose(2, 0, 1)[None])


def time_call(fn, repeats=5, warmup=1):
    """Median execution time of fn() in milliseconds."""
    for _ in range(warmup):
        fn()
    t = []
    for _ in range(max(1, repeats)):
        t0 = time.perf_counter()
        fn()
        t.append((time.perf_counter() - t0) * 1000.0)
    return float(np.median(t))


def step_overlap(tracks, step):
    """Overlap and displacement of every annotated object between the frames t and t + step of a track table
    (columns frame, id, x, y, w, h): IoU of its two boxes, displacement of the centre (px) and box size."""
    import pandas as pd
    a = tracks[["frame", "id", "x", "y", "w", "h"]]
    b = a.copy()
    b["frame"] = b["frame"] - step
    m = a.merge(b, on=["id", "frame"], suffixes=("", "_n"))
    ix = (np.minimum(m.x + m.w, m.x_n + m.w_n) - np.maximum(m.x, m.x_n)).clip(lower=0)
    iy = (np.minimum(m.y + m.h, m.y_n + m.h_n) - np.maximum(m.y, m.y_n)).clip(lower=0)
    inter = ix * iy
    union = m.w * m.h + m.w_n * m.h_n - inter
    return pd.DataFrame({"frame": m.frame, "id": m.id, "iou": inter / union.clip(lower=1e-6),
                         "displacement_px": np.hypot(m.x_n + m.w_n / 2 - m.x - m.w / 2, m.y_n + m.h_n / 2 - m.y - m.h / 2),
                         "size_px": np.minimum(m.w, m.h)})


# =====================================================================================
# Topic 2, practical lessons 2.3-2.4: noise models, quality and suitability measures
# =====================================================================================
def add_poisson_noise(img, photons_per_level=2.0, seed=0):
    """Photon (shot) noise: every value I is replaced by Poisson(k I) / k with k photons per grey level,
    so the variance of the noise equals I / k and grows with the brightness (signal-dependent noise)."""
    rng = np.random.default_rng(seed)
    k = max(float(photons_per_level), 1e-3)
    return _u8(rng.poisson(_f(img) * k) / k)


def add_column_noise(img, sigma=8.0, seed=0, row_sigma=0.0):
    """Fixed-pattern noise of a sensor read out by columns (stripes typical of uncooled thermal
    cameras between two non-uniformity corrections): every column receives its own offset N(0, sigma^2),
    optionally every row an offset N(0, row_sigma^2)."""
    rng = np.random.default_rng(seed)
    h, w = img.shape[:2]
    off = rng.normal(0, sigma, (1, w)) + (rng.normal(0, row_sigma, (h, 1)) if row_sigma > 0 else 0)
    x = _f(img)
    return _u8(x + (off[..., None] if x.ndim == 3 else off))


def column_correction(img, smooth_px=15):
    """Correction of column stripes (scene-based non-uniformity correction, simplest form): the mean of
    every column is compared with a smoothed profile of the column means; the difference is the offset of
    the column and is subtracted. Smooth structures of the scene wider than smooth_px are kept."""
    x = _f(img)
    planes = [x] if x.ndim == 2 else [x[..., c] for c in range(x.shape[2])]
    out = []
    for p in planes:
        m = p.mean(axis=0)
        ms = cv2.GaussianBlur(m.reshape(1, -1).astype(np.float32), (0, 0), smooth_px).ravel()
        out.append(p - (m - ms)[None, :])
    return _u8(out[0] if x.ndim == 2 else np.dstack(out))


def edge_f1(ref, test, low=60, high=140, tol=1):
    """Agreement of the Canny edges of a test frame with the edges of a reference frame: an edge pixel is
    matched when the other map has an edge within tol pixels; returns dict(precision, recall, f1)."""
    e1 = cv2.Canny(gray(ref), low, high) > 0
    e2 = cv2.Canny(gray(test), low, high) > 0
    k = np.ones((2 * tol + 1, 2 * tol + 1), np.uint8)
    d1 = cv2.dilate(e1.astype(np.uint8), k) > 0
    d2 = cv2.dilate(e2.astype(np.uint8), k) > 0
    p = float((e2 & d1).sum()) / max(int(e2.sum()), 1)
    r = float((e1 & d2).sum()) / max(int(e1.sum()), 1)
    return dict(precision=p, recall=r, f1=2 * p * r / max(p + r, 1e-9))


def rank_correlation(a, b, sample=200000, seed=0):
    """Spearman rank correlation of the grey levels of two frames of the same scene (pixel by pixel).
    A global monotonic mapping of the brightness (stretching, gamma, global HE) gives 1 up to ties;
    a local mapping (CLAHE) changes the order of brightness between regions and gives less than 1."""
    x, y = gray(a).ravel().astype(np.float64), gray(b).ravel().astype(np.float64)
    if x.size > sample:
        idx = np.random.default_rng(seed).choice(x.size, sample, replace=False)
        x, y = x[idx], y[idx]
    def ranks(v):
        order = np.argsort(v, kind="mergesort")
        r = np.empty(v.size, np.float64)
        r[order] = np.arange(v.size)
        _, inv, cnt = np.unique(v, return_inverse=True, return_counts=True)
        sums = np.bincount(inv, weights=r)
        return (sums / cnt)[inv]                       # mean rank of tied values
    rx, ry = ranks(x), ranks(y)
    return float(np.corrcoef(rx, ry)[0, 1])


def hue_shift(ref, test, s_min=40, v_min=40):
    """Mean absolute change of the hue (degrees, circular) of the saturated pixels between two colour frames."""
    h1 = cv2.cvtColor(to_rgb(ref), cv2.COLOR_RGB2HSV)
    h2 = cv2.cvtColor(to_rgb(test), cv2.COLOR_RGB2HSV)
    m = (h1[..., 1] >= s_min) & (h1[..., 2] >= v_min)
    if not m.any():
        return float("nan")
    d = np.abs(h1[..., 0][m].astype(np.int32) - h2[..., 0][m].astype(np.int32)) * 2
    return float(np.minimum(d, 360 - d).mean())


def linear_stretch(img, p_low=1.0, p_high=99.0):
    """Global linear stretch of the brightness between two percentiles to 0...255 (the operation of the
    automatic gain control of a thermal camera); monotonic, keeps the order of brightness."""
    g = gray(img)
    lo, hi = np.percentile(g, [p_low, p_high])
    hi = max(hi, lo + 1)
    lut = np.clip((np.arange(256) - lo) * 255.0 / (hi - lo), 0, 255).astype(np.uint8)
    return cv2.LUT(img, lut)


def gamma_correction(img, gamma=0.6):
    """Global power-law mapping I' = 255 (I / 255)^gamma; gamma < 1 brightens the dark part of the range."""
    lut = np.clip(255.0 * (np.arange(256) / 255.0) ** float(gamma), 0, 255).astype(np.uint8)
    return cv2.LUT(img, lut)


def ambe(a, b):
    """Absolute mean brightness error |mean(b) - mean(a)| of the grey levels."""
    return abs(float(gray(b).mean()) - float(gray(a).mean()))


# =============================================================================
# 9. Topic 3: edges, binarisation, contours, ground geometry, FPV video
# =============================================================================

def label_boundaries(labels, ignore=(0,)):
    """Boundaries between regions of a label image (pixel labels of a data set): a pixel is a boundary
    pixel when one of its 4 neighbours has another label; pixels of the classes in `ignore` are excluded."""
    lab = np.asarray(labels)
    b = np.zeros(lab.shape, bool)
    b[:, 1:] |= lab[:, 1:] != lab[:, :-1]
    b[:, :-1] |= lab[:, 1:] != lab[:, :-1]
    b[1:, :] |= lab[1:, :] != lab[:-1, :]
    b[:-1, :] |= lab[1:, :] != lab[:-1, :]
    for c in ignore:
        b &= lab != c
    return b


def edge_map_scores(pred, ref, tol=2):
    """Agreement of two binary edge maps within `tol` pixels: precision (share of predicted edge pixels near a
    reference edge), recall (share of reference edge pixels near a predicted edge) and F1."""
    p, r = np.asarray(pred) > 0, np.asarray(ref) > 0
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * tol + 1, 2 * tol + 1))
    rd = cv2.dilate(r.astype(np.uint8), k) > 0
    pd_ = cv2.dilate(p.astype(np.uint8), k) > 0
    prec = float((p & rd).sum()) / max(int(p.sum()), 1)
    rec = float((r & pd_).sum()) / max(int(r.sum()), 1)
    return dict(precision=prec, recall=rec, f1=2 * prec * rec / max(prec + rec, 1e-9),
                edge_share=float(p.mean()))


def otsu(img, mask=None):
    """Otsu threshold computed by its definition: for every threshold t the between-class variance
    sigma_B^2(t) = w0(t) w1(t) (mu0(t) - mu1(t))^2 of the histogram is computed, and the threshold with the
    largest value is chosen. Returns (t, sigma_B^2 for t = 0...255, histogram)."""
    g = gray(img)
    v = g[mask > 0] if mask is not None else g.ravel()
    h = np.bincount(v.ravel(), minlength=256).astype(np.float64)
    p = h / max(h.sum(), 1)
    w0 = np.cumsum(p)
    mu = np.cumsum(p * np.arange(256))
    mt = mu[-1]
    w1 = 1 - w0
    with np.errstate(divide="ignore", invalid="ignore"):
        sb = (mt * w0 - mu) ** 2 / (w0 * w1)
    sb = np.nan_to_num(sb, nan=0.0, posinf=0.0)
    return int(np.argmax(sb)), sb, h


def oblique_gsd(altitude, angle, rows, height=512, vfov_deg=20.0, hfov_deg=25.0, width=640):
    """Ground sampling distance of a tilted camera in the image rows, m/px: (across, along).
    The depression angle of the line of sight of row y is delta = angle + (y - H/2)/H * vfov
    (angle = 90 deg for a vertical view); the slant range is h / sin(delta), the footprint of a pixel
    across the line of sight is range * IFOV and along it range * IFOV / sin(delta)."""
    rows = np.asarray(rows, float)
    dep = np.radians(np.clip(angle + (rows - height / 2.0) / height * vfov_deg, 1.0, 179.0))
    rng = altitude / np.sin(dep)
    across = rng * 2 * math.tan(math.radians(hfov_deg) / 2) / width
    return across, across / np.sin(dep)


def camera_matrix(width, height, hfov_deg, vfov_deg=None):
    """Pinhole camera matrix K from the size of the image and the field of view."""
    fx = width / 2.0 / math.tan(math.radians(hfov_deg) / 2)
    fy = fx if vfov_deg is None else height / 2.0 / math.tan(math.radians(vfov_deg) / 2)
    return np.array([[fx, 0, (width - 1) / 2.0], [0, fy, (height - 1) / 2.0], [0, 0, 1.0]])


def ground_homography(altitude, angle, K, size, gsd_out=None, max_range_m=None):
    """Homography that maps an image of a tilted camera to a top view of the flat ground (perspective
    transformation of Lesson 3.3). The camera is at the height h over the plane Z = 0 and looks forward
    with the depression angle `angle` (90 = vertical view); image x is the ground X axis.
    H_g = K [r1, r2, -h r3] maps the ground point (X, Y, 1) to the image; the returned matrix maps
    image pixels to pixels of the top view with the step gsd_out, m/px (forward = up).
    Returns (H, (width, height) of the top view, gsd_out, extent (Xmin, Xmax, Ymin, Ymax) in metres)."""
    th = math.radians(angle)
    R = np.array([[1, 0, 0], [0, -math.sin(th), -math.cos(th)], [0, math.cos(th), -math.sin(th)]], float)
    Hg = K @ np.column_stack([R[:, 0], R[:, 1], -altitude * R[:, 2]])
    Hi = np.linalg.inv(Hg)
    w, h = size
    corners = np.array([[0, 0, 1], [w - 1, 0, 1], [w - 1, h - 1, 1], [0, h - 1, 1]], float).T
    g = Hi @ corners
    X, Y = g[0] / g[2], g[1] / g[2]
    if max_range_m:
        Y = np.minimum(Y, max(float(max_range_m), float(Y.min()) + 20.0))    # at least 20 m beyond the nearest row
    if gsd_out is None:
        gsd_out = altitude / math.sin(th) / K[0, 0]          # GSD across the line of sight at the image centre
    x0, x1, y0, y1 = X.min(), X.max(), Y.min(), Y.max()
    W, Ht = int(math.ceil((x1 - x0) / gsd_out)), int(math.ceil((y1 - y0) / gsd_out))
    S = np.array([[1 / gsd_out, 0, -x0 / gsd_out], [0, -1 / gsd_out, y1 / gsd_out], [0, 0, 1]])
    return S @ Hi, (W, Ht), gsd_out, (x0, x1, y0, y1)


def transform_boxes(boxes, H):
    """Boxes [x1, y1, x2, y2] after the perspective transformation H (box of the four mapped corners)."""
    out = []
    for b in boxes:
        c = np.array([[b[0], b[1], 1], [b[2], b[1], 1], [b[2], b[3], 1], [b[0], b[3], 1]], float).T
        m = H @ c
        x, y = m[0] / m[2], m[1] / m[2]
        out.append([x.min(), y.min(), x.max(), y.max()])
    return np.array(out, float).reshape(-1, 4)


def thermal_mask(img, polarity="warm", method="top-hat + Otsu", se_px=31, k=4.0, block=51, offset=-5):
    """Binary mask of thermal signatures. polarity: 'warm' (brighter than the surroundings in a white-hot
    frame), 'cold' (darker) or 'both'. method: 'Otsu (global)', 'top-hat + Otsu', 'top-hat + mean + k*std',
    'adaptive (local mean)'. The white top-hat I - open(I) keeps bright details smaller than the structuring
    element se_px, the black top-hat close(I) - I keeps dark details."""
    g = cv2.GaussianBlur(gray(img), (3, 3), 0)
    parts = (["warm"] if polarity == "warm" else ["cold"] if polarity == "cold" else ["warm", "cold"])
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(se_px) | 1, int(se_px) | 1))
    out = np.zeros(g.shape, np.uint8)
    for pol in parts:
        src = g if pol == "warm" else 255 - g
        if method.startswith("Otsu"):
            _, m = cv2.threshold(src, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        elif method.startswith("adaptive"):
            m = cv2.adaptiveThreshold(src, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, int(block) | 1, offset)
        else:
            th = cv2.morphologyEx(src, cv2.MORPH_TOPHAT, se)
            if "Otsu" in method:
                _, m = cv2.threshold(th, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            else:
                t = th.astype(np.float32)
                m = ((t > t.mean() + k * t.std()) * 255).astype(np.uint8)
        out |= m
    return out


def contour_features(c, gsd_across=None, gsd_along=None):
    """Geometric characteristics of a contour (OpenCV point array): area and perimeter, px; box;
    length and width of the minimum-area rectangle; aspect ratio L/W; extent = area / box area;
    solidity = area / area of the convex hull; circularity 4 pi A / P^2 (1 for a circle).
    With the GSD of the row of the centroid (m/px across and along the line of sight) the contour is
    scaled to metres and its area, length and width in metres are added."""
    a = float(cv2.contourArea(c))
    p = float(cv2.arcLength(c, True))
    x, y, w, h = cv2.boundingRect(c)
    m = cv2.moments(c)
    cx, cy = (m["m10"] / m["m00"], m["m01"] / m["m00"]) if m["m00"] > 0 else (x + w / 2, y + h / 2)
    (_, _), (rw, rh), ang = cv2.minAreaRect(c)
    L, W = max(rw, rh), min(rw, rh)
    hull = float(cv2.contourArea(cv2.convexHull(c)))
    f = dict(cx=cx, cy=cy, box=[x, y, x + w, y + h], area_px=a, perimeter_px=p, length_px=L, width_px=W,
             aspect=L / max(W, 1e-6), extent=a / max(w * h, 1e-6), solidity=a / max(hull, 1e-6),
             circularity=4 * math.pi * a / max(p * p, 1e-6))
    if gsd_across is not None:
        pts = c.reshape(-1, 2).astype(np.float32) * np.float32([gsd_across, gsd_along])
        (_, _), (mw, mh), _ = cv2.minAreaRect(pts)
        f.update(area_m2=a * gsd_across * gsd_along, length_m=max(mw, mh), width_m=min(mw, mh))
    return f


class TIIRacing:
    """TII-RATM ("Race Against the Machine"): flights of an FPV racing quadrotor in an indoor arena with
    racing gates, piloted by human FPV pilots and autonomous, speeds above 21 m/s. Camera Arducam IMX219,
    640 x 480 px, 120 frames per second (125 measured from the time stamps), wide-angle lens with a field of
    view of 155 x 115 deg, tilted up by 30-50 deg; IMU 500 Hz (gyroscope, accelerometer), motion capture,
    labels of the gates. Bosello M. et al. IEEE Robotics and Automation Letters 9(4), 3799-3806 (2024).
    Licence CC BY 4.0.

    The frames of a flight are stored in random order in a zip archive inside the release archive. The first
    call of clip() downloads the camera archive of the flight once (about 200 MB, transferred in parallel
    parts) and keeps in the cache the frames of the requested interval and of the preset SEGMENTS."""

    RELEASE = "https://github.com/tii-racing/drone-racing-dataset/releases/download/v3.0.0/"
    RAW = "https://raw.githubusercontent.com/tii-racing/drone-racing-dataset/main/"
    # local header offset and compressed size of the members in the release archive piloted.zip
    FLIGHTS = {"flight-16p-trackRATM": dict(asset="piloted_zipchunk01", camera=(1574086012, 201903685),
                                            camera_csv=(1549423339, 33752), imu=(1546851674, 488328),
                                            meta=(1574085613, 258))}
    SEGMENTS = {"flight-16p-trackRATM": {"A: take-off and first gates (0.5-4.5 s)": (0.5, 4.5),
                                         "B: slalom between gates (15.0-19.0 s)": (15.0, 19.0)}}
    ROTATION_KEY = {50: "trackRATM", 40: "lemniscate", 60: "piloted"}

    def __init__(self, flight="flight-16p-trackRATM"):
        if flight not in self.FLIGHTS:
            raise ValueError(f"flight {flight} is not prepared; available: {list(self.FLIGHTS)}")
        self.flight = flight
        self.info = self.FLIGHTS[flight]
        self.url = self.RELEASE + self.info["asset"]

    def _member(self, key):
        off, csize = self.info[key]
        path = _cache("tii", self.flight, key + ".bin")
        if os.path.exists(path):
            return open(path, "rb").read()
        raw = _read_url(self.url, {"Range": f"bytes={off}-{off + 30 + 1024 + csize}"}, min_bytes=30 + csize)
        if raw[:4] != b"PK\x03\x04":
            raise IOError(f"bad local header of {key}")
        n, m = struct.unpack("<HH", raw[26:30])
        data = zlib.decompressobj(-15).decompress(raw[30 + n + m: 30 + n + m + csize])
        with open(path + ".part", "wb") as fh:
            fh.write(data)
        os.replace(path + ".part", path)
        return data

    def metadata(self):
        """Flight metadata (YAML of the data set) as a flat dict."""
        out, sect = {}, ""
        for line in self._member("meta").decode().splitlines():
            if not line.strip() or ":" not in line:
                continue
            key, val = line.split(":", 1)
            ind = len(key) - len(key.lstrip())
            key = key.strip()
            if ind == 0:
                sect = key
            full = key if ind == 0 else f"{sect}.{key}"
            val = val.strip()
            if val:
                try:
                    val = float(val) if "." in val else int(val)
                except ValueError:
                    pass
                out[full] = val
        return out

    def frame_table(self):
        """Table of the camera frames: index, file, timestamp_us, t (s from the first frame)."""
        import pandas as pd
        t = pd.read_csv(io.BytesIO(self._member("camera_csv")))
        t["index"] = np.arange(len(t))
        t["t"] = (t.timestamp - t.timestamp.iloc[0]) / 1e6
        return t.rename(columns={"filename": "file"})[["index", "file", "timestamp", "t"]]

    def imu(self):
        """IMU table: timestamp_us, t (s from the first camera frame), accel_x/y/z m/s^2, gyro_x/y/z rad/s
        (axes of the body of the drone)."""
        import pandas as pd
        d = pd.read_csv(io.BytesIO(self._member("imu")))
        d["t"] = (d.timestamp - self.frame_table().timestamp.iloc[0]) / 1e6
        return d

    def calibration(self):
        """Camera matrix K and distortion coefficients D (pinhole model with radial and tangential terms)
        of the piloted trackRATM flights."""
        path = fetch(self.RAW + "camera_calibration/calib_p-trackRATM.json", _cache("tii", "calib_p-trackRATM.json"))
        d = json.load(open(path))
        return np.array(d["mtx"], float), np.array(d["dist"], float).ravel()

    def camera_to_body(self):
        """Rotation matrix R (3 x 3) from the camera axes (x right, y down, z forward) to the body axes of
        the drone, for the camera angle of the flight (file drone_to_camera.json of the data set)."""
        path = fetch(self.RAW + "camera_calibration/drone_to_camera.json", _cache("tii", "drone_to_camera.json"))
        q = json.load(open(path))["rotation"][self.ROTATION_KEY.get(int(self.metadata().get("camera.angle", 50)), "trackRATM")]
        x, y, z, w = q["x"], q["y"], q["z"], q["w"]
        return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
                         [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
                         [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])

    def _download_camera_zip(self, path, part=16 << 20, threads=6):
        off, csize = self.info["camera"]
        hdr = _read_url(self.url, {"Range": f"bytes={off}-{off + 29}"}, min_bytes=30)
        n, m = struct.unpack("<HH", hdr[26:30])
        start = off + 30 + n + m
        pieces = [(start + p, min(start + csize, start + p + part) - 1) for p in range(0, csize, part)]
        with open(path, "wb") as fh:
            fh.truncate(csize)

        def get(rg):
            a, b = rg
            data = _read_url(self.url, {"Range": f"bytes={a}-{b}"}, timeout=120, min_bytes=b - a + 1)
            with open(path, "r+b") as fh:
                fh.seek(a - start)
                fh.write(data)
            return len(data)

        with ThreadPoolExecutor(threads) as ex:
            return sum(ex.map(get, pieces))

    def _extract(self, wanted):
        """Stream the camera archive of the flight and keep the frames with the indices in `wanted`."""
        tmp = _cache("tii", self.flight, "_camera_deflate.part")
        self._download_camera_zip(tmp)
        d = zlib.decompressobj(-15)
        buf, kept = b"", 0
        with open(tmp, "rb") as fh:
            while True:
                chunk = fh.read(4 << 20)
                buf += d.decompress(chunk) if chunk else d.flush()
                while len(buf) >= 30 and buf[:4] == b"PK\x03\x04":
                    flag, comp = struct.unpack("<HH", buf[6:10])
                    csz, usz, nl, el = struct.unpack("<IIHH", buf[18:30])
                    need = 30 + nl + el + csz
                    if len(buf) < need:
                        break
                    name = buf[30:30 + nl].decode("utf-8", "replace")
                    data = buf[30 + nl + el:need]
                    buf = buf[need:]
                    base = name.rsplit("/", 1)[-1]
                    if base.endswith(".jpg") and int(base.split("_")[0]) in wanted:
                        if comp == 8:
                            data = zlib.decompressobj(-15).decompress(data)
                        with open(_cache("tii", self.flight, "frames", base), "wb") as out:
                            out.write(data)
                        kept += 1
                if not chunk or (len(buf) >= 4 and buf[:4] == b"PK\x01\x02"):
                    break
        os.remove(tmp)
        return kept

    def clip(self, t0, t1, step=1):
        """Frames (RGB, 480 x 640 x 3) of the interval t0...t1 s with every `step`-th frame, and their table."""
        tab = self.frame_table()
        sel = tab[(tab.t >= t0) & (tab.t < t1)].iloc[::max(1, int(step))]
        miss = [i for i, f in zip(sel["index"], sel.file) if not os.path.exists(_cache("tii", self.flight, "frames", f))]
        if miss:
            wanted = set(miss)
            for a, b in self.SEGMENTS.get(self.flight, {}).values():
                wanted |= set(tab[(tab.t >= a) & (tab.t < b)]["index"])
            self._extract(wanted)
        frames = [_rgb(_cache("tii", self.flight, "frames", f)) for f in sel.file]
        return frames, sel.reset_index(drop=True)


def undistort_maps(K, D, size, alpha=0.0):
    """Maps of cv2.remap that remove the lens distortion, and the new camera matrix.
    alpha = 0 keeps only valid pixels (the image is cropped), alpha = 1 keeps the whole field of view."""
    newK, _ = cv2.getOptimalNewCameraMatrix(K, D, size, alpha, size)
    m1, m2 = cv2.initUndistortRectifyMap(K, D, None, newK, size, cv2.CV_16SC2)
    return m1, m2, newK


def gyro_rotation(t, w, t1, t2):
    """Rotation vector (rad) of the body between the times t1 and t2 obtained by integration of the angular
    rate w (N x 3, rad/s) measured at the times t (s): R = prod exp([w_k dt_k])."""
    m = (t >= t1) & (t <= t2)
    idx = np.where(m)[0]
    if idx.size == 0:
        return np.zeros(3)
    tt = np.concatenate([[t1], t[idx], [t2]])
    ww = np.vstack([w[max(idx[0] - 1, 0)], w[idx], w[idx[-1]]])
    R = np.eye(3)
    for k in range(len(tt) - 1):
        dt = tt[k + 1] - tt[k]
        if dt <= 0:
            continue
        Rk, _ = cv2.Rodrigues((ww[k] * dt).reshape(3, 1))
        R = R @ Rk
    rv, _ = cv2.Rodrigues(R)
    return rv.ravel()


def rotation_from_homography(H, K):
    """Rotation of the camera between two frames from their homography, assuming a pure rotation:
    H = K R K^-1, so R = K^-1 H K scaled to det = 1 and projected on the nearest rotation matrix.
    Returns the rotation vector (rad) of R in the axes of the camera."""
    if H is None or not np.all(np.isfinite(H)):
        return np.full(3, np.nan)
    R = np.linalg.inv(K) @ H @ K
    d = np.linalg.det(R)
    if not np.isfinite(d) or abs(d) < 1e-12:
        return np.full(3, np.nan)                          # degenerate homography: no rotation can be computed
    R = R / np.cbrt(d)
    U, _, Vt = np.linalg.svd(R)
    R = U @ Vt
    if np.linalg.det(R) < 0:
        R = -R
    rv, _ = cv2.Rodrigues(R)
    return rv.ravel()


def smooth_rotations(rotvecs, sigma):
    """Smoothed orientations of the camera: rotvecs (N x 3) are absolute orientations; every orientation is
    replaced by the Gaussian-weighted mean of its neighbours (sigma in frames) computed in the tangent space
    of the orientation itself. Returns N x 3 rotation vectors."""
    Rs = [cv2.Rodrigues(np.asarray(r, float).reshape(3, 1))[0] for r in rotvecs]
    n = len(Rs)
    half = int(max(1, round(3 * sigma)))
    out = []
    for i in range(n):
        a, b = max(0, i - half), min(n, i + half + 1)
        wts = np.exp(-0.5 * ((np.arange(a, b) - i) / max(sigma, 1e-6)) ** 2)
        wts /= wts.sum()
        acc = np.zeros(3)
        for j, wj in zip(range(a, b), wts):
            d, _ = cv2.Rodrigues(Rs[i].T @ Rs[j])
            acc += wj * d.ravel()
        Rm = Rs[i] @ cv2.Rodrigues(acc.reshape(3, 1))[0]
        out.append(cv2.Rodrigues(Rm)[0].ravel())
    return np.array(out)


def interframe_psnr(frames, valid=None):
    """Inter-frame transformation fidelity (ITF): mean PSNR between consecutive frames, dB; a stabilised
    video has a higher ITF. `valid` (optional list of masks) limits the comparison to valid pixels."""
    vals = []
    for k in range(1, len(frames)):
        a, b = _f(gray(frames[k - 1])), _f(gray(frames[k]))
        m = np.ones(a.shape, bool) if valid is None else (valid[k - 1] & valid[k])
        if m.sum() < 100:
            continue
        mse = float(((a - b)[m] ** 2).mean())
        vals.append(10 * math.log10(255.0 ** 2 / max(mse, 1e-6)))
    return float(np.mean(vals)) if vals else float("nan")



# =====================================================================================
# Topic 4. Machine learning and neural networks in the recognition of military objects
# =====================================================================================
class MilitaryVehicles:
    """Photographs of 24 types of military vehicles: 9444 images (7823 for training, 1621 for
    testing) collected from web sites; Kricheli J. S. et al., arXiv:2407.15192 (2024);
    Hugging Face data set leibnitz-lab/military_vehicles, licence declared by the authors: MIT
    (the rights to the individual photographs are not stated by the authors).

    The parquet files hold every image twice: in the tree of 24 fine types (file names of
    1-3 digits, "0.jpg") and in the tree of the 7 coarse classes of the authors (4 digits,
    "0000.jpg"). The fine tree is used; the coarse class of the authors is attached to every
    image by its MD5 checksum (column `source_group`), the grouping of the course is in `group`."""

    URL = "https://huggingface.co/api/datasets/leibnitz-lab/military_vehicles/parquet/default/{}/0.parquet"
    GROUPS = {"T-14": "Tank", "T-62": "Tank", "T-64": "Tank", "T-72": "Tank", "T-80": "Tank", "T-90": "Tank",
              "BMP-1": "IFV", "BMP-2": "IFV", "BMP-T15": "IFV", "BMD": "IFV",
              "BTR-60": "APC/recon", "BTR-70": "APC/recon", "BTR-80": "APC/recon", "BRDM": "APC/recon",
              "MT_LB": "APC/recon", "2S19_MSTA": "Artillery", "D-30": "Artillery",
              "BM-30": "MLRS", "Tornado": "MLRS", "TOS-1": "MLRS",
              "Pantsir-S1": "Air defence/radar", "30N6E": "Air defence/radar",
              "Iskander": "Missile system", "Rs-24": "Missile system"}
    GROUP_ORDER = ("Tank", "IFV", "APC/recon", "Artillery", "MLRS", "Air defence/radar", "Missile system")
    TYPES = {"T-14": "main battle tank (Armata)", "T-62": "main battle tank", "T-64": "main battle tank",
             "T-72": "main battle tank", "T-80": "main battle tank", "T-90": "main battle tank",
             "BMP-1": "tracked infantry fighting vehicle", "BMP-2": "tracked infantry fighting vehicle",
             "BMP-T15": "heavy infantry fighting vehicle T-15 (Armata)", "BMD": "airborne infantry fighting vehicle",
             "BTR-60": "wheeled armoured personnel carrier", "BTR-70": "wheeled armoured personnel carrier",
             "BTR-80": "wheeled armoured personnel carrier", "BRDM": "wheeled armoured scout car",
             "MT_LB": "tracked multi-purpose armoured tractor", "2S19_MSTA": "152 mm self-propelled howitzer",
             "D-30": "122 mm towed howitzer", "BM-30": "300 mm multiple rocket launcher (Smerch)",
             "Tornado": "multiple rocket launcher (Tornado)", "TOS-1": "heavy thermobaric rocket launcher",
             "Pantsir-S1": "self-propelled anti-aircraft gun and missile system",
             "30N6E": "fire-control radar of the S-300 system", "Iskander": "tactical ballistic missile system",
             "Rs-24": "intercontinental ballistic missile system (Yars) on a wheeled launcher"}

    def __init__(self):
        import hashlib
        import pandas as pd
        import pyarrow.parquet as pq
        parts = []
        for sp in ("train", "test"):
            path = fetch(self.URL.format(sp), _cache("military_vehicles", sp + ".parquet"))
            t = pq.read_table(path)
            names = json.loads(t.schema.metadata[b"huggingface"])["info"]["features"]["label"]["names"]
            d = t.to_pandas()
            d["file"] = [x["path"] for x in d.image]
            d["bytes"] = [x["bytes"] for x in d.image]
            d["label_name"] = [names[i] for i in d.label]
            d["md5"] = [hashlib.md5(b).hexdigest() for b in d.bytes]
            d["split"] = sp
            parts.append(d.drop(columns=["image", "label"]))
        a = pd.concat(parts, ignore_index=True)
        fine = a.file.str.len() < 8
        coarse = a[~fine].groupby("md5").label_name.agg(lambda s: s.value_counts().index[0])
        t = a[fine].rename(columns={"label_name": "cls"}).reset_index(drop=True)
        t["source_group"] = t.md5.map(coarse)
        t["group"] = t.cls.map(self.GROUPS)
        hw = [cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_GRAYSCALE).shape for b in t.bytes]
        t["height"], t["width"] = [h for h, w in hw], [w for h, w in hw]
        tr = t[t.split == "train"]
        t["dup_in_split"] = t.duplicated(["split", "md5"])
        cls_train = tr.groupby("md5").cls.agg(lambda s: ", ".join(sorted(set(s))))
        t["in_train"] = (t.split == "test") & t.md5.isin(tr.md5)
        t["train_label"] = np.where(t.in_train, t.md5.map(cls_train), "")
        self.table = t[["split", "file", "cls", "group", "source_group", "width", "height", "md5",
                        "dup_in_split", "in_train", "train_label", "bytes"]]
        self.CLASSES = tuple(sorted(t.cls.unique()))

    def image(self, i, size=None, gray=False):
        """Image i of the table: RGB (or grey) array, optionally resized to size x size."""
        b = self.table.bytes.iat[i] if not isinstance(i, (bytes, bytearray)) else i
        img = cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_GRAYSCALE if gray else cv2.IMREAD_COLOR)
        if not gray:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if size:
            img = cv2.resize(img, (int(size), int(size)), interpolation=cv2.INTER_AREA)
        return img

    def hashes(self):
        """dHash (8 bytes) of every image of the table (computed once)."""
        if getattr(self, "_hashes", None) is None:
            self._hashes = np.stack([dhash(self.image(i, gray=True)) for i in range(len(self.table))])
        return self._hashes

    def clean(self, near=None):
        """Indices of the cleaned training and test parts: a copy of an image repeated inside a part is
        removed, and every test image that also occurs in the training part is removed. With near = d
        also the test images whose dHash differs from the hash of a training image by at most d bits
        (near duplicates: the same photograph re-compressed or resized) are removed."""
        t = self.table
        tr = t.index[(t.split == "train") & ~t.dup_in_split].values
        te = t.index[(t.split == "test") & ~t.dup_in_split & ~t.in_train].values
        if near is not None:
            h = self.hashes()
            d, _ = hamming_nearest(h[te], h[tr])
            te = te[d > int(near)]
        return tr, te

    def subset(self, idx, n_per_class, seed=0):
        """At most n_per_class indices of every fine type from idx (random, reproducible)."""
        rng = np.random.default_rng(seed)
        t = self.table.loc[idx]
        out = [rng.permutation(g.index.values)[:int(n_per_class)] for _, g in t.groupby("cls")]
        return np.sort(np.concatenate(out))


class SULAND:
    """SULAND: frames of video sequences of a camera moved over grass, fallen leaves and gravel
    with surrogates of two anti-personnel surface landmines, PFM-1 ("butterfly") and PMA-2
    ("starfish"); iPhone 13, 1440 x 1920 pixels (portrait), boxes in the YOLO format.
    In-distribution part recorded in Italy (data-iid: train 34, val 5, test 6 sequences) and
    out-of-distribution part recorded in the USA (data-ood: val 10 sequences).
    Vivoli E., Bertini M., Capineri L. Remote Sensing 16(4), 677 (2024); refined annotation
    SULAND_v2: Lekhak S. et al., arXiv:2607.28996 (2026). Licence CC BY 4.0.

    Only the annotation archive (7-10 MB) is downloaded in full; the frames are read one by one
    from the image archive of 16.7 GB with HTTP range requests. version="v1" gives the original
    annotation of the authors, including its errors (in data-ood the class numbers are inverted)."""

    BASE = "https://huggingface.co/datasets/SagarLekhak/SULAND_v2_RGB_Surface_Landmine_Dataset/resolve/main/"
    ANNOTATIONS = {"v2": "Annotation_files_for_SULAND_v2.zip", "v1": "Annotations_files_for_SULAND_v1.zip"}
    CLASSES = ("PFM-1", "PMA-2")
    NICKNAMES = {"PFM-1": "butterfly", "PMA-2": "starfish"}
    COLORS = {"PFM-1": (255, 64, 64), "PMA-2": (0, 229, 255)}
    _images = None

    def __init__(self, version="v2"):
        import re as _re
        import zipfile
        import pandas as pd
        self.version = version
        path = fetch(self.BASE + self.ANNOTATIONS[version], _cache("suland", self.ANNOTATIONS[version]))
        z = zipfile.ZipFile(path)
        frames, boxes = [], []
        for n in z.namelist():
            if not (n.endswith(".txt") and "/labels/" in n):
                continue
            p = n.split("/")
            i = p.index("labels")
            dom, split, seq, stem = p[i - 2], p[i - 1], p[i + 1], p[-1][:-4]
            fr = int(_re.search(r"(\d+)$", stem).group(1))
            lines = [l.split() for l in z.read(n).decode().strip().splitlines() if l.strip()]
            for l in lines:
                c = int(l[0])
                boxes.append((dom, split, seq, fr, c, self.CLASSES[c], *map(float, l[1:5])))
            frames.append((dom, split, seq, fr, stem, f"SULAND_images/{dom}/{split}/images/{seq}/{stem}.jpg",
                           len(lines), sum(int(l[0]) == 0 for l in lines), sum(int(l[0]) == 1 for l in lines)))
        self.frames = pd.DataFrame(frames, columns=["domain", "split", "seq", "frame", "stem", "member", "n_boxes",
                                                    "n_pfm1", "n_pma2"]).sort_values(["domain", "split", "seq", "frame"]).reset_index(drop=True)
        self.boxes = pd.DataFrame(boxes, columns=["domain", "split", "seq", "frame", "cls_id", "cls", "cx", "cy", "w", "h"])
        self._by_frame = {k: g for k, g in self.boxes.groupby(["seq", "frame"])}
        self._member = {(r.seq, r.frame): r.member for r in self.frames.itertuples()}

    def sequences(self):
        """Table of the sequences: part, number of frames, frames with mines, boxes of both classes."""
        f = self.frames
        g = f.groupby(["domain", "split", "seq"]).agg(frames=("frame", "size"), first=("frame", "min"), last=("frame", "max"),
                                                      with_mines=("n_boxes", lambda s: int((s > 0).sum())),
                                                      boxes=("n_boxes", "sum"), pfm1=("n_pfm1", "sum"), pma2=("n_pma2", "sum"))
        return g.reset_index()

    @classmethod
    def _zip(cls):
        if SULAND._images is None:
            SULAND._images = RemoteZip(cls.BASE + "SULAND_images.zip", "suland_images")
        return SULAND._images

    def path(self, seq, frame):
        return self._zip()._fetch_member(self._member[(seq, int(frame))])

    def prefetch(self, items, threads=8):
        """items: list of (sequence, frame); downloads the frames (about 0.5 MB each)."""
        return self._zip().get([self._member[(s, int(f))] for s, f in items], threads)

    def objects(self, seq, frame, width=1440, height=1920):
        """Boxes of a frame in pixels: list of dicts with 'cls' and 'box' [x1, y1, x2, y2]."""
        g = self._by_frame.get((seq, int(frame)))
        if g is None:
            return []
        return [dict(cls=r.cls, box=yolo_to_xyxy(r.cx, r.cy, r.w, r.h, width, height)) for r in g.itertuples()]

    def load(self, seq, frame, gray=False):
        """Frame (RGB or grey array) and its boxes in pixels."""
        img = cv2.imread(self.path(seq, frame), cv2.IMREAD_GRAYSCALE if gray else cv2.IMREAD_COLOR)
        if not gray:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img, self.objects(seq, frame, img.shape[1], img.shape[0])

    def window(self, seq, length):
        """First frame of the run of `length` consecutive frames of a sequence with most frames with mines."""
        f = self.frames[self.frames.seq == seq].set_index("frame").n_boxes.gt(0).astype(int)
        s = f.rolling(int(length)).sum().shift(-(int(length) - 1))
        return int(s.idxmax())

    def crop_set(self, seqs, length=30, size=64, scale=1.5, n_background=2, seed=0, threads=8):
        """Crops of a run of `length` consecutive frames of every sequence: every box (square of
        scale x the longer side of the box around its centre) and n_background squares of the same
        size without mines from the same frame; grey crops size x size. Returns a DataFrame with
        seq, frame, label ('PFM-1', 'PMA-2', 'background') and img."""
        import pandas as pd
        rng = np.random.default_rng(seed)
        items = []
        for s in seqs:
            a = self.window(s, length)
            items += [(s, a + k) for k in range(int(length)) if (s, a + k) in self._member]
        self.prefetch(items, threads)
        rows = []
        for s, fr in items:
            img, objs = self.load(s, fr, gray=True)
            H, W = img.shape
            sq = []
            for o in objs:
                x1, y1, x2, y2 = o["box"]
                side = int(max(24, scale * max(x2 - x1, y2 - y1)))
                x0 = int(min(max((x1 + x2) / 2 - side / 2, 0), W - side))
                y0 = int(min(max((y1 + y2) / 2 - side / 2, 0), H - side))
                sq.append((x0, y0, side))
                rows.append(dict(seq=s, frame=fr, label=o["cls"],
                                 img=cv2.resize(img[y0:y0 + side, x0:x0 + side], (size, size), interpolation=cv2.INTER_AREA)))
            k = tries = 0
            while k < n_background and tries < 300:
                tries += 1
                side = int(rng.choice([q[2] for q in sq])) if sq else 120
                x0, y0 = int(rng.integers(0, W - side)), int(rng.integers(0, H - side))
                if all(x0 + side <= q[0] or q[0] + q[2] <= x0 or y0 + side <= q[1] or q[1] + q[2] <= y0 for q in sq):
                    rows.append(dict(seq=s, frame=fr, label="background",
                                     img=cv2.resize(img[y0:y0 + side, x0:x0 + side], (size, size), interpolation=cv2.INTER_AREA)))
                    k += 1
        return pd.DataFrame(rows)


def yolo_to_xyxy(cx, cy, w, h, width, height):
    """YOLO box (centre and size as fractions of the image) -> [x1, y1, x2, y2] in pixels."""
    return [(cx - w / 2) * width, (cy - h / 2) * height, (cx + w / 2) * width, (cy + h / 2) * height]


def xyxy_to_yolo(box, width, height):
    """[x1, y1, x2, y2] in pixels -> (cx, cy, w, h) as fractions of the image (YOLO format)."""
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2 / width, (y1 + y2) / 2 / height, (x2 - x1) / width, (y2 - y1) / height)


def xyxy_to_coco(box):
    """[x1, y1, x2, y2] -> [x, y, width, height] (COCO format, pixels)."""
    x1, y1, x2, y2 = box
    return [x1, y1, x2 - x1, y2 - y1]


def dhash(img, n=8):
    """Difference hash: n x n bits (sign of the difference of neighbouring pixels of the image reduced
    to (n + 1) x n); returned as n * n / 8 bytes (uint8). Close images have close hashes."""
    g = gray(img) if img.ndim == 3 else img
    r = cv2.resize(g, (n + 1, n), interpolation=cv2.INTER_AREA).astype(np.int16)
    return np.packbits((r[:, 1:] > r[:, :-1]).ravel())


def hamming_nearest(A, B, chunk=256):
    """For every hash of A (Na x k uint8) the smallest Hamming distance to the hashes of B and its index."""
    A, B = np.asarray(A, np.uint8), np.asarray(B, np.uint8)
    dist, idx = np.empty(len(A), int), np.empty(len(A), int)
    for s in range(0, len(A), chunk):
        x = np.bitwise_xor(A[s:s + chunk, None, :], B[None, :, :])
        d = np.unpackbits(x, axis=2).sum(2)
        dist[s:s + chunk], idx[s:s + chunk] = d.min(1), d.argmin(1)
    return dist, idx


def hog_length(size, cell=8, block=2, orientations=9):
    """Length of the HOG vector of a size x size image: blocks x block^2 x orientations."""
    nb = size // cell - block + 1
    return nb * nb * block * block * orientations


def hog_features(images, size=64, orientations=9, cell=8, block=2):
    """HOG vectors (N x D float32) of grey images resized to size x size (scikit-image)."""
    from skimage.feature import hog
    out = []
    for im in images:
        g = gray(im) if im.ndim == 3 else im
        if g.shape != (size, size):
            g = cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA)
        out.append(hog(g, orientations=orientations, pixels_per_cell=(cell, cell), cells_per_block=(block, block)))
    return np.asarray(out, np.float32)


IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], np.float32)


def imagenet_batch(images, size=224):
    """RGB uint8 images -> float32 array N x 3 x size x size normalised as for ImageNet models."""
    x = np.stack([cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA) for im in images]).astype(np.float32) / 255.0
    return ((x - IMAGENET_MEAN) / IMAGENET_STD).transpose(0, 3, 1, 2).copy()


def cnn_features(model, images, size=224, batch=64, device="cpu"):
    """Output of the convolutional part of a torchvision MobileNet (features + global average pooling):
    N x C float32. `images` is a list of RGB arrays or a function i -> image with len given by a range."""
    import torch
    model = model.eval().to(device)
    out = []
    with torch.no_grad():
        for s in range(0, len(images), batch):
            x = torch.from_numpy(imagenet_batch([images[i] for i in range(s, min(s + batch, len(images)))], size)).to(device)
            f = model.avgpool(model.features(x)).flatten(1)
            out.append(f.cpu().numpy())
    return np.concatenate(out)


def count_parameters(model):
    """Number of parameters of a torch model (all, trainable)."""
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)


def conv_params(k, c_in, c_out, separable=False):
    """Number of weights of a k x k convolution layer (without bias): standard k^2*C_in*C_out or
    depthwise separable k^2*C_in + C_in*C_out."""
    return k * k * c_in + c_in * c_out if separable else k * k * c_in * c_out


# ---- diagrams of networks -------------------------------------------------------------
_EDGE, _IN, _HID, _OUT, _OP = "#455a64", "#e3f2fd", "#fff3e0", "#e8f5e9", "#f3e5f5"


def _node(ax, x, y, r, fc, text=None, fs=8):
    import matplotlib.patches as mp
    ax.add_patch(mp.Circle((x, y), r, fc=fc, ec=_EDGE, lw=1.0, zorder=3))
    if text:
        ax.text(x, y, text, ha="center", va="center", fontsize=fs, zorder=4)


def _box(ax, x, y, w, h, fc, text, fs=8.5):
    import matplotlib.patches as mp
    ax.add_patch(mp.FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.02,rounding_size=0.05",
                                   fc=fc, ec=_EDGE, lw=1.0, zorder=3))
    ax.text(x, y, text, ha="center", va="center", fontsize=fs, zorder=4)


def _arrow(ax, a, b, lw=1.0, color=_EDGE):
    ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="-|>", lw=lw, color=color), zorder=2)


def draw_neuron(ax, inputs=("x1", "x2"), weights=("w1", "w2"), bias="b", activation="sigmoid", output="y"):
    """Diagram of an artificial neuron: inputs, weights, weighted sum with bias, activation, output."""
    n = len(inputs)
    ys = np.linspace(0.85, 0.15, n) if n > 1 else np.array([0.5])
    for y, xi, wi in zip(ys, inputs, weights):
        _node(ax, 0.08, y, 0.055, _IN, xi)
        _arrow(ax, (0.14, y), (0.43, 0.5 + (y - 0.5) * 0.15))
        ax.text(0.27, 0.5 + (y - 0.5) * 0.6 + 0.03, wi, fontsize=9, color="#1565c0", ha="center")
    _node(ax, 0.08, -0.05, 0.045, "#eeeeee", "1", 8)
    _arrow(ax, (0.13, -0.03), (0.43, 0.44))
    ax.text(0.26, 0.12, bias, fontsize=9, color="#1565c0")
    _node(ax, 0.5, 0.5, 0.08, _HID, "Σ", 13)
    _arrow(ax, (0.58, 0.5), (0.66, 0.5))
    _box(ax, 0.75, 0.5, 0.17, 0.2, _OP, "")
    z = np.linspace(-1, 1, 40)
    act = {"sigmoid": 1 / (1 + np.exp(-5 * z)), "tanh": (np.tanh(3 * z) + 1) / 2, "ReLU": np.maximum(z, 0),
           "step": (z > 0).astype(float)}.get(activation, z)
    ax.plot(0.75 + z * 0.065, 0.43 + act * 0.12, color="#6a1b9a", lw=1.5, zorder=5)
    ax.text(0.75, 0.64, f"φ: {activation}", ha="center", fontsize=8.5)
    _arrow(ax, (0.84, 0.5), (0.95, 0.5))
    ax.text(0.97, 0.5, output, va="center", fontsize=11)
    ax.text(0.5, 0.24, "z = Σ wᵢxᵢ + b", ha="center", fontsize=9)
    ax.set_xlim(0, 1.03)
    ax.set_ylim(-0.12, 1.0)
    ax.set_aspect("equal")
    ax.axis("off")


def draw_mlp(ax, sizes, names=None, max_neurons=7, input_names=None, output_names=None):
    """Graph of a fully connected network: layers of neurons (circles) with all connections; a layer
    larger than max_neurons is drawn with max_neurons circles and a gap marked by dots."""
    L = len(sizes)
    xs = np.linspace(0.08, 0.92, L)
    pos = []
    for x, n in zip(xs, sizes):
        m = min(n, max_neurons)
        ys = np.linspace(0.9, 0.1, m) if m > 1 else np.array([0.5])
        pos.append((x, ys, n > max_neurons))
    for (x0, y0, _), (x1, y1, _) in zip(pos[:-1], pos[1:]):
        for a in y0:
            for b in y1:
                ax.plot([x0, x1], [a, b], color="#b0bec5", lw=0.4, zorder=1)
    for k, ((x, ys, cut), n) in enumerate(zip(pos, sizes)):
        fc = _IN if k == 0 else (_OUT if k == L - 1 else _HID)
        for j, y in enumerate(ys):
            if cut and j == len(ys) // 2:
                ax.text(x, y, "⋮", ha="center", va="center", fontsize=14, zorder=4)
                continue
            ax.scatter([x], [y], s=150, c=fc, edgecolors=_EDGE, linewidths=1.0, zorder=3)
        label = (names[k] if names else "") + f"\n{n} neurons" if k else (names[k] if names else "input") + f"\n{n} values"
        ax.text(x, -0.02, label, ha="center", va="top", fontsize=8.5)
    if input_names:
        for y, t in zip(pos[0][1], input_names):
            ax.text(pos[0][0] - 0.04, y, t, ha="right", va="center", fontsize=7.5)
    if output_names:
        for y, t in zip(pos[-1][1], output_names):
            ax.text(pos[-1][0] + 0.04, y, t, ha="left", va="center", fontsize=7.5)
    ax.set_xlim(-0.12 if input_names else -0.02, 1.2 if output_names else 1.02)
    ax.set_ylim(-0.2, 1.0)
    ax.axis("off")


def draw_cnn(ax, stages, title=None):
    """Feature maps of a convolutional network: stages = list of (label, height, width, channels);
    every map is a box with depth proportional to log(channels), height to log(size)."""
    import matplotlib.patches as mp
    x = 0.0
    for k, (lab, h, w, c) in enumerate(stages):
        hh = 0.25 + 0.12 * math.log2(max(h, 1))
        dd = 0.08 + 0.05 * math.log2(max(c, 1))
        fc = _IN if k == 0 else (_OUT if k == len(stages) - 1 else _HID)
        if h > 1:
            for j in range(3):
                ax.add_patch(mp.Rectangle((x + j * dd / 3, -hh / 2 + j * 0.05), 0.12, hh, fc=fc, ec=_EDGE, lw=0.8, zorder=3 + j))
            ax.text(x + 0.1, -hh / 2 - 0.12, f"{h}×{w}×{c}", ha="center", va="top", fontsize=7.5)
            ax.text(x + 0.1, hh / 2 + 0.25 + 0.35 * (k % 2), lab, ha="center", va="bottom", fontsize=8)
            x_next = x + 0.12 + dd + 0.35
        else:
            ax.add_patch(mp.Rectangle((x, -0.6), 0.08, 1.2, fc=fc, ec=_EDGE, lw=0.8, zorder=3))
            ax.text(x + 0.04, -0.72, f"{c}", ha="center", va="top", fontsize=7.5)
            ax.text(x + 0.04, 0.85 + 0.35 * (k % 2), lab, ha="center", va="bottom", fontsize=8)
            x_next = x + 0.08 + 0.35
        if k < len(stages) - 1:
            _arrow(ax, (x_next - 0.3, 0), (x_next - 0.05, 0), lw=0.9)
        x = x_next
    ax.set_xlim(-0.1, x)
    ax.set_ylim(-1.6, 2.2)
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=9.5)


def draw_mobilenet_block(ax, c_in=24, expansion=88, c_out=24, k=3, se=False, stride=1):
    """Inverted residual block of MobileNet V2/V3: 1x1 expansion, k x k depthwise convolution,
    optional squeeze-and-excitation, 1x1 linear projection, residual connection when
    stride = 1 and C_in = C_out; the number of weights of every layer is written under it."""
    steps = [("input", f"{c_in} ch.", _IN, None),
             ("1×1 conv\nexpansion", f"{c_in}→{expansion}", _OP, c_in * expansion),
             (f"{k}×{k} depthwise\nconv, stride {stride}", f"{expansion} ch.", _OP, k * k * expansion)]
    if se:
        steps.append(("squeeze-\nexcitation", "channel weights", _OP, 2 * expansion * max(8, expansion // 4)))
    steps += [("1×1 conv\nprojection", f"{expansion}→{c_out}", _OP, expansion * c_out), ("output", f"{c_out} ch.", _OUT, None)]
    xs = np.linspace(0.07, 0.93, len(steps))
    for x, (t, sub, fc, n) in zip(xs, steps):
        _box(ax, x, 0.5, 0.13, 0.3, fc, t, 8)
        ax.text(x, 0.3, sub, ha="center", va="top", fontsize=7.5)
        if n is not None:
            ax.text(x, 0.2, f"{n:,} weights".replace(",", " "), ha="center", va="top", fontsize=7.5, color="#1565c0")
    for a, b in zip(xs[:-1], xs[1:]):
        _arrow(ax, (a + 0.065, 0.5), (b - 0.065, 0.5))
    if stride == 1 and c_in == c_out:
        ax.annotate("", xy=(xs[-1], 0.66), xytext=(xs[0], 0.66),
                    arrowprops=dict(arrowstyle="-|>", lw=1.0, color="#2e7d32", connectionstyle="arc3,rad=-0.12"))
        ax.text(0.5, 0.86, "residual connection: output = input + F(input)", ha="center", fontsize=8, color="#2e7d32")
    std = k * k * expansion * expansion
    ax.text(0.5, 0.02, (f"a standard {k}×{k} convolution {expansion}→{expansion} channels would need {std:,} weights; "
                        f"the depthwise one needs {k * k * expansion:,}").replace(",", " "), ha="center", fontsize=8)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.05, 1.0)
    ax.axis("off")



# ---- Topic 4, Lessons 4.3-4.4: verification of thermal candidates, Grad-CAM -------------------
def crop_square(img, cx, cy, side):
    """Square fragment side x side centred at (cx, cy); parts outside the frame are filled by
    reflection. The result is a new array (not a view of the frame)."""
    side = int(max(8, round(side)))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    H, W = img.shape[:2]
    xa, ya, xb, yb = max(0, x0), max(0, y0), min(W, x0 + side), min(H, y0 + side)
    return cv2.copyMakeBorder(img[ya:yb, xa:xb], ya - y0, y0 + side - yb, xa - x0, x0 + side - xb, cv2.BORDER_REFLECT)


def thermal_candidates(img, objects, gsd, size=48, scale=1.5, targets=("Person",), k=6.0):
    """Candidates of the classical warm-object detector of Topic 3 in a white-hot thermal frame,
    labelled by the annotation: 'target' (the centre lies in a box of a target class), 'background'
    (false alarm: outside all annotated boxes) or 'other object' (inside a box of another class).
    Every candidate gets a grey fragment of scale x the longer side of its box, size x size px.
    Returns a list of dicts with box, label, side (px) and chip."""
    boxes, _ = detect_warm_objects(img, gsd, k=k)
    if len(boxes) == 0:
        return []
    m = match_detections(boxes, objects, set(targets))
    lab = {tuple(b): "target" for b in m["tp"]}
    lab.update({tuple(b): "background" for b in m["fp"]})
    lab.update({tuple(b): "other object" for b in m["other"]})
    g = gray(img)
    out = []
    for b in boxes:
        side = max(b[2] - b[0], b[3] - b[1], 8)
        chip = crop_square(g, (b[0] + b[2]) / 2, (b[1] + b[3]) / 2, scale * side)
        out.append(dict(box=[float(v) for v in b], label=lab[tuple(b)], side=float(side),
                        chip=cv2.resize(chip, (size, size), interpolation=cv2.INTER_AREA)))
    return out


def grad_cam(model, x, layer, class_index=None):
    """Grad-CAM (Selvaraju et al., 2017): map of the regions of the input that raise the score of a
    class. The feature maps A_k of `layer` are weighted by the mean gradient of the class score over
    each map, summed and passed through ReLU. x: tensor 1 x 3 x H x W. Returns (map H x W in 0...1,
    class index, probabilities)."""
    import torch
    acts, grads = [], []
    h1 = layer.register_forward_hook(lambda m, i, o: acts.append(o))
    h2 = layer.register_full_backward_hook(lambda m, gi, go: grads.append(go[0]))
    model.eval()
    try:
        x = x.clone().requires_grad_(True)
        out = model(x)
        c = int(out.argmax(1)) if class_index is None else int(class_index)
        model.zero_grad()
        out[0, c].backward()
        a, g = acts[0][0].detach(), grads[0][0].detach()
        cam = torch.relu((g.mean((1, 2))[:, None, None] * a).sum(0)).cpu().numpy()
    finally:
        h1.remove()
        h2.remove()
    cam = cv2.resize(cam, (x.shape[3], x.shape[2]), interpolation=cv2.INTER_LINEAR)
    cam = cam / cam.max() if cam.max() > 0 else cam
    return cam, c, torch.softmax(out.detach(), 1)[0].cpu().numpy()
'''

with open(os.path.join(WORK_DIR, "aitbc.py"), "w", encoding="utf-8") as fh:
    fh.write(LIB_SOURCE)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
import aitbc
aitbc = importlib.reload(aitbc)
aitbc.setup_plots()

if os.environ.get("AITBC_DATA"):
    cache_dir = os.environ["AITBC_DATA"]
elif CACHE_ON_DRIVE and IN_COLAB:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    cache_dir = "/content/drive/MyDrive/AITBC/data_cache"
else:
    cache_dir = os.path.join(WORK_DIR, "aitbc_data")
aitbc.set_data_root(cache_dir)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

t0 = time.time()
mv = aitbc.MilitaryVehicles()                      # 24 types of military vehicles
suland = aitbc.SULAND("v2")                        # refined annotation of SULAND
GALLERY_SEQ = ["ITA-v13", "ITA-v32", "ITA-v2", "ITA-v44", "US2", "US10"]
GALLERY = [(s, int(f)) for s in GALLERY_SEQ
           for f in suland.frames[(suland.frames.seq == s) & (suland.frames.n_boxes > 0)].frame.quantile([0.3, 0.7], interpolation="nearest")]
if PREFETCH_ALL:
    suland_v1 = aitbc.SULAND("v1")                 # original annotation of the authors
    suland.prefetch(GALLERY)
VERSIONS = {"Python": platform.python_version(), "NumPy": np.__version__, "OpenCV": cv2.__version__,
            "pandas": pd.__version__, "scikit-learn": sklearn.__version__, "scikit-image": skimage.__version__,
            "PyTorch": torch.__version__, "torchvision": torchvision.__version__, "aitbc": aitbc.__version__}
n_files, size_mb = aitbc.cache_size()
dev = torch.cuda.get_device_name(0) if DEVICE == "cuda" else f"CPU, {os.cpu_count()} logical cores"
display(pd.DataFrame({"Value": list(VERSIONS.values()) + ["Google Colab" if IN_COLAB else "local Jupyter", dev,
                                                           f"{aitbc.DATA_ROOT}: {n_files} files, {size_mb:.0f} MB",
                                                           f"{time.time() - t0:.0f} s"]},
                     index=list(VERSIONS) + ["environment", "computing device", "cache of the data sets", "preparation time"]))

### Block 0.2. Course project

**Purpose.** To connect the course project created in Notebook 02. The results of this notebook are saved in it: figures and tables in `outputs`, the trained model in the ONNX format in `models` and the parameters of the lessons in `project.json`.

**Parameters.** `USE_DRIVE` — the project is on Google Drive (Colab asks for access to the Google account); otherwise it is on the disk of the session. `PROJECT_PATH` — folder of the project; the same as in Notebooks 02–05.

**Output.** The folders of the project and the lessons already recorded in `project.json`.

In [ ]:
#@title 0.2 Course project { display-mode: "form" }
USE_DRIVE = True #@param {type:"boolean"}
PROJECT_PATH = "AITBC/AITBC_project" #@param {type:"string"}

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    PROJECT_ROOT = os.path.join("/content/drive/MyDrive", PROJECT_PATH)
else:
    PROJECT_ROOT = os.path.join(WORK_DIR, PROJECT_PATH)
    if USE_DRIVE:
        print("Google Drive is available only in Google Colab; the project is used in", PROJECT_ROOT)
for d in ["data/images", "data/video", "data/thermal", "data/annotations", "frames/visible", "frames/thermal",
          "models", "outputs/figures", "outputs/video", "outputs/reports"]:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
FIG_DIR = os.path.join(PROJECT_ROOT, "outputs", "figures")
REPORT_DIR = os.path.join(PROJECT_ROOT, "outputs", "reports")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")


def update_project_json(lesson, extra=None):
    path = os.path.join(PROJECT_ROOT, "project.json")
    meta = json.load(open(path, encoding="utf-8")) if os.path.exists(path) else {"project": os.path.basename(PROJECT_ROOT)}
    meta.setdefault("lessons", [])
    if lesson not in meta["lessons"]:
        meta["lessons"].append(lesson)
    meta["updated"] = time.strftime("%Y-%m-%d %H:%M:%S")
    meta["versions"] = VERSIONS
    meta.update(extra or {})
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(meta, fh, indent=2, ensure_ascii=False, default=str)
    print(f"project.json updated: lesson {lesson}" + (f", keys: {', '.join(extra)}" if extra else ""))


meta = json.load(open(os.path.join(PROJECT_ROOT, "project.json"), encoding="utf-8")) if os.path.exists(os.path.join(PROJECT_ROOT, "project.json")) else {}
print("Project:", PROJECT_ROOT)
print("Lessons recorded in project.json:", ", ".join(meta.get("lessons", [])) or "none (new project)")

---
# Lesson 4.1 (Lecture). Machine learning in the recognition of military objects

**Learning questions**
1. The tasks of the recognition of military objects and the place of machine learning in them.
2. Formation and annotation of a data set on frames of a video camera.
3. Features of the histogram of oriented gradients (HOG).
4. Classifiers: the support vector machine (SVM) and the random forest (Random Forest).
5. Metrics of the quality of classification: accuracy, precision, recall and F1.

**Recognition** answers the question *what is the object*. Three tasks are distinguished:
- **classification** — one class for an image or for a fragment of a frame (a crop): "T-72", "BTR-80", "background";
- **detection** — the boxes of all objects of the given classes in a frame and the class of each box (Topic 5);
- **segmentation** — the class of every pixel (Topic 5).

**Machine learning** replaces the rules written by the developer with a function fitted to examples: a set of images with known classes (the training sample) defines the parameters of the classifier, and a separate set (the test sample) estimates its quality. The chain of the classical approach is *image → features (HOG) → classifier (SVM, Random Forest) → class*; in a neural network (Lesson 4.2) the features are also fitted to the examples. In both cases the result is not better than the data: the lecture begins with the data set, its audit and its annotation.

### Block 4.1.1. The task of recognition and the data set of military vehicles

**Purpose.** To get acquainted with a real data set for the classification of military vehicles: the classes, the number of examples, the sizes and the sources of the images.

**Data set.** 9444 photographs collected by the authors from web sites (Kricheli et al., 2024): 7823 in the training part and 1621 in the test part, 24 fine types. The authors also give 7 coarse classes. The course uses its own grouping by the purpose of the vehicle, because two groups of the authors join different kinds of weapons:

| Group of the course | Types | Class of the authors |
|---|---|---|
| Tank | T-14, T-62, T-64, T-72, T-80, T-90 | Tank |
| IFV (infantry fighting vehicle) | BMP-1, BMP-2, BMP-T15, BMD | BMP, BMD |
| APC/recon (armoured personnel carrier and scout car) | BTR-60, BTR-70, BTR-80, BRDM, MT-LB | BTR, MT_LB |
| Artillery | 2S19 Msta (self-propelled howitzer), D-30 (**towed** howitzer) | Self Propelled Artillery |
| MLRS (multiple rocket launcher) | BM-30, Tornado, TOS-1 | Self Propelled Artillery |
| Air defence/radar | Pantsir-S1, 30N6E | Air Defense |
| Missile system | Iskander, RS-24 | Air Defense |

**Parameters.**
- `LEVEL` — the labels shown: the 24 types, the 7 groups of the course or the 7 classes of the authors.
- `EXAMPLES_PER_CLASS` — the number of examples of every class in the gallery; `SEED` — the choice of the examples.

**Expected result.** The table of the classes with the numbers of images, the gallery and the distribution of the sizes of the images.

**What to observe.** The number of training images differs between the types by a factor of 3.6 (from 119 for 30N6E to 424 for T-72), and the test part is distributed differently: from 6 % of the images of a type (BMP-1, T-64) to 35 % (BM-30). 90 % of the images are not larger than 300 × 300 pixels: they are web thumbnails of different origin. Among them there are photographs of real vehicles, renderings, boxes of scale models, infographics, explosions and photographs where the vehicle is not visible at all (with `SEED = 0`: the box of a model of the THAAD system under "30N6E", a person in an office under "RS-24"). The label of the type does not guarantee what the image contains.

**Control question 1.** Why does the class "Air Defense" of the authors make the task harder for a classifier, and what can a classifier learn from it?

In [ ]:
#@title 4.1.1 Classes and images of the data set of military vehicles { display-mode: "form" }
LEVEL = "24 types" #@param ["24 types", "7 groups of the course", "7 classes of the authors"]
EXAMPLES_PER_CLASS = 1 #@param {type:"slider", min:1, max:4, step:1}
SEED = 0 #@param {type:"integer"}

COL = {"24 types": "cls", "7 groups of the course": "group", "7 classes of the authors": "source_group"}
T = mv.table
col = COL[LEVEL]
counts = T.pivot_table(index=col, columns="split", values="file", aggfunc="count", fill_value=0)[["train", "test"]]
if col == "cls":
    counts.insert(0, "description", [aitbc.MilitaryVehicles.TYPES[c] for c in counts.index])
    counts.insert(1, "group of the course", [aitbc.MilitaryVehicles.GROUPS[c] for c in counts.index])
    counts.insert(2, "class of the authors", [T[T.cls == c].source_group.mode()[0] for c in counts.index])
else:
    counts.insert(0, "types (training images)", [", ".join(f"{k} {v}" for k, v in T[(T[col] == g) & (T.split == "train")].cls.value_counts().items())
                                                 for g in counts.index])
counts["share of test, %"] = (100 * counts.test / (counts.train + counts.test)).round(1)
display(counts.style.format(precision=1).set_caption(f"{LEVEL}: {len(counts)} classes, {int(counts.train.sum())} training and {int(counts.test.sum())} test images"))

rng = np.random.default_rng(SEED)
classes = list(counts.index)
ncol = 8 if EXAMPLES_PER_CLASS == 1 else EXAMPLES_PER_CLASS * 2
items = []
for c in classes:
    idx = T.index[(T[col] == c) & (T.split == "train")].values
    items += [(c, i) for i in rng.choice(idx, EXAMPLES_PER_CLASS, replace=False)]
nrow = int(math.ceil(len(items) / ncol))
fig, ax = plt.subplots(nrow, ncol, figsize=(2.3 * ncol, 2.0 * nrow))
for a in np.ravel(ax):
    a.axis("off")
for a, (c, i) in zip(np.ravel(ax), items):
    a.imshow(mv.image(i))
    a.set_title(f"{c}\n{T.width[i]}×{T.height[i]}", fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson41_gallery.png"), dpi=80)
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
counts[["train", "test"]].plot.bar(ax=ax[0], width=0.8)
ax[0].set_ylabel("images")
ax[0].set_title(LEVEL)
ax[0].tick_params(axis="x", labelsize=7.5)
ax[1].scatter(T.width, T.height, s=4, alpha=0.3)
ax[1].set_xlabel("width, px")
ax[1].set_ylabel("height, px")
ax[1].set_xscale("log")
ax[1].set_yscale("log")
ax[1].set_title(f"sizes of the images: median {int(T.width.median())} × {int(T.height.median())} px")
plt.tight_layout()
plt.show()
share_small = float(((T.width <= 300) & (T.height <= 300)).mean())
print(f"Images not larger than 300 × 300 px: {100 * share_small:.0f} %; ratio of the largest to the smallest class "
      f"in the training part: {counts.train.max() / counts.train.min():.1f}")

### Block 4.1.2. Audit of the data set: duplicates and contradictory labels

**Purpose.** To check the data set before training: to find repeated images inside a part, the images of the test part that also occur in the training part, and the contradictions of the labels of the same image.

**Method.**
- An **exact duplicate** is an image with the same file contents; it is found by the MD5 checksum of the file.
- A **near duplicate** is the same photograph after a change of the size, of the compression or of a small crop. It is found by the *difference hash* (dHash): the image is reduced to 9 × 8 pixels in grey, and every one of the 64 bits is the sign of the difference of two neighbouring pixels in a row. The distance between two hashes is the number of different bits (the Hamming distance $d_H$, from 0 to 64).

**Parameters.**
- `HAMMING_MAX` — the largest Hamming distance at which two images are regarded as near duplicates.
- `SHOW_PAIRS` — the number of pairs in the gallery.

**Expected result.** The table of the audit and the gallery of pairs *test image — the nearest training image* with their labels and the distance.

**What to observe.** In the test part 18 images are byte-for-byte copies of training images, and **all 18 have a different label in the training part** (BM-30 and Tornado, BTR-60 and BTR-80, T-90 and T-72 and others). Such an image cannot be classified correctly by any model, and a model that remembers the training image is counted wrong. After the removal of the exact copies 98 test images (6 %) still have a near duplicate at $d_H \le 2$ in the training part: the same photograph re-compressed or resized, 7 of them with a different label. At $d_H \ge 4$ also different vehicles on a uniform background (renderings) come together: the threshold is a compromise checked by eye. Blocks 4.1.5, 4.2.2 and 4.2.5 use the test part without exact copies and without near duplicates at $d_H \le 2$: 1502 images.

**Control question 2.** Why must the duplicates between the training and the test part be removed before the estimation of the quality, and what is done with the contradictory labels?

In [ ]:
#@title 4.1.2 Exact and near duplicates, contradictory labels { display-mode: "form" }
HAMMING_MAX = 2 #@param {type:"slider", min:0, max:12, step:1}
SHOW_PAIRS = 8 #@param {type:"slider", min:4, max:16, step:4}

T = mv.table
TRAIN_IDX, TEST_IDX = mv.clean()                     # parts without exact copies (MD5)
t0 = time.time()
HASHES = mv.hashes()                                 # dHash of all images (computed once)
print(f"dHash of {len(T)} images: {time.time() - t0:.1f} s")
dist, nn = aitbc.hamming_nearest(HASHES[TEST_IDX], HASHES[TRAIN_IDX])
near = pd.DataFrame({"test": TEST_IDX, "train": TRAIN_IDX[nn], "d": dist})
near["cls_test"] = T.cls.values[near.test]
near["cls_train"] = T.cls.values[near.train]
near["same"] = near.cls_test == near.cls_train
sel = near[near.d <= HAMMING_MAX]
exact = T[T.in_train]
audit = pd.DataFrame({"value": [
    int(T.dup_in_split[T.split == "train"].sum()), int(T.dup_in_split[T.split == "test"].sum()),
    len(exact), int((exact.train_label != exact.cls).sum()),
    f"{len(TRAIN_IDX)} / {len(TEST_IDX)}",
    len(sel), int((sel.cls_test != sel.cls_train).sum()), len(mv.clean(near=2)[1])]},
    index=["repeated copies inside the training part (MD5)", "repeated copies inside the test part (MD5)",
           "test images that are copies of training images (MD5)", "   of them with a different label",
           "cleaned parts: training / test images",
           f"near duplicates (dHash, d ≤ {HAMMING_MAX}) of cleaned test images in the training part",
           "   of them with a different label",
           "test images used in blocks 4.1.5, 4.2.2, 4.2.5 (without exact copies and near duplicates d ≤ 2)"])
display(audit)
display(exact.groupby(["cls", "train_label"]).size().rename("test images").reset_index()
        .rename(columns={"cls": "label in the test part", "train_label": "label of the same file in the training part"}))

pool = sel if len(sel) >= SHOW_PAIRS else near
show = pool.sort_values(["same", "d"]).head(SHOW_PAIRS // 2)                       # first the contradictory labels
show = pd.concat([show, pool[pool.same].sort_values("d", ascending=False).head(SHOW_PAIRS - len(show))])
fig, ax = plt.subplots(2, len(show), figsize=(2.3 * len(show), 4.6), squeeze=False)
for j, r in enumerate(show.itertuples()):
    ax[0, j].imshow(mv.image(r.test))
    ax[0, j].set_title(f"test: {r.cls_test}\nd = {r.d}", fontsize=8, color="k" if r.cls_test == r.cls_train else "#c62828")
    ax[1, j].imshow(mv.image(r.train))
    ax[1, j].set_title(f"training: {r.cls_train}", fontsize=8)
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson41_duplicates.png"), dpi=80)
plt.show()
counts_d = near.d.value_counts().sort_index()
print("Number of cleaned test images by the distance to the nearest training image, d = 0...8:",
      ", ".join(f"{int(counts_d.get(k, 0))}" for k in range(9)))

### Block 4.1.3. Annotation of frames: the YOLO format, statistics and quality of the annotation

**Purpose.** To study the annotation of objects on the frames of a video camera by the example of the SULAND data set: the format of the boxes, the transformation of coordinates, the statistics of a data set for detection and the errors of the annotation.

**SULAND.** A camera (iPhone 13, 1440 × 1920 pixels, vertical frame) is moved at a height of about one metre over grass, fallen leaves and gravel, as a camera of a ground robot for the survey of an area. On the ground lie surrogates of two anti-personnel surface landmines: **PFM-1** ("butterfly", a scattered mine) and **PMA-2** ("starfish"). The frames are cut from 55 video sequences: 45 recorded in Italy (training, validation and test parts) and 10 recorded in the USA (a separate part with other soil and lighting). The refined annotation **SULAND_v2** (2026) corrects the original annotation **v1** of the authors (2024): missing boxes, false boxes, inaccurate boundaries and the inverted numbers of the classes in the USA part.

**YOLO format.** For every frame `image.jpg` there is a text file `image.txt`, one line per object:

`class x_c y_c w h`, where $x_c, y_c$ are the coordinates of the centre of the box and $w, h$ its width and height, as fractions of the width $W$ and the height $H$ of the frame (0…1). A frame without objects has an empty file. Transformation into pixels:

$$x_1 = (x_c - w/2)\,W,\quad y_1 = (y_c - h/2)\,H,\quad x_2 = (x_c + w/2)\,W,\quad y_2 = (y_c + h/2)\,H;$$

in the COCO format the box is $[x_1, y_1, x_2 - x_1, y_2 - y_1]$ in pixels.

**Parameters.**
- `SEQUENCE` — a sequence (ITA — Italy, US — USA); `FRAME_POSITION` — the position of the frame in the sequence (0 — first, 1 — last frame).
- `SHOW_V1` — draw also the boxes of the original annotation v1 (white).

**Expected result.** The frame with the boxes, the lines of its file in the YOLO format with their transformation into pixels, the statistics of the parts and the comparison of the two versions of the annotation.

**What to observe.** 72–76 % of the Italian frames contain no mine (19 % of the USA frames): a detector must learn the frames without objects too. The median box of a mine is about 95 × 80 px in the Italian part (6.6 × 4.2 % of the frame 1440 × 1920) and about 190 × 170 px in the USA part. Between v1 and v2 the boxes of the same mines in the Italian part differ by a median IoU of 0.62 (34 % of them by IoU < 0.5), 1951 boxes were added and 436 removed; in the USA part 3602 of 3856 boxes changed their class (the inverted numbers of the classes in v1). The annotation is a source of errors comparable with the errors of the model.

**Control question 3.** A frame of 1440 × 1920 px has the line `1 0.5 0.25 0.1 0.05`. Which mine is it, and what are its box in pixels and its box in the COCO format?

In [ ]:
#@title 4.1.3 Annotation of SULAND frames, statistics, versions v1 and v2 { display-mode: "form" }
SEQUENCE = "ITA-v13" #@param ["ITA-v13", "ITA-v32", "ITA-v2", "ITA-v44", "US2", "US10"]
FRAME_POSITION = 0.3 #@param {type:"slider", min:0, max:1, step:0.05}
SHOW_V1 = True #@param {type:"boolean"}

if "suland_v1" not in globals():
    suland_v1 = aitbc.SULAND("v1")
fr_seq = suland.frames[(suland.frames.seq == SEQUENCE) & (suland.frames.n_boxes > 0)].frame.values
frame_no = int(fr_seq[int(round(FRAME_POSITION * (len(fr_seq) - 1)))])
img, objs = suland.load(SEQUENCE, frame_no)
H, W = img.shape[:2]
vis = aitbc.draw_boxes(img, objs, colors=aitbc.SULAND.COLORS, thickness=4, font=1.4)
objs1 = suland_v1.objects(SEQUENCE, frame_no, W, H)
if SHOW_V1:
    for o in objs1:
        x1, y1, x2, y2 = [int(round(v)) for v in o["box"]]
        cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 255, 255), 3, cv2.LINE_AA)
part = suland.frames[suland.frames.seq == SEQUENCE].iloc[0]
fig, ax = plt.subplots(1, 2, figsize=(15, 9), gridspec_kw={"width_ratios": [1, 1]})
ax[0].imshow(vis)
ax[0].set_title(f"{SEQUENCE} ({part.domain}, {part.split}), frame {frame_no}: v2 coloured" + (", v1 white" if SHOW_V1 else ""), fontsize=9)
ax[0].axis("off")
if objs:
    o = objs[0]
    x1, y1, x2, y2 = [int(round(v)) for v in o["box"]]
    m = int(1.0 * max(x2 - x1, y2 - y1))
    ax[1].imshow(vis[max(0, y1 - m):y2 + m, max(0, x1 - m):x2 + m])
    ax[1].set_title(f"enlarged: {o['cls']} ({aitbc.SULAND.NICKNAMES[o['cls']]}), box {x2 - x1} × {y2 - y1} px", fontsize=9)
ax[1].axis("off")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"lesson41_annotation_{SEQUENCE}_{frame_no}.png"), dpi=70)
plt.show()

rows = []
for r in suland.boxes[(suland.boxes.seq == SEQUENCE) & (suland.boxes.frame == frame_no)].itertuples():
    b = aitbc.yolo_to_xyxy(r.cx, r.cy, r.w, r.h, W, H)
    rows.append({"YOLO line (v2)": f"{r.cls_id} {r.cx:.4f} {r.cy:.4f} {r.w:.4f} {r.h:.4f}", "class": r.cls,
                 "[x1, y1, x2, y2], px": [round(v) for v in b], "COCO [x, y, w, h], px": [round(v) for v in aitbc.xyxy_to_coco(b)]})
display(pd.DataFrame(rows))
print("Boxes of the same frame in v1 (class, [x1, y1, x2, y2]):", [(o["cls"], [round(v) for v in o["box"]]) for o in objs1] or "none")

# statistics of the parts (annotation v2)
F, Bx = suland.frames, suland.boxes
stats = F.groupby(["domain", "split"]).agg(sequences=("seq", "nunique"), frames=("frame", "size"),
                                           frames_without_mines=("n_boxes", lambda s: f"{100 * (s == 0).mean():.0f} %"),
                                           boxes_PFM1=("n_pfm1", "sum"), boxes_PMA2=("n_pma2", "sum"))
wpx = Bx.w * 1440
hpx = Bx.h * 1920
stats["median box, px"] = [f"{wpx[(Bx.domain == d) & (Bx.split == s)].median():.0f} × {hpx[(Bx.domain == d) & (Bx.split == s)].median():.0f}"
                           for d, s in stats.index]
display(stats.style.set_caption("SULAND_v2: parts of the data set"))


# comparison of the versions of the annotation: a box of v2 is matched with the box of v1 of the largest IoU (IoU ≥ 0.1)
def compare_versions(ga, gb):
    # ga, gb: {(sequence, frame): boxes} of the old and the new annotation
    out = dict(added=0, removed=0, class_changed=0, ious=[])
    for key in set(ga) | set(gb):
        A = [] if key not in ga else [(r.cls, aitbc.yolo_to_xyxy(r.cx, r.cy, r.w, r.h, 1, 1)) for r in ga[key].itertuples()]
        B = [] if key not in gb else [(r.cls, aitbc.yolo_to_xyxy(r.cx, r.cy, r.w, r.h, 1, 1)) for r in gb[key].itertuples()]
        used = set()
        for cb, bb in B:
            best, bi = 0.0, None
            for i, (ca, ba) in enumerate(A):
                if i not in used:
                    v = aitbc.box_iou(ba, bb)
                    if v > best:
                        best, bi = v, i
            if bi is None or best < 0.1:
                out["added"] += 1
            else:
                used.add(bi)
                out["ious"].append(best)
                out["class_changed"] += int(A[bi][0] != cb)
        out["removed"] += len(A) - len(used)
    return out


if "VERSION_CMP" not in globals():
    VERSION_CMP = {}
    for dom in ("data-iid", "data-ood"):
        VERSION_CMP[dom] = compare_versions({k: g for k, g in suland_v1._by_frame.items() if g.domain.iat[0] == dom},
                                            {k: g for k, g in suland._by_frame.items() if g.domain.iat[0] == dom})
cmp_rows = {}
for dom, c in VERSION_CMP.items():
    iou = np.array(c["ious"])
    cmp_rows[dom] = {"boxes v1": int((suland_v1.boxes.domain == dom).sum()), "boxes v2": int((suland.boxes.domain == dom).sum()),
                     "added in v2": c["added"], "removed in v2": c["removed"], "class changed": c["class_changed"],
                     "median IoU of the same box v1-v2": round(float(np.median(iou)), 3),
                     "share of the same boxes with IoU < 0.5": f"{100 * (iou < 0.5).mean():.0f} %"}
display(pd.DataFrame(cmp_rows).T.style.set_caption("Original annotation v1 and refined annotation v2"))
fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
ax[0].hist(np.clip(wpx, 0, 600), bins=60, alpha=0.7, label="width")
ax[0].hist(np.clip(hpx, 0, 600), bins=60, alpha=0.7, label="height")
ax[0].set_xlabel("size of the box, px (frame 1440 × 1920)")
ax[0].legend()
ax[1].hist(np.array(VERSION_CMP["data-iid"]["ious"]), bins=40)
ax[1].set_xlabel("IoU of the same mine in v1 and v2 (Italy)")
plt.tight_layout()
plt.show()

### Block 4.1.4. HOG features

**Purpose.** To compute the histogram of oriented gradients (HOG, Dalal and Triggs, 2005) of an image of a military vehicle and of a fragment of a SULAND frame, and to relate its parameters to the length of the feature vector.

**Formulas.**
1. Gradient of brightness in every pixel: $g_x = I(x+1, y) - I(x-1, y)$, $g_y = I(x, y+1) - I(x, y-1)$; magnitude $m = \sqrt{g_x^2 + g_y^2}$; orientation $\theta = \arctan(g_y / g_x)$ in $[0°, 180°)$ (the sign of the edge is not used).
2. **Cell** of $c \times c$ pixels: a histogram of $n_o$ bins of orientation, every pixel adds its magnitude $m$ to the bin of its orientation $\theta$.
3. **Block** of $b \times b$ cells: the histograms of its cells are joined into a vector $v$ and normalised, $v \leftarrow v / \sqrt{\lVert v \rVert^2 + \varepsilon^2}$ (L2-Hys: values above 0.2 are clipped and the vector is normalised again). The normalisation makes the features weakly dependent on the illumination and the contrast; the blocks overlap with a step of one cell.
4. Length of the vector of an image of $S \times S$ pixels:
$$N = \left(\frac{S}{c} - b + 1\right)^2 b^2\, n_o; \qquad S = 64,\; c = 8,\; b = 2,\; n_o = 9:\quad N = 7^2 \cdot 4 \cdot 9 = 1764.$$

**Parameters.**
- `IMAGE` — the image: a military vehicle of the data set of block 4.1.1 or a fragment of a SULAND frame around a mine or without a mine (background).
- `SIZE` — the size $S$ to which the image is reduced; `CELL` — $c$; `BLOCK` — $b$; `ORIENTATIONS` — $n_o$.

**Expected result.** The image, the magnitude of the gradient, the picture of the HOG (in every cell the dominating orientations as segments) and the comparison of the length $N$ by the formula with the length computed by `skimage.feature.hog`.

**What to observe.** The HOG keeps the shape (the contours of the hull, the barrel, the wheels; the rays of the PMA-2) and loses the colour and the small texture. A small cell gives a long vector with details, a large cell a short and coarse vector. The background of grass gives cells with many orientations of similar weight, the mine gives cells with a dominating orientation.

**Control question 4.** What length has the HOG vector for $S = 96$, $c = 8$, $b = 2$, $n_o = 9$, and why does it grow faster than the area of the image in cells?

In [ ]:
#@title 4.1.4 Histogram of oriented gradients { display-mode: "form" }
IMAGE = "T-72 (tank)" #@param ["T-72 (tank)", "BM-30 (multiple rocket launcher)", "BTR-80 (armoured personnel carrier)", "SULAND: PFM-1 (butterfly)", "SULAND: PMA-2 (starfish)", "SULAND: background"]
SIZE = 64 #@param {type:"slider", min:32, max:128, step:16}
CELL = 8 #@param [4, 8, 16] {type:"raw"}
BLOCK = 2 #@param [1, 2, 3] {type:"raw"}
ORIENTATIONS = 9 #@param [6, 9, 12, 18] {type:"raw"}

from skimage.feature import hog


def suland_crop(kind, scale=1.5):
    # fragment of a gallery frame of block 0.1: around the first mine of the class, or without mines
    for s, f in GALLERY:
        img, objs = suland.load(s, f)
        if kind == "background":
            x0, y0 = 100, img.shape[0] - 400
            return img[y0:y0 + 220, x0:x0 + 220], f"{s}, frame {f}: background"
        for o in objs:
            if o["cls"] == kind:
                x1, y1, x2, y2 = o["box"]
                side = int(scale * max(x2 - x1, y2 - y1))
                cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
                return img[max(0, cy - side // 2):cy + side // 2, max(0, cx - side // 2):cx + side // 2], f"{s}, frame {f}: {kind}"
    raise ValueError(kind)


if IMAGE.startswith("SULAND"):
    kind = "background" if "background" in IMAGE else IMAGE.split(": ")[1].split(" ")[0]
    src, caption = suland_crop(kind)
else:
    t_ = IMAGE.split(" ")[0]
    i_ = mv.table.index[(mv.table.cls == t_) & (mv.table.split == "train")][3]
    src, caption = mv.image(i_), f"military vehicles: {t_}, image {i_}"
g = cv2.resize(aitbc.gray(src), (SIZE, SIZE), interpolation=cv2.INTER_AREA)
if SIZE // CELL < BLOCK:
    raise ValueError("the image must contain at least BLOCK cells")
t0 = time.time()
vec, hog_img = hog(g, orientations=ORIENTATIONS, pixels_per_cell=(CELL, CELL), cells_per_block=(BLOCK, BLOCK), visualize=True)
dt = (time.time() - t0) * 1000
gx = cv2.Sobel(g.astype(np.float32), cv2.CV_32F, 1, 0, ksize=1)
gy = cv2.Sobel(g.astype(np.float32), cv2.CV_32F, 0, 1, ksize=1)
fig, ax = plt.subplots(1, 4, figsize=(17, 4.4))
ax[0].imshow(src)
ax[0].set_title(caption, fontsize=8.5)
ax[1].imshow(g, cmap="gray")
ax[1].set_title(f"grey, {SIZE} × {SIZE} px, cells {CELL} × {CELL}", fontsize=8.5)
for k in range(0, SIZE + 1, CELL):
    ax[1].axhline(k - 0.5, color="y", lw=0.4)
    ax[1].axvline(k - 0.5, color="y", lw=0.4)
ax[2].imshow(np.hypot(gx, gy), cmap="magma")
ax[2].set_title("magnitude of the gradient m", fontsize=8.5)
ax[3].imshow(hog_img, cmap="gray")
ax[3].set_title(f"HOG: {ORIENTATIONS} orientations per cell", fontsize=8.5)
for a in ax:
    a.axis("off")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson41_hog.png"), dpi=80)
plt.show()
n_formula = aitbc.hog_length(SIZE, CELL, BLOCK, ORIENTATIONS)
display(pd.DataFrame({"value": [f"{SIZE // CELL} × {SIZE // CELL}", f"{SIZE // CELL - BLOCK + 1} × {SIZE // CELL - BLOCK + 1}",
                                n_formula, len(vec), f"{dt:.1f} ms"]},
                     index=["cells", "blocks (step of one cell)", "length N by the formula", "length computed by skimage",
                            "time of the computation for one image"]))

### Block 4.1.5. SVM and Random Forest classifiers; accuracy, precision, recall, F1

**Purpose.** To train classifiers of military vehicles on HOG vectors and to estimate them by the metrics of classification on the cleaned test part (block 4.1.2).

**Classifiers.**
- **Linear SVM.** The decision function $f(x) = w^\top x + b$; the hyperplane is chosen with the largest margin to the nearest examples of the classes (support vectors): $\min_{w, b}\ \tfrac{1}{2}\lVert w \rVert^2 + C \sum_i \max(0,\, 1 - y_i f(x_i))$. Here it is trained by stochastic gradient descent (`SGDClassifier`, hinge loss); several classes — "one class against all".
- **SVM with the RBF kernel.** $f(x) = \sum_i \alpha_i y_i \exp(-\gamma \lVert x - x_i \rVert^2) + b$: the boundary between the classes is curved. $C$ is the price of an error on the training examples: a large $C$ follows the training examples closely.
- **Random Forest.** $T$ decision trees, each trained on a random sample of the examples with a random subset of the features in every node; the class is chosen by the vote of the trees.

**Metrics.** For class $k$: $TP$ — images of class $k$ recognised as $k$, $FP$ — images of other classes recognised as $k$, $FN$ — images of class $k$ recognised as another class.
$$\text{precision}_k = \frac{TP}{TP + FP},\qquad \text{recall}_k = \frac{TP}{TP + FN},\qquad F1_k = \frac{2\,\text{precision}_k\,\text{recall}_k}{\text{precision}_k + \text{recall}_k},\qquad \text{accuracy} = \frac{\text{correct}}{\text{all}}.$$
The macro average is the mean over the classes: every class counts equally, regardless of its number of images. The confusion matrix shows in row $k$ the shares of the images of class $k$ recognised as every class (the diagonal is the recall).

**Parameters.**
- `LEVEL` — 24 types or 7 groups of the course; `N_PER_CLASS` — the largest number of training images of every type (400 — practically all).
- `CLASSIFIER`, `C` (SVM), `N_TREES` (Random Forest). HOG: 64 × 64 px, cells 8 × 8, blocks 2 × 2, 9 orientations (1764 values).

**Expected result.** The table of the metrics, the metrics of every class, the confusion matrix, the most frequent confusions and examples of errors.

**What to observe.** With 150 images per type the SVM with the RBF kernel reaches an accuracy of 0.18 on the 24 types (random choice: 0.04) and 0.39 on the 7 groups (random choice: 0.14); the linear SVM 0.14 (with C = 0.01). The most frequent confusions are between vehicles with a similar silhouette: BTR-60 → BTR-70, BM-30 → Iskander, RS-24 → BM-30, Iskander → RS-24 (long wheeled chassis with a launcher); on the 7 groups IFV → Tank and APC/recon → Tank. The HOG of a thumbnail reduced to 64 × 64 px does not contain the details that distinguish the types. The difference between the accuracy and the macro F1 points to unequal classes.

**Control question 5.** A classifier recognises 90 of 100 tanks; of 120 images recognised as tanks 30 are not tanks. What are the precision, the recall and the F1 for the class "tank"?

In [ ]:
#@title 4.1.5 HOG + SVM / Random Forest: training and metrics { display-mode: "form" }
LEVEL_41 = "24 types" #@param ["24 types", "7 groups of the course"]
N_PER_CLASS = 150 #@param {type:"slider", min:25, max:400, step:25}
CLASSIFIER = "RBF SVM" #@param ["RBF SVM", "linear SVM", "Random Forest"]
C = 10 #@param [0.01, 0.1, 1, 10, 100] {type:"raw"}
N_TREES = 300 #@param {type:"slider", min:50, max:500, step:50}

import warnings
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.exceptions import ConvergenceWarning

TRAIN_IDX, TEST_IDX = mv.clean(near=2)          # without exact copies and near duplicates (block 4.1.2)
if "HOG_ALL" not in globals():
    t0 = time.time()
    HOG_ALL = aitbc.hog_features([mv.image(i, size=64, gray=True) for i in range(len(mv.table))], size=64)
    print(f"HOG of {len(mv.table)} images: {HOG_ALL.shape[1]} values, {time.time() - t0:.0f} s")
lab_col = "cls" if LEVEL_41 == "24 types" else "group"
tr = mv.subset(TRAIN_IDX, N_PER_CLASS, seed=0)
y_tr, y_te = mv.table[lab_col].values[tr], mv.table[lab_col].values[TEST_IDX]
if CLASSIFIER == "RBF SVM":
    model = make_pipeline(StandardScaler(), SVC(C=C, gamma="scale"))
elif CLASSIFIER == "linear SVM":
    model = make_pipeline(StandardScaler(), SGDClassifier(loss="hinge", alpha=1.0 / (C * len(tr)), max_iter=100, tol=1e-4, random_state=0))
else:
    model = RandomForestClassifier(N_TREES, n_jobs=-1, random_state=0)
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    model.fit(HOG_ALL[tr], y_tr)
t_fit = time.time() - t0
t0 = time.time()
pred = model.predict(HOG_ALL[TEST_IDX])
t_pred = (time.time() - t0) / len(TEST_IDX) * 1000
labels = sorted(set(y_te) | set(y_tr)) if lab_col == "cls" else list(aitbc.MilitaryVehicles.GROUP_ORDER)
P, R, F1, S = precision_recall_fscore_support(y_te, pred, labels=labels, zero_division=0)
acc = accuracy_score(y_te, pred)
summary = pd.DataFrame({"value": [f"{len(tr)} ({N_PER_CLASS} per type at most)", len(TEST_IDX), f"{1 / len(labels):.3f}",
                                  f"{acc:.3f}", f"{P.mean():.3f}", f"{R.mean():.3f}", f"{F1.mean():.3f}",
                                  f"{t_fit:.1f} s", f"{t_pred:.2f} ms"]},
                       index=["training images", "test images (cleaned)", "accuracy of a random choice", "accuracy",
                              "precision (macro)", "recall (macro)", "F1 (macro)", "time of training", "time of recognition of one image"])
display(summary.style.set_caption(f"HOG + {CLASSIFIER}, {LEVEL_41}"))
per_class = pd.DataFrame({"precision": P, "recall": R, "F1": F1, "test images": S}, index=labels).round(3)
display(per_class.sort_values("F1", ascending=False).style.background_gradient(subset=["F1"], cmap="RdYlGn", vmin=0, vmax=1)
        .format(precision=3).set_caption("Metrics of every class"))
cm = confusion_matrix(y_te, pred, labels=labels)
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(0.42 * len(labels) + 4, 0.38 * len(labels) + 3))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=90, fontsize=8)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("recognised class")
ax.set_ylabel("true class")
if len(labels) <= 8:
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=8, color="w" if cmn[i, j] > 0.5 else "k")
plt.colorbar(im, fraction=0.03)
ax.set_title(f"Confusion matrix (shares of the true class), accuracy {acc:.3f}", fontsize=9.5)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"lesson41_confusion_{'types' if lab_col == 'cls' else 'groups'}.png"), dpi=80)
plt.show()
off = [(cm[i, j], labels[i], labels[j]) for i in range(len(labels)) for j in range(len(labels)) if i != j]
display(pd.DataFrame(sorted(off, reverse=True)[:8], columns=["test images", "true class", "recognised as"]))
wrong = np.where(pred != y_te)[0][:8]
fig, ax = plt.subplots(1, len(wrong), figsize=(2.2 * len(wrong), 2.4))
for a, k in zip(np.ravel(ax), wrong):
    a.imshow(mv.image(TEST_IDX[k]))
    a.set_title(f"true: {y_te[k]}\nrecognised: {pred[k]}", fontsize=7.5)
    a.axis("off")
plt.tight_layout()
plt.show()
RESULTS_41 = globals().get("RESULTS_41", {})
RESULTS_41[(LEVEL_41, CLASSIFIER, N_PER_CLASS)] = dict(accuracy=round(acc, 3), f1_macro=round(float(F1.mean()), 3))
per_class.to_csv(os.path.join(REPORT_DIR, f"lesson41_metrics_{CLASSIFIER.replace(' ', '_')}_{'types' if lab_col == 'cls' else 'groups'}.csv"))
update_project_json("4.1", {"lesson_4_1": dict(level=LEVEL_41, classifier=CLASSIFIER, n_per_class=N_PER_CLASS, C=C,
                                               accuracy=round(acc, 3), f1_macro=round(float(F1.mean()), 3))})

### Block 4.1.6. Division of the data set: random frames or whole sequences

**Purpose.** To measure how the estimation of the quality depends on the way the frames of video sequences are divided into the training and the test part.

**Experiment.** From every one of `N_SEQUENCES` Italian sequences of SULAND a run of `FRAMES_PER_SEQUENCE` consecutive frames with the most mines is taken. From every frame the fragments around the mines (a square of 1.5 × the longer side of the box) and two fragments of the same size without mines are cut and reduced to 64 × 64 px; the classifier works on their HOG vectors. The quality is estimated by cross-validation with `N_FOLDS` folds in two ways:
- **random frames** — every frame goes into a random fold; the neighbouring frames of the same sequence (the same mine, the same background, the same light) are both in the training and in the test part;
- **whole sequences** — all frames of a sequence go into the same fold; the test part contains only places and mines not seen during the training, as in real use.

**Parameters.** `N_SEQUENCES`, `FRAMES_PER_SEQUENCE` (frames of about 0.5 MB are downloaded once), `TASK` — mine or background / PFM-1 or PMA-2, `CLASSIFIER`, `N_FOLDS`.

**Expected result.** The table of the accuracy and of the macro F1 for the two divisions and examples of the fragments.

**What to observe.** With 12 sequences of 30 frames (1099 fragments: 720 background, 306 PMA-2, 73 PFM-1) the SVM with the RBF kernel reaches an accuracy of 0.98 by random frames and 0.83 by whole sequences in the task "mine or background"; for "PFM-1 or PMA-2" the macro F1 falls from 0.98 to 0.46. The division by random frames overestimates the quality: the classifier recognises the place, not the mine. For this reason the authors of SULAND give the parts as whole sequences, and the test of a model is made on sequences (flights, places, days) not used in the training.

**Control question 6.** A data set contains frames of 20 reconnaissance flights over 5 areas. How should the training, validation and test parts be formed, and why?

In [ ]:
#@title 4.1.6 Random frames or whole sequences: cross-validation on SULAND fragments { display-mode: "form" }
N_SEQUENCES = 12 #@param {type:"slider", min:4, max:20, step:1}
FRAMES_PER_SEQUENCE = 30 #@param {type:"slider", min:10, max:40, step:5}
TASK = "mine or background" #@param ["mine or background", "PFM-1 or PMA-2"]
CLASSIFIER_46 = "RBF SVM" #@param ["RBF SVM", "linear SVM", "Random Forest"]
N_FOLDS = 4 #@param {type:"slider", min:3, max:6, step:1}

import warnings
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score

seq_tab = suland.sequences()
SEQ_POOL = seq_tab[(seq_tab.domain == "data-iid") & (seq_tab.pfm1 > 40) & (seq_tab.pma2 > 40)].sort_values("boxes", ascending=False).seq.tolist()
key = (N_SEQUENCES, FRAMES_PER_SEQUENCE)
CROPS = globals().get("CROPS", {})
if key not in CROPS:
    t0 = time.time()
    CROPS[key] = suland.crop_set(SEQ_POOL[:N_SEQUENCES], FRAMES_PER_SEQUENCE, size=64, seed=0)
    print(f"{len(CROPS[key])} fragments of {CROPS[key].groupby(['seq', 'frame']).ngroups} frames: {time.time() - t0:.0f} s")
crops = CROPS[key]
mask = (crops.label != "background").values if TASK == "PFM-1 or PMA-2" else np.ones(len(crops), bool)
cr = crops[mask].reset_index(drop=True)
y = (cr.label != "background").astype(int).values if TASK == "mine or background" else (cr.label == "PMA-2").astype(int).values
X = aitbc.hog_features(list(cr.img), size=64)
print("Fragments by class:", cr.label.value_counts().to_dict())


def make_model():
    if CLASSIFIER_46 == "RBF SVM":
        return make_pipeline(StandardScaler(), SVC(C=10, gamma="scale"))
    if CLASSIFIER_46 == "linear SVM":
        return make_pipeline(StandardScaler(), SGDClassifier(loss="hinge", alpha=1e-3, max_iter=200, tol=1e-4, random_state=0))
    return RandomForestClassifier(300, n_jobs=-1, random_state=0)


n_folds = min(N_FOLDS, cr.seq.nunique())            # a fold must contain at least one whole sequence
if n_folds < N_FOLDS:
    print(f"N_FOLDS = {N_FOLDS} is larger than the number of sequences: {n_folds} folds are used")
rows = {}
for name, groups in (("random frames", (cr.seq + "-" + cr.frame.astype(str)).values), ("whole sequences", cr.seq.values)):
    acc, f1 = [], []
    for k_tr, k_te in StratifiedGroupKFold(n_folds, shuffle=True, random_state=0).split(X, y, groups):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            m = make_model().fit(X[k_tr], y[k_tr])
        p = m.predict(X[k_te])
        acc.append(accuracy_score(y[k_te], p))
        f1.append(f1_score(y[k_te], p, average="macro"))
    rows[name] = {"accuracy": np.mean(acc), "± (folds)": np.std(acc), "F1 (macro)": np.mean(f1)}
LEAK = pd.DataFrame(rows).T
display(LEAK.style.format(precision=3).set_caption(f"{TASK}, HOG + {CLASSIFIER_46}, {N_SEQUENCES} sequences × {FRAMES_PER_SEQUENCE} frames, {n_folds} folds"))
fig, ax = plt.subplots(1, 2, figsize=(15, 3.2), gridspec_kw={"width_ratios": [1, 3]})
LEAK[["accuracy", "F1 (macro)"]].plot.bar(ax=ax[0], rot=0, ylim=(0, 1))
ax[0].set_title("estimated quality", fontsize=9)
show = pd.concat([crops[crops.label == c].head(6) for c in ("PFM-1", "PMA-2", "background")])
tiles = [np.pad(i, 2, constant_values=255) for i in show.img]
ax[1].imshow(np.hstack(tiles), cmap="gray")
ax[1].set_title("fragments 64 × 64 px: 6 PFM-1, 6 PMA-2, 6 background", fontsize=9)
ax[1].axis("off")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson41_division.png"), dpi=80)
plt.show()
LEAK.to_csv(os.path.join(REPORT_DIR, "lesson41_division_of_data_set.csv"))

---
# Lesson 4.2 (Group class). Neural networks: from the neuron to MobileNet, transfer learning and ONNX

**Learning questions**
1. The artificial neuron and the fully connected neural network.
2. The convolutional neural network (CNN): convolution, feature maps, pooling.
3. MobileNet: an architecture for onboard computers of UAVs and ground robots.
4. Transfer learning: the use of a network trained on a large data set for a new task.
5. Export of the trained model to the ONNX format for the onboard computer.

**Order of work of the group.** Every block is run with the default parameters, the result is discussed by the group, then every subgroup changes one parameter (the subgroups take different values) and reports its measurement. The answers to the control questions and the measurements are written in the **Report of Lesson 4.2** at the end of the lesson.

### Block 4.2.1. The artificial neuron

**Purpose.** To study the computation of a neuron and the training of its weights by gradient descent on real data: the fragments of SULAND frames with a mine and without a mine (block 4.1.6).

**Formulas.**
- Neuron: $z = w_1 x_1 + w_2 x_2 + b$, output $y = \varphi(z)$; activation functions: sigmoid $\sigma(z) = 1 / (1 + e^{-z})$, $\tanh z$, ReLU $\max(0, z)$, step $[z > 0]$. The decision "mine" is $z > 0$: the boundary between the classes is the straight line $w_1 x_1 + w_2 x_2 + b = 0$.
- Training of a neuron with the sigmoid (logistic regression) minimises the cross-entropy $L = -\frac{1}{n}\sum_i [t_i \ln y_i + (1 - t_i) \ln (1 - y_i)]$, $t_i \in \{0, 1\}$; its gradient is $\partial L / \partial w_j = \frac{1}{n}\sum_i (y_i - t_i)\, x_{ij}$, $\partial L / \partial b = \frac{1}{n}\sum_i (y_i - t_i)$, and one step of gradient descent is $w \leftarrow w - \eta\, \partial L / \partial w$.
- Inputs: the HOG vector of 1764 values of a fragment is reduced to two numbers $x_1, x_2$ by the principal component analysis (the two directions of the largest variance of the HOG vectors), scaled to zero mean and unit variance.

**Parameters.** `W1`, `W2`, `B` — the weights set by hand; `ACTIVATION`; `TRAIN` — train the weights by gradient descent from zero with `LEARNING_RATE` $\eta$ during `EPOCHS` steps.

**Expected result.** The diagram of the neuron, the fragments on the plane $(x_1, x_2)$ with the boundary of the neuron set by hand (dashed) and of the trained neuron (solid), the accuracy of both and the curve of the loss.

**What to observe.** One neuron separates the classes by a straight line: with two inputs it recognises the mines with an accuracy of about 0.8, the overlapping region remains. A too large $\eta$ makes the loss oscillate, a too small one requires many steps. The activation function does not move the boundary $z = 0$; it changes the output value and its gradient.

**Control question 7.** How many weights has one neuron with the full HOG vector of 1764 values at the input, and how does its boundary between the classes look in this space?

In [ ]:
#@title 4.2.1 Neuron with two inputs on SULAND fragments { display-mode: "form" }
W1 = 1.0 #@param {type:"slider", min:-3, max:3, step:0.1}
W2 = -1.0 #@param {type:"slider", min:-3, max:3, step:0.1}
B = 0.0 #@param {type:"slider", min:-3, max:3, step:0.1}
ACTIVATION = "sigmoid" #@param ["sigmoid", "tanh", "ReLU", "step"]
TRAIN = True #@param {type:"boolean"}
LEARNING_RATE = 0.5 #@param [0.01, 0.1, 0.5, 2.0, 8.0] {type:"raw"}
EPOCHS = 200 #@param {type:"slider", min:10, max:1000, step:10}

from sklearn.decomposition import PCA
CROPS = globals().get("CROPS", {})
if not CROPS:
    seq_tab = suland.sequences()
    pool = seq_tab[(seq_tab.domain == "data-iid") & (seq_tab.pfm1 > 40) & (seq_tab.pma2 > 40)].sort_values("boxes", ascending=False).seq.tolist()
    CROPS[(12, 30)] = suland.crop_set(pool[:12], 30, size=64, seed=0)
crops_n = CROPS[(12, 30)] if (12, 30) in CROPS else list(CROPS.values())[0]
Xh = aitbc.hog_features(list(crops_n.img), size=64)
Z2 = PCA(2, random_state=0).fit_transform(Xh)
Z2 = (Z2 - Z2.mean(0)) / Z2.std(0)
t = (crops_n.label != "background").astype(int).values
act = {"sigmoid": lambda z: 1 / (1 + np.exp(-z)), "tanh": np.tanh, "ReLU": lambda z: np.maximum(z, 0), "step": lambda z: (z > 0).astype(float)}[ACTIVATION]
acc_hand = float(((Z2 @ np.array([W1, W2]) + B > 0).astype(int) == t).mean())
w, b, losses = np.zeros(2), 0.0, []
if TRAIN:
    for _ in range(EPOCHS):
        yv = 1 / (1 + np.exp(-(Z2 @ w + b)))
        losses.append(float(-np.mean(t * np.log(yv + 1e-9) + (1 - t) * np.log(1 - yv + 1e-9))))
        w -= LEARNING_RATE * Z2.T @ (yv - t) / len(t)
        b -= LEARNING_RATE * float(np.mean(yv - t))
acc_tr = float(((Z2 @ w + b > 0).astype(int) == t).mean()) if TRAIN else float("nan")
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6), gridspec_kw={"width_ratios": [1.1, 1, 1]})
aitbc.draw_neuron(ax[0], inputs=("x₁", "x₂"), weights=(f"w₁={W1:g}", f"w₂={W2:g}"), bias=f"b={B:g}", activation=ACTIVATION)
xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
if TRAIN:
    ax[1].contourf(xx, yy, 1 / (1 + np.exp(-(w[0] * xx + w[1] * yy + b))), levels=20, cmap="RdBu_r", alpha=0.25)
for lab, c, mk in (("background", "#1565c0", "o"), ("PFM-1", "#c62828", "^"), ("PMA-2", "#ef6c00", "s")):
    m_ = (crops_n.label == lab).values
    ax[1].scatter(Z2[m_, 0], Z2[m_, 1], s=9, c=c, marker=mk, label=lab, alpha=0.6)
xs = np.linspace(-3, 3, 2)
if abs(W2) > 1e-9:
    ax[1].plot(xs, -(W1 * xs + B) / W2, "k--", lw=1.4, label=f"by hand: accuracy {acc_hand:.3f}")
if TRAIN and abs(w[1]) > 1e-9:
    ax[1].plot(xs, -(w[0] * xs + b) / w[1], "k-", lw=2, label=f"trained: accuracy {acc_tr:.3f}")
ax[1].set_xlim(-3, 3)
ax[1].set_ylim(-3, 3)
ax[1].set_xlabel("x₁ (1st principal component of HOG)")
ax[1].set_ylabel("x₂ (2nd principal component)")
ax[1].legend(fontsize=7.5, loc="lower right")
ax[2].plot(losses)
ax[2].set_xlabel("step of gradient descent")
ax[2].set_ylabel("cross-entropy L")
ax[2].set_title(f"η = {LEARNING_RATE:g}: w₁ = {w[0]:.2f}, w₂ = {w[1]:.2f}, b = {b:.2f}" if TRAIN else "training switched off", fontsize=9)
ax[2].grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson42_neuron.png"), dpi=80)
plt.show()
print(f"{len(t)} fragments ({int(t.sum())} with a mine). Accuracy of the neuron set by hand: {acc_hand:.3f}"
      + (f"; trained: {acc_tr:.3f}, loss {losses[0]:.3f} → {losses[-1]:.3f}" if TRAIN else ""))

### Block 4.2.2. The fully connected network

**Purpose.** To train a fully connected (multilayer) network on the HOG vectors of military vehicles and to compare it with the SVM of block 4.1.5.

**Formulas.** Layer $l$: $h^{(l)} = \varphi\left(W^{(l)} h^{(l-1)} + b^{(l)}\right)$, $\varphi$ — ReLU; output layer: $p_k = e^{z_k} / \sum_j e^{z_j}$ (softmax, the probabilities of the $K$ classes). Number of parameters of a layer with $n_{in}$ inputs and $n_{out}$ neurons: $(n_{in} + 1)\, n_{out}$. The weights are trained by the Adam method on the cross-entropy $-\ln p_{\text{true class}}$, by mini-batches of 64 vectors; one epoch is one pass over the training part. Dropout switches off a random share of the neurons in every step of the training and reduces the fitting to the training examples.

**Parameters.** `LEVEL_42`, `N_PER_CLASS_42` — as in block 4.1.5; `HIDDEN` — the numbers of neurons of the hidden layers; `EPOCHS`, `LEARNING_RATE_42`, `DROPOUT`.

**Expected result.** The graph of the network with the sizes of its layers, the number of parameters by the formula and in the model, the curves of the accuracy on the training and the test part and the final metrics.

**What to observe.** The network 1764 → 256 → 64 → 7 has 468 743 parameters for 3569 training images: the accuracy on the training part reaches 0.999 after about 15 epochs, the test accuracy stays at about 0.37, slightly below the SVM of block 4.1.5 (0.39 on the 7 groups). The gap between the two curves is overfitting; dropout and fewer neurons reduce it. With HOG at the input the network cannot obtain more than the features contain.

**Control question 8.** Compute the number of parameters of the network 1764 → 128 → 7 and compare it with the number of training images.

In [ ]:
#@title 4.2.2 Fully connected network on HOG vectors { display-mode: "form" }
LEVEL_42 = "7 groups of the course" #@param ["7 groups of the course", "24 types"]
N_PER_CLASS_42 = 150 #@param {type:"slider", min:25, max:400, step:25}
HIDDEN = "256-64" #@param ["64", "128", "256-64", "512-128"]
EPOCHS_42 = 30 #@param {type:"slider", min:5, max:80, step:5}
LEARNING_RATE_42 = 0.001 #@param [0.0003, 0.001, 0.003, 0.01] {type:"raw"}
DROPOUT = 0.3 #@param {type:"slider", min:0, max:0.6, step:0.1}

import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
TRAIN_IDX, TEST_IDX = mv.clean(near=2)          # without exact copies and near duplicates (block 4.1.2)
if "HOG_ALL" not in globals():
    HOG_ALL = aitbc.hog_features([mv.image(i, size=64, gray=True) for i in range(len(mv.table))], size=64)
lab_col = "cls" if LEVEL_42 == "24 types" else "group"
labels = sorted(mv.table.cls.unique()) if lab_col == "cls" else list(aitbc.MilitaryVehicles.GROUP_ORDER)
tr = mv.subset(TRAIN_IDX, N_PER_CLASS_42, seed=0)
mu, sd = HOG_ALL[tr].mean(0), HOG_ALL[tr].std(0) + 1e-6
Xtr = torch.tensor((HOG_ALL[tr] - mu) / sd)
Xte = torch.tensor((HOG_ALL[TEST_IDX] - mu) / sd)
ytr = torch.tensor([labels.index(v) for v in mv.table[lab_col].values[tr]])
yte = np.array([labels.index(v) for v in mv.table[lab_col].values[TEST_IDX]])
sizes = [Xtr.shape[1]] + [int(v) for v in HIDDEN.split("-")] + [len(labels)]
layers = []
for a, b_ in zip(sizes[:-2], sizes[1:-1]):
    layers += [nn.Linear(a, b_), nn.ReLU(), nn.Dropout(DROPOUT)]
layers.append(nn.Linear(sizes[-2], sizes[-1]))
torch.manual_seed(0)
net = nn.Sequential(*layers)
opt = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE_42)
lossf = nn.CrossEntropyLoss()
hist = []
t0 = time.time()
for ep in range(EPOCHS_42):
    net.train()
    perm = torch.randperm(len(Xtr))
    for s in range(0, len(Xtr), 64):
        bi = perm[s:s + 64]
        opt.zero_grad()
        loss = lossf(net(Xtr[bi]), ytr[bi])
        loss.backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        acc_tr_ = (net(Xtr).argmax(1) == ytr).float().mean().item()
        p_te = net(Xte).argmax(1).numpy()
    hist.append((acc_tr_, accuracy_score(yte, p_te)))
t_train = time.time() - t0
n_formula = sum((a + 1) * b_ for a, b_ in zip(sizes[:-1], sizes[1:]))
n_model = aitbc.count_parameters(net)[0]
fig, ax = plt.subplots(1, 2, figsize=(17, 4.4), gridspec_kw={"width_ratios": [1.3, 1]})
names = ["HOG vector"] + [f"hidden {k + 1}, ReLU" for k in range(len(sizes) - 2)] + ["output, softmax"]
aitbc.draw_mlp(ax[0], sizes, names=names, output_names=labels if len(labels) <= 7 else None)
hist_a = np.array(hist)
ax[1].plot(range(1, EPOCHS_42 + 1), hist_a[:, 0], label="training part")
ax[1].plot(range(1, EPOCHS_42 + 1), hist_a[:, 1], label="test part")
ax[1].set_xlabel("epoch")
ax[1].set_ylabel("accuracy")
ax[1].set_ylim(0, 1.02)
ax[1].grid(alpha=0.3)
ax[1].legend()
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson42_fully_connected.png"), dpi=80)
plt.show()
f1m = f1_score(yte, p_te, average="macro")
rows = {"layers": " → ".join(map(str, sizes)), "parameters by the formula Σ(n_in + 1)·n_out": n_formula,
        "parameters in the model": n_model, "training images": len(tr),
        "accuracy: training / test part": f"{hist_a[-1, 0]:.3f} / {hist_a[-1, 1]:.3f}", "F1 (macro), test": f"{f1m:.3f}",
        "time of training": f"{t_train:.1f} s"}
ref = {k: v for k, v in globals().get("RESULTS_41", {}).items() if k[0] == LEVEL_42 and k[2] == N_PER_CLASS_42}
for k, v in ref.items():
    rows[f"HOG + {k[1]} (block 4.1.5): accuracy / F1"] = f"{v['accuracy']:.3f} / {v['f1_macro']:.3f}"
display(pd.DataFrame({"value": list(rows.values())}, index=list(rows)))
MLP_RESULT = dict(level=LEVEL_42, layers=sizes, accuracy=round(float(hist_a[-1, 1]), 3), f1_macro=round(float(f1m), 3))

### Block 4.2.3. The convolutional network: convolution and feature maps

**Purpose.** To see how a convolutional network trained on ImageNet (MobileNetV3-small, 1.2 million images of 1000 classes) transforms an image of a military vehicle layer by layer.

**Formulas.**
- Convolution: the value of the output map $c$ in the point $(x, y)$ is $a_c(x, y) = \varphi\left(\sum_{c'} \sum_{i, j} K_{c, c'}(i, j)\, I_{c'}(x + i, y + j) + b_c\right)$: the same kernel $K$ of $k \times k$ weights is applied in every point. A layer with $C_{in}$ input and $C_{out}$ output maps has $k^2 C_{in} C_{out}$ weights, independent of the size of the image.
- Size of the output map: $W_{out} = \lfloor (W_{in} + 2P - k) / S \rfloor + 1$, $P$ — padding, $S$ — stride; MobileNetV3-small reduces 224 × 224 to 7 × 7 by five layers with $S = 2$.
- Global average pooling turns the last maps (7 × 7 × 576) into a vector of 576 values: the features used in block 4.2.5.

**Parameters.** `TYPE` — the type of the vehicle; `IMAGE_NUMBER`; `LAYER` — the number of the block of the network whose output maps are shown (0 — the first convolution, 12 — the last one); `N_MAPS` — the number of maps.

**Expected result.** The 16 kernels of the first layer, the output maps of the chosen block, the table of the blocks (size of the output, number of parameters), the diagram of the network and the five classes of ImageNet that the network gives for the image.

**What to observe.** The kernels of the first layer are detectors of edges and colour transitions, as the Sobel operator of Topic 3. The maps of the first blocks repeat the contours of the vehicle; the maps of the last blocks have 7 × 7 cells and respond to parts of objects. Among the 1000 classes of ImageNet there are "tank", "half track", "amphibious vehicle", "cannon", "missile", "projectile": the features of ImageNet are close to the task of recognition of military vehicles.

**Control question 9.** What is the size of the output map of a convolution 3 × 3 with stride 2 and padding 1 for an input of 224 × 224, and how many weights has such a layer with 3 input and 16 output maps?

In [ ]:
#@title 4.2.3 Kernels and feature maps of MobileNetV3-small { display-mode: "form" }
TYPE = "T-72" #@param ["2S19_MSTA", "30N6E", "BM-30", "BMD", "BMP-1", "BMP-2", "BMP-T15", "BRDM", "BTR-60", "BTR-70", "BTR-80", "D-30", "Iskander", "MT_LB", "Pantsir-S1", "Rs-24", "T-14", "T-62", "T-64", "T-72", "T-80", "T-90", "TOS-1", "Tornado"]
IMAGE_NUMBER = 3 #@param {type:"slider", min:0, max:20, step:1}
LAYER = 1 #@param {type:"slider", min:0, max:12, step:1}
N_MAPS = 16 #@param [8, 16, 32] {type:"raw"}

from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
if "MNV3S" not in globals():
    MNV3S_W = MobileNet_V3_Small_Weights.IMAGENET1K_V1
    MNV3S = mobilenet_v3_small(weights=MNV3S_W).eval()
i_img = mv.table.index[(mv.table.cls == TYPE) & (mv.table.split == "train")][IMAGE_NUMBER]
rgb = mv.image(i_img)
x = torch.from_numpy(aitbc.imagenet_batch([rgb], 224))
outs, shapes = {}, []
hooks = [blk.register_forward_hook(lambda m, i, o, k=k: outs.__setitem__(k, o.detach())) for k, blk in enumerate(MNV3S.features)]
with torch.no_grad():
    logits = MNV3S(x)
for h in hooks:
    h.remove()
for k, blk in enumerate(MNV3S.features):
    o = outs[k]
    shapes.append({"block": k, "type": type(blk).__name__, "output H × W × C": f"{o.shape[2]} × {o.shape[3]} × {o.shape[1]}",
                   "parameters": aitbc.count_parameters(blk)[0]})
fig = plt.figure(figsize=(18, 5.2))
ax0 = fig.add_subplot(1, 3, 1)
ax0.imshow(cv2.resize(rgb, (224, 224), interpolation=cv2.INTER_AREA))
ax0.set_title(f"{TYPE}: image {i_img}, input 224 × 224 × 3", fontsize=9)
ax0.axis("off")
k1 = MNV3S.features[0][0].weight.detach().numpy()          # 16 x 3 x 3 x 3
k1 = (k1 - k1.min((1, 2, 3), keepdims=True)) / (np.ptp(k1, axis=(1, 2, 3), keepdims=True) + 1e-9)   # every kernel to 0...1
tiles = np.vstack([np.hstack([np.pad(cv2.resize(k1[r * 8 + c].transpose(1, 2, 0), (24, 24), interpolation=cv2.INTER_NEAREST),
                                     ((2, 2), (2, 2), (0, 0)), constant_values=1) for c in range(8)]) for r in range(2)])
ax1 = fig.add_subplot(1, 3, 2)
ax1.imshow(tiles)
ax1.set_title("16 kernels 3 × 3 × 3 of the first convolution (as colour images)", fontsize=9)
ax1.axis("off")
o = outs[LAYER][0].numpy()
order = np.argsort(-o.reshape(o.shape[0], -1).mean(1))[:N_MAPS]
ncol = 8
grid = np.vstack([np.hstack([np.pad(cv2.resize((o[c] - o[c].min()) / (np.ptp(o[c]) + 1e-6), (64, 64), interpolation=cv2.INTER_NEAREST),
                                    2, constant_values=np.nan) for c in order[r * ncol:(r + 1) * ncol]]) for r in range(max(1, len(order) // ncol))])
ax2 = fig.add_subplot(1, 3, 3)
cmap_maps = plt.get_cmap("viridis").copy()
cmap_maps.set_bad("white")
ax2.imshow(np.ma.masked_invalid(grid), cmap=cmap_maps)
ax2.set_title(f"block {LAYER}: {N_MAPS} of {o.shape[0]} maps {o.shape[1]} × {o.shape[2]} with the largest mean", fontsize=9)
ax2.axis("off")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"lesson42_feature_maps_block{LAYER}.png"), dpi=80)
plt.show()
display(pd.DataFrame(shapes).set_index("block").T)
stages = [("input", 224, 224, 3)] + [(f"block {k}", outs[k].shape[2], outs[k].shape[3], outs[k].shape[1]) for k in (0, 1, 3, 6, 8, 11, 12)] \
    + [("global average pooling", 1, 1, 576), ("classifier ImageNet", 1, 1, 1000)]
fig, ax = plt.subplots(figsize=(18, 3.2))
aitbc.draw_cnn(ax, stages, title=f"MobileNetV3-small: {aitbc.count_parameters(MNV3S)[0]:,} parameters".replace(",", " "))
plt.show()
prob = torch.softmax(logits, 1)[0].numpy()
top = np.argsort(-prob)[:5]
print("Five classes of ImageNet for the image:", "; ".join(f"{MNV3S_W.meta['categories'][j]} {prob[j]:.2f}" for j in top))

### Block 4.2.4. MobileNet: depthwise separable convolution and the cost of computation

**Purpose.** To compare the number of parameters, the number of operations and the time of inference of MobileNet with other convolutional networks, and to explain the source of the difference.

**Formulas.** A standard convolution $k \times k$ of $C_{in}$ into $C_{out}$ maps of size $H \times W$ needs $k^2 C_{in} C_{out}$ weights and $k^2 C_{in} C_{out} H W$ multiply-accumulate operations (MAC). The **depthwise separable** convolution replaces it by a depthwise convolution $k \times k$ (one kernel per map: $k^2 C_{in}$ weights) and a pointwise convolution $1 \times 1$ ($C_{in} C_{out}$ weights):
$$\frac{k^2 C_{in} + C_{in} C_{out}}{k^2 C_{in} C_{out}} = \frac{1}{C_{out}} + \frac{1}{k^2} \approx \frac{1}{9}\ \text{for}\ k = 3.$$
The **inverted residual block** of MobileNet V2/V3 expands the maps by a convolution $1 \times 1$, filters them by a depthwise convolution and projects them back by a convolution $1 \times 1$; when the input and the output have the same size, the input is added to the output (residual connection).

**Parameters.** `INPUT_SIZE` — the size of the input image; `BATCH` — the number of images processed together; `N_RUNS` — the number of repetitions of the measurement of the time.

**Expected result.** The diagram of a real block of MobileNetV3-small with the numbers of weights, the table of five networks (parameters, size of the file of the weights, MAC, time per image on the device of the session) and the dependence of the time on the size of the input.

**What to observe.** MobileNetV3-small has 2.5 million parameters and 56 million MAC per image of 224 × 224, ResNet-50 25.6 million parameters and 4.1 billion MAC: 72 times fewer operations at a lower, but comparable accuracy on ImageNet (67.7 % against 80.9 %). The time does not fall in proportion to the MAC (on a processor of 16 cores 8 ms against 49 ms, 6 times): the depthwise convolutions use the processor less efficiently than the standard ones. The number of operations grows with the square of the size of the input.

**Control question 10.** By how many times does a depthwise separable convolution 3 × 3 reduce the number of weights of a layer with 96 input and 96 output maps?

In [ ]:
#@title 4.2.4 Parameters, operations and time of inference of five networks { display-mode: "form" }
INPUT_SIZE = 224 #@param [128, 160, 224, 320] {type:"raw"}
BATCH = 1 #@param [1, 8, 32] {type:"raw"}
N_RUNS = 5 #@param {type:"slider", min:2, max:20, step:1}

import torch.nn as nn
from torchvision import models as tvm


def count_macs(model, size):
    # multiply-accumulate operations of the convolution and linear layers for one image size x size
    macs = []

    def hook(m, i, o):
        if isinstance(m, nn.Conv2d):
            macs.append(o.numel() // o.shape[0] * (m.in_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1])
        elif isinstance(m, nn.Linear):
            macs.append(m.in_features * m.out_features)
    hs = [m.register_forward_hook(hook) for m in model.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
    with torch.no_grad():
        model(torch.zeros(1, 3, size, size))
    for h in hs:
        h.remove()
    return sum(macs)


def time_per_image(model, size, batch, runs):
    x = torch.randn(batch, 3, size, size, device=DEVICE)
    model = model.to(DEVICE).eval()
    with torch.no_grad():
        model(x)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(runs):
            model(x)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
    return (time.time() - t0) / runs / batch * 1000


NETS = {"MobileNetV3-small": tvm.mobilenet_v3_small, "MobileNetV3-large": tvm.mobilenet_v3_large,
        "MobileNetV2": tvm.mobilenet_v2, "ResNet-18": tvm.resnet18, "ResNet-50": tvm.resnet50}
TOP1 = {"MobileNetV3-small": 67.7, "MobileNetV3-large": 75.3, "MobileNetV2": 72.2, "ResNet-18": 69.8, "ResNet-50": 80.9}
rows = []
for name, ctor in NETS.items():
    m = ctor(weights=None).eval()          # architecture only: the weights are not needed to count and to time
    n = aitbc.count_parameters(m)[0]
    rows.append({"network": name, "parameters, million": n / 1e6, "weights float32, MB": n * 4 / 2 ** 20,
                 f"MAC per image {INPUT_SIZE}², million": count_macs(m, INPUT_SIZE) / 1e6,
                 f"time per image, ms ({DEVICE}, batch {BATCH})": time_per_image(m, INPUT_SIZE, BATCH, N_RUNS),
                 "top-1 ImageNet, % (torchvision)": TOP1[name]})
COST = pd.DataFrame(rows).set_index("network")
display(COST.style.format(precision=1).set_caption(f"Input {INPUT_SIZE} × {INPUT_SIZE}; {os.cpu_count()} logical cores" if DEVICE == "cpu" else f"Input {INPUT_SIZE} × {INPUT_SIZE}; GPU"))
small = tvm.mobilenet_v3_small(weights=None)
blk = small.features[3].block                     # inverted residual block 24 -> 88 -> 24 of MobileNetV3-small
c_in, c_exp, c_out = blk[0][0].in_channels, blk[0][0].out_channels, blk[2][0].out_channels
k_dw = blk[1][0].kernel_size[0]
fig, ax = plt.subplots(1, 2, figsize=(18, 3.8), gridspec_kw={"width_ratios": [1.6, 1]})
aitbc.draw_mobilenet_block(ax[0], c_in, c_exp, c_out, k_dw, se=False, stride=blk[1][0].stride[0])
ax[0].set_title(f"block 3 of MobileNetV3-small: {aitbc.count_parameters(small.features[3])[0]:,} parameters (with batch normalisation)".replace(",", " "), fontsize=9)
sizes_ = [96, 128, 160, 224, 320]
ax[1].plot(sizes_, [count_macs(small, s) / 1e6 for s in sizes_], "o-")
ax[1].set_xlabel("size of the input, px")
ax[1].set_ylabel("MAC per image, million")
ax[1].set_title("MobileNetV3-small: operations and the size of the input", fontsize=9)
ax[1].grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson42_mobilenet_cost.png"), dpi=80)
plt.show()
def spaced(n):
    return f"{n:,}".replace(",", " ")


for C_ in (16, 96, 576):
    print(f"3×3 convolution {C_} → {C_} maps: standard {spaced(aitbc.conv_params(3, C_, C_))} weights; depthwise separable "
          f"{spaced(aitbc.conv_params(3, C_, C_, True))} (ratio {aitbc.conv_params(3, C_, C_) / aitbc.conv_params(3, C_, C_, True):.1f})")
COST.to_csv(os.path.join(REPORT_DIR, "lesson42_cost_of_networks.csv"))

### Block 4.2.5. Transfer learning: features of a network trained on ImageNet

**Purpose.** To recognise military vehicles with the features of MobileNetV3 trained on ImageNet and to compare the result with HOG + SVM on the same training images.

**Method.** The convolutional part of the network with the weights of ImageNet is **frozen** (its weights are not changed): every image (224 × 224, normalised by the mean and the standard deviation of ImageNet) is turned into a vector of features of 576 (small) or 960 (large) values. On these vectors a single linear layer with softmax is trained (multinomial logistic regression, `C` — the inverse strength of the regularisation). This is the first stage of transfer learning; in Notebook 07 (Lesson 4.4) the last blocks of the network are also trained (fine-tuning).

**Parameters.** `LEVEL_45`, `N_PER_CLASS_45` — as in block 4.1.5; `BACKBONE`; `INPUT_SIZE_45`; `C_LOGISTIC`; `LEARNING_CURVE` — repeat the training for 25…400 images per type (the features of all 7751 training images are computed: 2–6 minutes on the processor).

**Expected result.** The table *HOG + SVM — CNN features + linear layer* on the same images, the metrics of the classes, the confusion matrix of the groups and, optionally, the learning curve.

**What to observe.** With 150 images per type the features of MobileNetV3-small give an accuracy of 0.42 on the 24 types and 0.63 on the 7 groups, against 0.18 and 0.39 for HOG + SVM on the same images; with all training images 0.45 and 0.67. The large network gives practically the same result: the limit is set by the data (small images, contradictory labels, renderings), not by the size of the network. For comparison, the authors of the data set report 54 % on the 24 types for the network ViT-B/16.

**Control question 11.** Why do the features of ImageNet, where there is no class "BTR-80", help to recognise a BTR-80, and in which conditions does this help stop?

In [ ]:
#@title 4.2.5 Frozen MobileNetV3 + trained linear layer { display-mode: "form" }
LEVEL_45 = "24 types" #@param ["24 types", "7 groups of the course"]
N_PER_CLASS_45 = 150 #@param {type:"slider", min:25, max:400, step:25}
BACKBONE = "MobileNetV3-small" #@param ["MobileNetV3-small", "MobileNetV3-large"]
INPUT_SIZE_45 = 224 #@param [128, 160, 224] {type:"raw"}
C_LOGISTIC = 0.05 #@param [0.01, 0.05, 0.2, 1.0] {type:"raw"}
LEARNING_CURVE = False #@param {type:"boolean"}

from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights, mobilenet_v3_large, MobileNet_V3_Large_Weights
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

BACKBONES = globals().get("BACKBONES", {})
if BACKBONE not in BACKBONES:
    BACKBONES[BACKBONE] = (mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1) if BACKBONE.endswith("small")
                           else mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)).eval()
backbone = BACKBONES[BACKBONE]
FEATS = globals().get("FEATS", {})
fkey = (BACKBONE, INPUT_SIZE_45)
FEATS.setdefault(fkey, {})


def features_of(idx):
    # CNN features of the images idx (computed once, kept in FEATS)
    need = [i for i in idx if i not in FEATS[fkey]]
    if need:
        t0 = time.time()
        f = aitbc.cnn_features(backbone, [mv.image(i) for i in need], size=INPUT_SIZE_45, batch=64, device=DEVICE)
        FEATS[fkey].update(zip(need, f))
        print(f"features of {len(need)} images ({f.shape[1]} values): {time.time() - t0:.0f} s on {DEVICE}")
    return np.stack([FEATS[fkey][i] for i in idx])


TRAIN_IDX, TEST_IDX = mv.clean(near=2)          # without exact copies and near duplicates (block 4.1.2)
lab_col = "cls" if LEVEL_45 == "24 types" else "group"
labels = sorted(mv.table.cls.unique()) if lab_col == "cls" else list(aitbc.MilitaryVehicles.GROUP_ORDER)
tr = mv.subset(TRAIN_IDX, N_PER_CLASS_45, seed=0)
F_tr, F_te = features_of(tr), features_of(TEST_IDX)
y_tr, y_te = mv.table[lab_col].values[tr], mv.table[lab_col].values[TEST_IDX]
t0 = time.time()
HEAD = make_pipeline(StandardScaler(), LogisticRegression(C=C_LOGISTIC, max_iter=3000)).fit(F_tr, y_tr)
t_fit = time.time() - t0
pred = HEAD.predict(F_te)
P, R, F1, S = precision_recall_fscore_support(y_te, pred, labels=labels, zero_division=0)
acc = accuracy_score(y_te, pred)
if "HOG_ALL" not in globals():
    HOG_ALL = aitbc.hog_features([mv.image(i, size=64, gray=True) for i in range(len(mv.table))], size=64)
svm = make_pipeline(StandardScaler(), SVC(C=10, gamma="scale")).fit(HOG_ALL[tr], y_tr)
p_svm = svm.predict(HOG_ALL[TEST_IDX])
_, _, F1s, _ = precision_recall_fscore_support(y_te, p_svm, labels=labels, zero_division=0)
cmpt = pd.DataFrame({"features": ["HOG 64 × 64 (1764 values)", f"{BACKBONE}, ImageNet, frozen ({F_tr.shape[1]} values)"],
                     "classifier": ["RBF SVM, C = 10", f"linear layer + softmax, C = {C_LOGISTIC}"],
                     "accuracy": [accuracy_score(y_te, p_svm), acc], "F1 (macro)": [F1s.mean(), F1.mean()]}).set_index("features")
display(cmpt.style.format(precision=3).set_caption(f"{LEVEL_45}: {len(tr)} training images ({N_PER_CLASS_45} per type at most), "
                                                    f"{len(TEST_IDX)} test images; training of the linear layer {t_fit:.1f} s"))
per_class = pd.DataFrame({"precision": P, "recall": R, "F1": F1, "test images": S}, index=labels).sort_values("F1")
if len(per_class) > 8:
    display(pd.concat([per_class.head(4), per_class.tail(4)]).style.format(precision=3).set_caption("4 worst and 4 best classes"))
else:
    display(per_class.style.format(precision=3).set_caption("Metrics of every class"))
g_true = pd.Series(y_te).map(aitbc.MilitaryVehicles.GROUPS).values if lab_col == "cls" else y_te
g_pred = pd.Series(pred).map(aitbc.MilitaryVehicles.GROUPS).values if lab_col == "cls" else pred
G = list(aitbc.MilitaryVehicles.GROUP_ORDER)
cm = confusion_matrix(g_true, g_pred, labels=G)
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(1, 2 if LEARNING_CURVE else 1, figsize=(15 if LEARNING_CURVE else 7.5, 5.6), squeeze=False)
ax = ax[0]
ax[0].imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(len(G)):
    for j in range(len(G)):
        ax[0].text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=8, color="w" if cmn[i, j] > 0.5 else "k")
ax[0].set_xticks(range(len(G)))
ax[0].set_xticklabels(G, rotation=60, ha="right", fontsize=8)
ax[0].set_yticks(range(len(G)))
ax[0].set_yticklabels(G, fontsize=8)
ax[0].set_title(f"groups ({'from the recognised types' if lab_col == 'cls' else 'recognised directly'}): accuracy {np.mean(g_true == g_pred):.3f}", fontsize=9)
if LEARNING_CURVE:
    curve = []
    F_all = features_of(TRAIN_IDX)
    pos = {i: k for k, i in enumerate(TRAIN_IDX)}
    for n in (25, 50, 100, 150, 250, 400):
        s_ = mv.subset(TRAIN_IDX, n, seed=0)
        h_ = make_pipeline(StandardScaler(), LogisticRegression(C=C_LOGISTIC, max_iter=3000)).fit(F_all[[pos[i] for i in s_]], mv.table[lab_col].values[s_])
        curve.append((len(s_), accuracy_score(y_te, h_.predict(F_te))))
    ax[1].plot(*zip(*curve), "o-")
    ax[1].set_xlabel("training images")
    ax[1].set_ylabel("test accuracy")
    ax[1].set_title(f"learning curve, {LEVEL_45}", fontsize=9)
    ax[1].grid(alpha=0.3)
    print("Learning curve (training images, accuracy):", [(n, round(a, 3)) for n, a in curve])
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson42_transfer_learning.png"), dpi=80)
plt.show()
HEAD_INFO = dict(level=LEVEL_45, backbone=BACKBONE, input_size=INPUT_SIZE_45, labels=list(HEAD.classes_),
                 accuracy=round(acc, 3), f1_macro=round(float(F1.mean()), 3), n_train=len(tr))

### Block 4.2.6. Export of the model to ONNX and check of the inference

**Purpose.** To join the frozen network and the trained linear layer of block 4.2.5 into one model, to save it in the ONNX format and to check that the inference programs of the onboard computer (ONNX Runtime, OpenCV DNN) give the same result as PyTorch.

**ONNX** (Open Neural Network Exchange) describes a model as a graph of standard operations (Conv, HardSwish, GlobalAveragePool, Gemm, Softmax…) with the weights; the file is executed without PyTorch by ONNX Runtime, OpenCV DNN, TensorRT and other programs on processors, GPUs and NPUs of onboard computers. The *opset* is the version of the set of operations: the program of inference must support the opset of the file.

**Model.** `image` (N × 3 × S × S, float32, normalised as for ImageNet) → convolutional part → global average pooling → linear layer → softmax → `probabilities` (N × K). The standardisation of the features of block 4.2.5 is included in the weights of the linear layer: $z = W\,\frac{f - \mu}{\sigma} + b = \frac{W}{\sigma} f + \left(b - W \frac{\mu}{\sigma}\right)$.

**Parameters.**
- `EXPORTER` — the classical exporter of PyTorch (tracing of the model, `dynamo=False`) or the new one (`dynamo=True`, opset not lower than 18; by default it writes the weights into a separate file `.onnx.data`, here `external_data=False` keeps the model in one file).
- `OPSET` — the version of the operations; `DYNAMIC_BATCH` — the number of images in the input is free; `N_CHECK` — the number of test images for the check.

**Expected result.** The file of the model and the file of its labels in `models` of the project, the check by the ONNX checker, the largest difference of the probabilities of PyTorch, ONNX Runtime and OpenCV DNN, the agreement of the recognised classes and the time per image.

**What to observe.** With the classical exporter (opset 17) the differences of the probabilities of ONNX Runtime and OpenCV DNN are of the order of $10^{-6}$ (rounding of float32), and the classes agree for all images. The file of the new exporter (opset 18) is executed correctly by ONNX Runtime, but in the check of the course OpenCV 4.12 (the version of Colab) did not read it, and OpenCV 5.0 read it and gave wrong probabilities (the classes agreed for 1–2 of 64 images) without any error message. For this reason a model is accepted for the onboard computer only after the check of its file by the program of inference of that computer. The file of MobileNetV3-small with the linear layer is about 3.6 MB. The preprocessing (size, mean, standard deviation, order of the channels RGB) is not a part of the file: it is written in the file of the labels and must be repeated exactly on the onboard computer.

**Control question 12.** Which information must accompany the ONNX file so that the onboard computer gives the same classes as the model in the notebook?

In [ ]:
#@title 4.2.6 Export of the recogniser to ONNX, check with ONNX Runtime and OpenCV DNN { display-mode: "form" }
EXPORTER = "classical (dynamo=False)" #@param ["classical (dynamo=False)", "new (dynamo=True)"]
OPSET = 17 #@param [13, 17, 18] {type:"raw"}
DYNAMIC_BATCH = True #@param {type:"boolean"}
N_CHECK = 64 #@param {type:"slider", min:16, max:256, step:16}

import importlib.util, subprocess
need = [p for p, mod in (("onnx", "onnx"), ("onnxruntime", "onnxruntime"), ("onnxscript", "onnxscript")) if importlib.util.find_spec(mod) is None]
if need:
    print("Installing:", ", ".join(need))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + need, check=True)
import onnx
import onnxruntime as ort
if "HEAD" not in globals():
    raise RuntimeError("Run block 4.2.5 first: it trains the linear layer of the recogniser.")
scaler, logreg = HEAD[0], HEAD[1]
backbone = BACKBONES[HEAD_INFO["backbone"]]
S_IN = HEAD_INFO["input_size"]


class Recogniser(torch.nn.Module):
    def __init__(self, net, scaler, logreg):
        super().__init__()
        self.features, self.pool = net.features, net.avgpool
        Wt = logreg.coef_ / scaler.scale_[None, :]
        bt = logreg.intercept_ - (logreg.coef_ * scaler.mean_ / scaler.scale_).sum(1)
        self.head = torch.nn.Linear(Wt.shape[1], Wt.shape[0])
        with torch.no_grad():
            self.head.weight.copy_(torch.tensor(Wt, dtype=torch.float32))
            self.head.bias.copy_(torch.tensor(bt, dtype=torch.float32))

    def forward(self, image):
        return torch.softmax(self.head(self.pool(self.features(image)).flatten(1)), dim=1)


model = Recogniser(backbone, scaler, logreg).eval()
tag = f"military_vehicles_{'types' if HEAD_INFO['level'] == '24 types' else 'groups'}_{HEAD_INFO['backbone'].replace('-', '').lower()}_{S_IN}"
onnx_path = os.path.join(MODEL_DIR, tag + ".onnx")
dummy = torch.from_numpy(aitbc.imagenet_batch([mv.image(TEST_IDX[0])], S_IN))
kw = dict(input_names=["image"], output_names=["probabilities"], opset_version=OPSET)
if DYNAMIC_BATCH:
    kw["dynamic_axes"] = {"image": {0: "batch"}, "probabilities": {0: "batch"}}
import warnings
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")        # the classical exporter is marked as deprecated since PyTorch 2.9
    if EXPORTER.startswith("new"):
        kw["external_data"] = False        # the new exporter otherwise writes the weights into a separate file .onnx.data
    torch.onnx.export(model, (dummy,), onnx_path, dynamo=EXPORTER.startswith("new"), **kw)
t_exp = time.time() - t0
m_onnx = onnx.load(onnx_path)
onnx.checker.check_model(m_onnx)
ops = pd.Series([n.op_type for n in m_onnx.graph.node]).value_counts()
opset_file = max(o.version for o in m_onnx.opset_import if o.domain in ("", "ai.onnx"))
labels_json = dict(labels=HEAD_INFO["labels"], input="image: N x 3 x S x S float32, RGB", size=S_IN,
                   mean=aitbc.IMAGENET_MEAN.tolist(), std=aitbc.IMAGENET_STD.tolist(), resize="cv2.INTER_AREA to S x S",
                   output="probabilities: N x K (softmax)", backbone=HEAD_INFO["backbone"] + ", ImageNet weights, frozen",
                   head="linear layer trained on " + str(HEAD_INFO["n_train"]) + " images", data="military vehicles (Kricheli et al., 2024)",
                   test_accuracy=HEAD_INFO["accuracy"], opset=opset_file)
json.dump(labels_json, open(os.path.join(MODEL_DIR, tag + "_labels.json"), "w", encoding="utf-8"), indent=2, ensure_ascii=False)

idx = np.sort(np.random.default_rng(0).choice(TEST_IDX, min(N_CHECK, len(TEST_IDX)), replace=False))   # random test images
xb = aitbc.imagenet_batch([mv.image(i) for i in idx], S_IN)
with torch.no_grad():
    t0 = time.time()
    p_torch = model(torch.from_numpy(xb)).numpy()
    t_torch = (time.time() - t0) / len(idx) * 1000
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
if DYNAMIC_BATCH:
    t0 = time.time()
    p_ort = sess.run(None, {"image": xb})[0]
else:
    t0 = time.time()
    p_ort = np.concatenate([sess.run(None, {"image": xb[k:k + 1]})[0] for k in range(len(xb))])
t_ort = (time.time() - t0) / len(idx) * 1000
size_mb = sum(os.path.getsize(f_) for f_ in (onnx_path, onnx_path + ".data") if os.path.exists(f_)) / 2 ** 20
rows = {"file": f"{onnx_path} ({size_mb:.2f} MB)", "exporter, opset of the file": f"{EXPORTER}, {opset_file}",
        "time of the export": f"{t_exp:.1f} s", "operations of the graph": ", ".join(f"{k} {v}" for k, v in ops.head(8).items()),
        "ONNX Runtime: largest difference of the probabilities": f"{np.abs(p_ort - p_torch).max():.2e}",
        "ONNX Runtime: agreement of the classes": f"{np.mean(p_ort.argmax(1) == p_torch.argmax(1)):.3f}",
        "time per image: PyTorch / ONNX Runtime (CPU), ms": f"{t_torch:.1f} / {t_ort:.1f}"}
try:
    net_cv = cv2.dnn.readNetFromONNX(onnx_path)
    p_cv = []
    t0 = time.time()
    for k in range(len(xb)):
        net_cv.setInput(xb[k:k + 1])
        p_cv.append(net_cv.forward())
    t_cv = (time.time() - t0) / len(idx) * 1000
    p_cv = np.concatenate(p_cv)
    rows["OpenCV DNN: largest difference of the probabilities"] = f"{np.abs(p_cv - p_torch).max():.2e}"
    rows["OpenCV DNN: agreement of the classes; time per image, ms"] = f"{np.mean(p_cv.argmax(1) == p_torch.argmax(1)):.3f}; {t_cv:.1f}"
except Exception as e:
    rows["OpenCV DNN"] = f"the file was not read by OpenCV {cv2.__version__}: {str(e).splitlines()[0][:120]}"
y_chk = mv.table["cls" if HEAD_INFO["level"] == "24 types" else "group"].values[idx]
cls_onnx = np.array(HEAD_INFO["labels"])[p_ort.argmax(1)]
rows[f"accuracy of the ONNX model on {len(idx)} random test images"] = f"{np.mean(cls_onnx == y_chk):.3f}"
display(pd.DataFrame({"value": list(rows.values())}, index=list(rows)))
k_show = min(6, len(idx))
fig, ax = plt.subplots(1, k_show, figsize=(2.6 * k_show, 2.8))
for a, k in zip(np.ravel(ax), range(k_show)):
    a.imshow(mv.image(idx[k]))
    j = p_ort[k].argmax()
    a.set_title(f"true: {y_chk[k]}\nONNX: {HEAD_INFO['labels'][j]} ({p_ort[k, j]:.2f})", fontsize=8, color="k" if HEAD_INFO["labels"][j] == y_chk[k] else "#c62828")
    a.axis("off")
plt.tight_layout()
plt.show()
update_project_json("4.2", {"lesson_4_2": dict(mlp=globals().get("MLP_RESULT"), transfer=HEAD_INFO,
                                               onnx=os.path.relpath(onnx_path, PROJECT_ROOT), opset=opset_file, exporter=EXPORTER,
                                               max_difference_onnxruntime=float(np.abs(p_ort - p_torch).max()))})

### Report of Lesson 4.2

Double-click this cell and write the answers.

1. Neuron (block 4.2.1): weights set by hand and trained, accuracy, effect of the learning rate; answer to control question 7:
2. Fully connected network (block 4.2.2): layers, number of parameters, accuracy on the training and the test part; answer to control question 8:
3. Convolutional network (block 4.2.3): sizes of the maps of the blocks, classes of ImageNet for the chosen vehicle; answer to control question 9:
4. MobileNet (block 4.2.4): parameters, MAC and time of the five networks on the device of the session; answer to control question 10:
5. Transfer learning (block 4.2.5): HOG + SVM and CNN features on the same images, worst classes; answer to control question 11:
6. ONNX (block 4.2.6): file, opset, differences of the probabilities, time per image; answer to control question 12:

---
## Self-check of Lessons 4.1–4.2

1. A data set of frames of FPV drones over a training area contains 30 flights. Which checks are made before the training (duplicates, labels, empty frames), and how is the test part formed?
2. Why does the accuracy alone not suffice to estimate a recogniser of 24 types of vehicles with unequal numbers of images?
3. What do the HOG features describe, and why do they recognise a mine on grass better than the type of a tank in a web photograph?
4. What is common and what is different in a fully connected and in a convolutional network, and why is the second one used for images?
5. Why is MobileNet used on the onboard computer of a UAV or a ground robot instead of ResNet-50, and what must be checked after the export to ONNX?

## Data sources and licences

The course library reads the files directly from the repositories of the data sets; the data are not redistributed with the notebook. When images or results are published, the data sets are cited.

1. **Military vehicles.** Kricheli J. S., Vo K., Datta A., Ozgur S., Shakarian P. Error Detection and Constraint Recovery in Hierarchical Multi-Label Classification without Prior Knowledge. arXiv:2407.15192 (2024). https://arxiv.org/abs/2407.15192 · Data: https://huggingface.co/datasets/leibnitz-lab/military_vehicles · Licence declared by the authors: MIT; the photographs were collected by the authors from web sites, the rights to the individual photographs are not stated: use them only for training and research.
2. **SULAND.** Vivoli E., Bertini M., Capineri L. Deep Learning-Based Real-Time Detection of Surface Landmines Using Optical Imaging. *Remote Sensing*, 16(4), 677 (2024). https://doi.org/10.3390/rs16040677 · Refined annotation SULAND_v2: Lekhak S., Pulakurthi P. R., Joshi L., Bhatta R., Ientilucci E. J. SULAND v2: A Refined RGB Dataset and Deep Learning Object Detection Benchmark for UAV/UGV-Based SUrface LANDmine Detection Under Domain Shift. arXiv:2607.28996 (2026) · Data: https://huggingface.co/datasets/SagarLekhak/SULAND_v2_RGB_Surface_Landmine_Dataset · Licence CC BY 4.0. The mines are surrogates; the results are not an evidence of the readiness of a detector for humanitarian demining.
3. **ImageNet weights of MobileNetV3, MobileNetV2, ResNet** — torchvision (BSD-3-Clause); Howard A. et al. Searching for MobileNetV3. *ICCV 2019*, 1314–1324.
4. **HOG.** Dalal N., Triggs B. Histograms of Oriented Gradients for Human Detection. *CVPR 2005*, 886–893.

**Next notebook.** Notebook 07 — Topic 4, Lessons 4.3 and 4.4 (practical classes): classification of thermal signatures by HOG + SVM (target — background) with the analysis of errors; transfer learning of MobileNet on military vehicles with fine-tuning and export to ONNX.